# Acute Encephalopathy and the Heart: One-Click Publication Pipeline

This notebook runs the whole study from the two research-platform exports to every publication output: cleaning of the
exports, re-classification with the 2024 international consensus definitions of infection-triggered encephalopathy
syndromes (ITES), all analyses, publication-ready tables (Word and Excel) and figures (EPS, PDF, TIFF, PNG), figure
legends, and a cleaned-data workbook. The numbers, tables, and figures are those of the manuscript and its Supplemental
Digital Content.

**How to run.** Put this notebook in the same folder as the two exports (the main export, with sheet `病案首页基本信息`,
and the supplementary export, with sheet `病案首页其他信息`) and the frozen dataset folder `AE_freeze_v1.2` (reference labels for
critical discharge against medical advice and organ support), and choose *Restart Kernel and Run All Cells*. The first run
parses the exports (2-3 minutes) and caches them in `./cache_aes_exports/`; a full run takes about 6-7 minutes on our test machine
(about 4 minutes once the exports are cached), mostly for the bootstrap and permutation steps. Requirements: Python 3.10 or later with pandas, NumPy, SciPy,
statsmodels, scikit-learn, Matplotlib, Pillow, and openpyxl; python-docx is installed automatically if it is missing.
Figures use Arial whenever it is installed; set `EXPORT_TIFF = False` in the first code cell to skip the TIFF copies.

**Outputs** (all in `./output_publication/`)

| Folder | Contents | Identifiers |
|---|---|---|
| `figures/` | Figures 1-3 and eFigure 1: EPS and PDF (vector, for submission), TIFF (7 in wide at 1,200 dpi, LZW), PNG (600 dpi preview) | None |
| `tables/` | `Table1.docx` and `Table2.docx` (one table per file), `Supplemental_eTables_eFigure.docx` (eTables 1-13 and eFigure 1), `Figure_legends.docx`, `Publication_tables.xlsx` (every table, one sheet each) | None |
| `analysis/` | Analysis results behind every table and figure: `Tables_AES_cardiac.xlsx`, `Tables_ITES.xlsx`, `manuscript_numbers.json`, `key_numbers.json`, `publication_tables.json` | None |
| `cleaned_data_INTERNAL/` | `AES_cleaned_dataset_INTERNAL.xlsx` (README, data dictionary, analysis dataset, ITES evidence, repeated measurements, cohort flow, descriptive summary) and CSV copies of the three datasets | Research IDs and admission dates: **internal use only; never upload as Supplemental Digital Content** |

**Adjudication.** When `output_ites/ites_adjudicated.csv` exists (columns `vis` and `adjudicated_class`, saved after
adjudication with `AES_ITES_classification.ipynb`), adjudicated categories replace the algorithmic ones, and every
table, figure, and dataset is regenerated with them.

**Data notes.**
* High-sensitivity cardiac troponin T (hs-cTnT; cobas e602, Roche Diagnostics; confirmed with the laboratory) is reported
  in ng/mL with a lower reporting limit of 0.003 ng/mL. The primary threshold is the assay's 99th-percentile upper
  reference limit (URL) of 0.014 ng/mL (`HSCTN_URL`, 14 ng/L); the laboratory report displays the reference upper limit
  rounded to 0.01 ng/mL. Twice the URL (28 ng/L) is analyzed as a sensitivity threshold.
* Laboratory results stored only in the qualitative column (for example "<0.003" or ">9000") are read as the reportable
  limit and flagged as censored; results without either value are ignored.
* No laboratory sampling times were exported; report times are used as the timing proxy.
* Every number is computed from the exported data; nothing is imputed, simulated, or typed by hand. Displayed numbers
  use half-up rounding.
* Printed outputs in this notebook contain counts and summary statistics only, never identifiers or free text.

## Part 1. Setup, loading, and cleaning of the exports

Admissions and discharge diagnoses are parsed, acute encephalopathy admissions are identified, the index admission is
selected (first acute-phase admission per child with a recorded discharge disposition), and laboratory,
electrocardiography, echocardiography, organ-support, and treatment variables are derived.

In [ ]:
import io, re, json, warnings, collections
from pathlib import Path
import numpy as np
import pandas as pd
import openpyxl
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
SEED = 20261009
rng = np.random.default_rng(SEED)
HSCTN_URL = 0.014       # ng/mL: 99th-percentile URL of the hs-cTnT assay (Roche cobas e602; report displays 0.01 after rounding)
HSCTN_URL_ALT = 2 * HSCTN_URL   # ng/mL: sensitivity threshold (twice the URL)
URL_NGL, ALT_NGL = int(round(HSCTN_URL * 1000)), int(round(HSCTN_URL_ALT * 1000))
from decimal import Decimal, ROUND_HALF_UP

def rnd(x, d=1):
    # Conventional half-up rounding for displayed numbers (avoids binary-float and half-to-even artifacts).
    return str(Decimal(f'{float(x):.10f}').quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP))

HERE = Path.cwd()
PUB = HERE / 'output_publication'          # everything this notebook writes
OUT = PUB / 'analysis'                     # analysis results behind every table and figure (no identifiers)
FIG = PUB / 'figures'                      # Figures 1-3 and eFigure 1
TAB = PUB / 'tables'                       # Tables 1-2, eTables 1-13, figure legends
DATA = PUB / 'cleaned_data_INTERNAL'       # cleaned datasets with research IDs and admission dates: internal use only
CACHE = HERE / 'cache_aes_exports'         # parsed exports (shared with the other notebooks)
for p in (PUB, OUT, FIG, TAB, DATA, CACHE):
    p.mkdir(parents=True, exist_ok=True)
EXPORT_TIFF, TIFF_WIDTH_IN, TIFF_DPI = True, 7.0, 1200   # TIFF copies: double-column width at 1,200 dpi (line art)

def pick_font():
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']:
        if name in available:
            return name
    return 'sans-serif'

FONT = pick_font()
plt.rcParams.update({
    'font.family': FONT, 'font.size': 13, 'axes.labelsize': 14, 'axes.labelweight': 'bold',
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 11, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False, 'axes.linewidth': 1.0,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
print('Figure font:', FONT, '(Arial is used whenever it is installed)')

In [ ]:
# Locate and load both exports (about 2.5 minutes on the first run; cached afterwards).
KEY_SHEET, SUPP_SHEET = '病案首页基本信息', '病案首页其他信息'

def find_workbooks(folder):
    main = supp = None
    for p in sorted(folder.glob('*.xlsx')):
        if p.name.startswith('~$'):
            continue
        try:
            wb = openpyxl.load_workbook(p, read_only=True); names = wb.sheetnames; wb.close()
        except Exception:
            continue
        if SUPP_SHEET in names:
            supp = p
        elif KEY_SHEET in names:
            main = p
    if main is None or supp is None:
        raise FileNotFoundError('Both the main export and the supplementary export must be in this folder.')
    return main, supp

def load_book(path, cache_name):
    f = CACHE / cache_name
    if f.exists() and f.stat().st_mtime > path.stat().st_mtime:
        return pd.read_pickle(f)
    book = pd.read_excel(path, sheet_name=None, dtype=object, engine='openpyxl')
    pd.to_pickle(book, f)
    return book

WB, WB_SUPP = find_workbooks(HERE)
sheets = load_book(WB, 'main.pkl')
supp = load_book(WB_SUPP, 'supplement.pkl')
print('Main export:', WB.name, '| sheets:', len(sheets))
print('Supplementary export:', WB_SUPP.name, '| sheets:', len(supp))

In [ ]:
# Admissions and free-text discharge diagnoses (same parser as the data-asset notebook).
def split_top(s):
    out, buf, depth = [], '', 0
    for ch in s:
        if ch in '(（[［':
            depth += 1
        elif ch in ')）]］':
            depth = max(0, depth - 1)
        if ch in ',，;；、' and depth == 0:
            out.append(buf); buf = ''
        else:
            buf += ch
    out.append(buf)
    return out

def parse_dx(s, kind='出院'):
    if not isinstance(s, str):
        return []
    s = re.sub(kind + r'诊断\s*\d*\s*[:：]', '\n', s.replace('\r', '\n'))
    items = []
    for line in s.split('\n'):
        line = line.strip()
        if not line:
            continue
        for part in re.split(r'(?:(?<=^)|(?<=[^\d\.]))\d{1,2}\s*[\.．、]\s*(?=\D)', line):
            for q in split_top(part):
                q = q.strip(' 　\t。.')
                if q:
                    items.append(q)
    return items

def uniq(seq):
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out

fp = sheets['病案首页基本信息'].copy()
fp['vis'] = fp['科研就诊编号'].astype(str)
fp['pid'] = fp['科研患者编号'].astype(str)
fp['admit'] = pd.to_datetime(fp['入院日期'], errors='coerce')
fp['discharge'] = pd.to_datetime(fp['出院日期'], errors='coerce')
fp['age_y'] = pd.to_numeric(fp['年龄（岁）'], errors='coerce')

dis = sheets['住院病历出院记录'].copy()
dis['dx_list'] = dis['出院诊断'].apply(parse_dx)
longest = lambda s: max([x for x in s if isinstance(x, str)] or [''], key=len)
agg = dis.groupby('科研就诊编号').agg(dx_list=('dx_list', lambda s: uniq(sum(s.tolist(), []))),
                                    course=('诊疗经过', longest), dstatus=('出院情况', longest)).reset_index()
agg['vis'] = agg['科研就诊编号'].astype(str)
adm = fp.merge(agg.drop(columns=['科研就诊编号']), on='vis', how='left')
adm['has_disrec'] = adm['dx_list'].apply(lambda x: isinstance(x, list))
adm['dx_list'] = adm['dx_list'].apply(lambda x: x if isinstance(x, list) else [])
for c in ['course', 'dstatus']:
    adm[c] = adm[c].fillna('')
print('Admissions:', len(adm), '| patients:', adm.pid.nunique())

In [ ]:
# Provisional syndrome labels (rule-based; identical to the data-asset notebook).
ORDER = [
    ('Metabolic_Leigh', r'亚急性坏死性|Leigh'),
    ('ANE', r'坏死性\)?脑|脑病[^,，;；]{0,14}坏死性|急性坏死性(?!.{0,4}(?:肠|筋膜|牙龈|肺|淋巴|胰))'),
    ('HSES_ASEM', r'出血性休克脑病|休克脑病|HSES'),
    ('Reye', r'瑞氏|Reye|瑞夷'),
    ('AESD', r'AESD|双相发作|双向发作|迟发性弥散|后期弥散|后期弥漫|晚期弥散|迟发性扩散'),
    ('FIRES', r'FIRES|FIERS|热性感染相关|发热感染相关|热性敏感性|感染相关性癫痫综合征|AERRPS|难治性反复部分性'),
    ('MERS', r'胼胝体压部|MERS|RESLES|胼胝体压迫|可逆性胼胝体|胼胝体可逆'),
    ('Septic_encephalopathy', r'脓毒症性\)?脑病|脓毒性\)?脑病|脓毒症相关性脑病|脓毒症脑病'),
    ('Pertussis_encephalopathy', r'百日咳脑病'),
    ('Influenza_encephalopathy', r'流感相关性\)?脑病|流感脑病|流感相关脑病|流感相关性脑炎'),
    ('Toxic_encephalopathy', r'中毒性脑病'),
    ('Bilirubin', r'胆红素脑病|核黄疸'),
    ('Neonatal_HIE_NE', r'新生儿.*脑病|新生儿缺氧缺血|新生儿缺血缺氧|\(新生儿\)缺氧'),
    ('HIE_other', r'缺氧缺血性脑病|缺血缺氧性脑病|缺氧性脑病|缺氧缺血性脑损伤'),
    ('DEE', r'癫痫性脑病|ESES|CSWS|睡眠中癫痫性电持续|Lennox|大田原|暴发抑制|发育性.*脑病|癫痫性\)脑病|West综合征|婴儿痉挛'),
    ('Metabolic', r'代谢性脑病|甘氨酸脑病|低血糖性脑病|低血糖昏迷|肝性脑病|尿毒症性脑病|Wernicke|韦尼克|线粒体脑|高氨血症脑病'),
    ('PRES_hypertensive', r'可逆性后部|PRES|高血压脑病|后部可逆'),
    ('Lupus', r'狼疮性脑病|狼疮脑病|神经精神狼疮|狼疮脑'),
    ('Autoimmune', r'自身免疫性脑炎|自体免疫性脑炎|NMDA|GABA|LGI1|免疫介导性脑炎|自身免疫相关性脑病|桥本脑病|自身免疫性\)?脑病|免疫性？\)|ADEM|急性播散性脑脊髓炎'),
    ('Infectious_encephalitis', r'病毒性脑炎|脑膜脑炎|^\(?病毒性\)?脑炎|化脓性脑膜炎|细菌性脑膜炎|结核性脑膜炎|^脑炎|汉坦病毒脑炎|疱疹病毒性脑炎|乙型脑炎'),
    ('Leukoencephalopathy', r'白质脑病|脑白质病'),
    ('AES_unspecified', r'急性脑病|急性\)\s*脑病|^脑病|弥散性脑病|中枢神经系统感染相关性脑病'),
]
AES = ['ANE', 'HSES_ASEM', 'Reye', 'AESD', 'FIRES', 'MERS', 'Septic_encephalopathy', 'Pertussis_encephalopathy',
       'Influenza_encephalopathy', 'Toxic_encephalopathy', 'AES_unspecified']
SEQ = re.compile(r'后遗症|恢复期|随诊|复查|治疗后|个人史')

def label_string(x):
    if '胼胝体' in x and re.search(r'先天性|畸形|发育不良|缺如', x) and '脑病' not in x:
        return None
    for k, pat in ORDER:
        if re.search(pat, x):
            return k
    return None

def visit_labels(L):
    out = {}
    for x in L:
        k = label_string(x)
        if k:
            out.setdefault(k, []).append(x)
    return out

def aes_info(lab):
    keys = [k for k in lab if k in AES]
    if not keys:
        return pd.Series([False, None, None])
    strs = sum([lab[k] for k in keys], [])
    acute = any(not SEQ.search(s) for s in strs)
    return pd.Series([True, next(k for k in AES if k in keys), acute])

# Coded discharge diagnoses are used only where no discharge summary exists (mostly in-hospital deaths).
dxc = supp['病案出院诊断归一'].copy(); dxc['vis'] = dxc['科研就诊编号'].astype(str)
coded = dxc.groupby('vis')['诊断疾病名称'].apply(lambda s: [str(x) for x in s])
adm['coded_list'] = adm.vis.map(coded).apply(lambda x: x if isinstance(x, list) else [])
adm['lab'] = [visit_labels(L if hd else C) for L, C, hd in zip(adm['dx_list'], adm['coded_list'], adm['has_disrec'])]
adm[['aes', 'aes_sub', 'aes_acute']] = adm['lab'].apply(aes_info)
adm['all_dx_text'] = [' | '.join(L + C) for L, C in zip(adm['dx_list'], adm['coded_list'])]
print('AES-labeled admissions:', int(adm.aes.sum()))

In [ ]:
# Index admission: the first acute-phase AES admission per patient; dispositions from the supplement.
oth = supp['病案首页其他信息'].copy(); oth['vis'] = oth['科研就诊编号'].astype(str)
PICU_WARD = re.compile(r'重症医学|儿童重症监护')     # PICU wards (surgical ICU not included)
trf = supp['患者转科信息'].copy(); trf['vis'] = trf['科研就诊编号'].astype(str)
trf['t'] = pd.to_datetime(trf['转科时间'], errors='coerce')
trf = trf.merge(adm[['vis', 'admit', 'discharge']], on='vis', how='inner')
trf = trf[(trf.t >= trf.admit - pd.Timedelta(days=1)) & (trf.t <= trf.discharge + pd.Timedelta(days=1))]
PICU_TRANSFER = set(trf[trf['转出科室'].astype(str).str.contains(PICU_WARD) | trf['转入科室'].astype(str).str.contains(PICU_WARD)].vis)
vinfo = supp['患者就诊信息'].copy(); vinfo['vis'] = vinfo['科研就诊编号'].astype(str)
adm = adm.merge(oth[['vis', '离院方式']], on='vis', how='left').merge(vinfo[['vis', '转归情况']], on='vis', how='left')
adm['covered'] = adm.vis.isin(set(oth.vis))
adm['death'] = adm['离院方式'].eq('死亡') | adm['转归情况'].eq('死亡')
adm['ama'] = adm['离院方式'].eq('非医嘱离院')

# Critical-condition wording at the end of the discharge summary (supplement version, else main export).
WD = re.compile(r'放弃治疗|放弃抢救|自动出院|拒绝.{0,6}(?:气管插管|治疗)|随时.{0,6}死亡|死亡风险|心跳骤停.{0,10}(?:风险|可能)|极可能.{0,4}死亡|带气管插管出院|拔除气管插管|撤机')
CRIT = re.compile(r'危重|病重|预后差|预后极差|预后不良|昏迷|电静息|脑死亡|脑疝')
ds = supp['住院病历出院记录'].copy(); ds['vis'] = ds['科研就诊编号'].astype(str)
ds_txt = ds.groupby('vis').apply(lambda g: ' '.join(str(x) for x in list(g['诊疗经过'])[-1:] + list(g['出院情况'])))
def critical_wording(row):
    t = ds_txt.get(row.vis, None)
    t = str(t) if t is not None else (row.course + ' ' + row.dstatus)
    tail = t[-600:]
    return bool(WD.search(tail) and CRIT.search(tail))

adm = adm.sort_values(['pid', 'admit'])
rows = []
for pid, g in adm[adm.aes == True].groupby('pid'):
    sub = next(k for k in AES if k in set(g.aes_sub))
    ac = g[g.aes_acute == True]
    r = ac.iloc[0] if len(ac) else g.iloc[0]
    wards = ' '.join(str(x) for x in [r['入院科别'], r['出院科别'], r['转科科别']] if isinstance(x, str))
    rows.append({'patient': pid, 'vis': r.vis, 'subtype': sub, 'acute_index': bool(r.aes_acute), 'admit': r.admit,
                 'discharge': r.discharge, 'year': r.admit.year, 'age_y': r.age_y, 'male': r['性别'] == '男性',
                 'icu': bool(PICU_WARD.search(wards)) or r.vis in PICU_TRANSFER, 'covered': bool(r.covered), 'death': bool(r.death), 'ama': bool(r.ama),
                 'disposition': r['离院方式'], 'has_disrec': bool(r.has_disrec), 'dx_text': r.all_dx_text,
                 'course': r.course, 'dstatus': r.dstatus})
coh = pd.DataFrame(rows)
idx = coh[coh.acute_index & coh.covered].copy().reset_index(drop=True)
idx['critical_ama_rule'] = idx.ama & idx.apply(critical_wording, axis=1)   # documentation rule (sensitivity label)

# Reference labels from the frozen dataset of this export (AE_freeze_v1.2): critical discharge against medical advice as adjudicated by a
# pediatric clinician (dama_critical_final; the freeze rule label is kept where the reviewer was unsure).
FREEZE_DIR = HERE / 'AE_freeze_v1.2'
if not (FREEZE_DIR / 'tables' / 'admission_summary.csv').exists():
    raise FileNotFoundError('Place the frozen dataset folder AE_freeze_v1.2 (with its tables/ subfolder) next to this notebook.')
FZ = pd.read_csv(FREEZE_DIR / 'tables' / 'admission_summary.csv', encoding='utf-8-sig', dtype={'visit_id': str}, low_memory=False).set_index('visit_id')
FZ_EVENTS = pd.read_csv(FREEZE_DIR / 'tables' / 'support_events.csv', encoding='utf-8-sig', dtype={'visit_id': str}, low_memory=False)
not_in_freeze = set(idx.vis) - set(FZ.index)
if not_in_freeze:
    raise ValueError(f'{len(not_in_freeze)} index admissions are missing from the frozen dataset.')

def fz_flag(col):
    return idx.vis.map(FZ[col].astype(str).str.lower().eq('true')).astype(bool)

idx['critical_ama'] = idx.ama & fz_flag('dama_critical_final')
idx['poor'] = idx.death | idx.critical_ama
print('AES patients:', len(coh), '| acute index admissions:', int(coh.acute_index.sum()),
      '| with a recorded disposition:', len(idx))
print('Deaths:', int(idx.death.sum()), '| critical AMA (clinician-adjudicated):', int(idx.critical_ama.sum()),
      '| by the documentation rule:', int(idx.critical_ama_rule.sum()), '| death or critical AMA:', int(idx.poor.sum()))

In [ ]:
# Laboratory, ECG, and echocardiography measurements (both exports, de-duplicated by report number).
def parse_lab_value(quant, qual=None):
    # Numeric result; when only the qualitative column holds it (for example "<0.003" or ">9000"), the value or the
    # reportable limit is used and the result is flagged as censored.
    v = pd.to_numeric(quant, errors='coerce')
    if qual is None:
        return v, pd.Series(False, index=v.index)
    q = qual.astype(str).str.strip().str.replace('〈', '<').str.replace('〉', '>').str.replace('＜', '<').str.replace('＞', '>')
    m = q.str.extract(r'^([<>]=?)?\s*(\d*\.?\d+)$')
    qv = pd.to_numeric(m[1], errors='coerce')
    use = v.isna() & qv.notna()
    return v.where(~use, qv), use & m[0].notna()

def qual_col(df, col):
    base = col.replace('-定量结果', '')
    for c in (base + '-定性结果', base + '-定性结果显示'):
        if c in df.columns:
            return df[c]
    return None

def lab_long(sheet, col):
    parts = []
    for book in (sheets, supp):
        if sheet in book and col in book[sheet].columns:
            df = book[sheet]
            val, cens = parse_lab_value(df[col], qual_col(df, col))
            parts.append(pd.DataFrame({'report': df['报告编号'].astype(str), 'vis': df['就诊编号'].astype(str),
                                       'value': val, 'censored': cens,
                                       'time': pd.to_datetime(df['报告时间'], errors='coerce')}))
    x = pd.concat(parts).dropna(subset=['value', 'time']).drop_duplicates('report')
    return x[x.vis.isin(set(idx.vis))]

def summarize(long, hours, prefix):
    x = long.merge(idx[['vis', 'admit', 'discharge']], on='vis')
    x['h'] = (x.time - x.admit).dt.total_seconds() / 3600
    within = x[(x.h >= -12) & (x.h <= hours)].sort_values('time')
    first = within.groupby('vis').first()
    out = pd.DataFrame({f'{prefix}_early': first.value, f'{prefix}_early_time': first.time})
    out[f'{prefix}_peak'] = x.groupby('vis').value.max()
    out[f'{prefix}_first_time'] = x.sort_values('time').groupby('vis').time.first()
    return out

tnt = summarize(lab_long('实验室检查_血液心肌酶谱测定', '超敏心肌钙蛋白定量-定量结果'), 48, 'tnt')
ckmb = summarize(lab_long('实验室检查_血液心肌酶谱测定', '肌酸激酶同工酶-ＭＢ定量-定量结果'), 48, 'ckmb')
bnp = summarize(lab_long('实验室检查_血液N末端B型钠尿肽前体测定', 'N末端B型利钠肽原定量-定量结果'), 48, 'bnp')
lac = summarize(lab_long('实验室检查_血液乳酸测定', '乳酸定量-定量结果'), 48, 'lac')

ecg = supp['心电图报告'].copy(); ecg['vis'] = ecg['科研就诊编号'].astype(str)
ecg['time'] = pd.to_datetime(ecg['心电检查时间'], errors='coerce')
ecg['HR'] = pd.to_numeric(ecg['HR'], errors='coerce'); ecg['QT'] = pd.to_numeric(ecg['QT'], errors='coerce')
ecg = ecg[ecg.vis.isin(set(idx.vis)) & ecg.HR.between(30, 300) & ecg.QT.between(150, 700)].copy()
ecg['value'] = ecg.QT / np.cbrt(60 / ecg.HR)                      # Fridericia
ecg['QTcB'] = ecg.QT / np.sqrt(60 / ecg.HR)                        # Bazett (equals the machine QTc)
ecg['report'] = ecg.index.astype(str)
qtc = summarize(ecg[['report', 'vis', 'value', 'time']], 48, 'qtcf')
qtc['qtcb_peak'] = ecg.groupby('vis').QTcB.max()
qtc['qtcb_early'] = summarize(ecg[['report', 'vis', 'QTcB', 'time']].rename(columns={'QTcB': 'value'}), 48, 'qtcb')['qtcb_early']

echo = supp['儿童心脏彩超'].copy(); echo['vis'] = echo['科研就诊编号'].astype(str)
echo['time'] = pd.to_datetime(echo['检查时间'], errors='coerce'); echo['value'] = pd.to_numeric(echo['射血分数'], errors='coerce')
echo = echo[echo.vis.isin(set(idx.vis))].dropna(subset=['value', 'time'])
echo['report'] = echo.index.astype(str)
x = echo.merge(idx[['vis', 'admit']], on='vis'); x['h'] = (x.time - x.admit).dt.total_seconds() / 3600
ef = pd.DataFrame({'lvef_min': x.groupby('vis').value.min(),
                   'lvef_early_min': x[(x.h >= -24) & (x.h <= 72)].groupby('vis').value.min(),
                   'lvef_first_time': x.sort_values('time').groupby('vis').time.first()})

for t in (tnt, ckmb, bnp, lac, qtc, ef):
    idx = idx.merge(t, left_on='vis', right_index=True, how='left')
idx = idx.reset_index(drop=True)
idx['tnt_early_ngL'] = idx.tnt_early * 1000
idx['tnt_peak_ngL'] = idx.tnt_peak * 1000
print('Index admissions with early hs-cTn:', int(idx.tnt_early.notna().sum()), '| NT-proBNP:', int(idx.bnp_early.notna().sum()),
      '| ECG QT/HR:', int(idx.qtcf_early.notna().sum()), '| LVEF:', int(idx.lvef_min.notna().sum()))

In [ ]:
# Organ-support and severity variables, cardiac arrest timing, era.
proc = supp['病案手术操作归一'].copy(); proc['vis'] = proc['科研就诊编号'].astype(str)
proc['date'] = pd.to_datetime(proc['手术操作日期'], errors='coerce')
proc = proc[proc.vis.isin(set(idx.vis))]
code_ = proc['手术操作编码'].astype(str)
vent_proc = set(proc[code_.str.match(r'^96\.7|^96\.04')].vis)
cpr_proc = proc[code_.str.match(r'^99\.6')].groupby('vis').date.min()

orders = supp['住院非用药医嘱'].copy(); orders['vis'] = orders['科研就诊编号'].astype(str)
orders = orders[orders.vis.isin(set(idx.vis))]
vent_order = set(orders[orders['住院非用药医嘱名称'].astype(str).str.contains('气管插管')].vis)

meds = pd.concat([sheets['住院用药医嘱'][['科研就诊编号', '住院用药医嘱药品名称', '住院用药医嘱用法']],
                  supp['住院用药医嘱'][['科研就诊编号', '住院用药医嘱药品名称', '住院用药医嘱用法']]])
meds['vis'] = meds['科研就诊编号'].astype(str)
meds = meds[meds.vis.isin(set(idx.vis))]
name = meds['住院用药医嘱药品名称'].astype(str); route = meds['住院用药医嘱用法'].astype(str)
VASO = r'(?<!异丙)肾上腺素|多巴胺|多巴酚丁胺|米力农|垂体后叶|血管加压素'   # isoproterenol (异丙肾上腺素) excluded
BOLUS_OR_LOCAL = r'雾化|气管|静脉注射|静推|皮下|外用|局部|口服|滴眼|滴鼻'
vaso = set(meds[name.str.contains(VASO) & ~route.str.contains(BOLUS_OR_LOCAL)].vis)
DRUG = {'methylprednisolone': r'甲泼尼龙|甲强龙|甲基强的松龙', 'ivig': r'丙种球蛋白|免疫球蛋白', 'tocilizumab': r'托珠单抗'}
for k, p in DRUG.items():
    idx[k] = idx.vis.isin(set(meds[name.str.contains(p)].vis))

idx['vasoactive'] = idx.vis.isin(vaso)

# Time-resolved organ support (order timestamps) for the time-anchored analyses.
meds_t = pd.concat([sheets['住院用药医嘱'][['科研就诊编号', '住院用药医嘱开立时间', '住院用药医嘱药品名称', '住院用药医嘱用法']],
                    supp['住院用药医嘱'][['科研就诊编号', '住院用药医嘱开立时间', '住院用药医嘱药品名称', '住院用药医嘱用法']]])
meds_t['vis'] = meds_t['科研就诊编号'].astype(str)
meds_t = meds_t[meds_t.vis.isin(set(idx.vis))]
meds_t['t'] = pd.to_datetime(meds_t['住院用药医嘱开立时间'], errors='coerce')
nm, rt = meds_t['住院用药医嘱药品名称'].astype(str), meds_t['住院用药医嘱用法'].astype(str)
infusion = meds_t[nm.str.contains(VASO) & ~rt.str.contains(BOLUS_OR_LOCAL)]
pressor = infusion[~infusion['住院用药医嘱药品名称'].astype(str).str.contains('多巴酚丁胺|米力农')]
idx['vaso_time'] = idx.vis.map(infusion.groupby('vis').t.min())
idx['pressor_time'] = idx.vis.map(pressor.groupby('vis').t.min())
orders['t'] = pd.to_datetime(orders['住院非用药医嘱开始时间'], errors='coerce')
onames = orders['住院非用药医嘱名称'].astype(str)
inv = orders[onames.str.contains('气管插管|呼吸机辅助呼吸|机械通气') & ~onames.str.contains('无创|鼻塞|CPAP|高流量')]
vent_proc_date = proc[code_.str.match(r'^96\.7|^96\.04')].groupby('vis').date.min()
idx['vent_time'] = pd.concat([idx.vis.map(inv.groupby('vis').t.min()), idx.vis.map(vent_proc_date)], axis=1).min(axis=1)
idx['ventilation'] = idx.vis.isin(vent_proc | vent_order | set(inv.vis))
# Reference definitions: vasoactive infusion and invasive ventilation (presence and first time) follow the frozen dataset rules (AE_freeze_v1.2,
# definitions section 3: continuous-infusion routes only, pituitrin reported separately; planned-procedure intubation and ventilator orders with
# non-invasive ventilation evidence not counted). Vasopressor-only start excludes dobutamine and milrinone.
own = idx[['vasoactive', 'ventilation']].copy()
idx['vasoactive'] = fz_flag('vasoactive_ever')
idx['vaso_time'] = pd.to_datetime(idx.vis.map(FZ.vasoactive_first_time), errors='coerce')
va_ev = FZ_EVENTS[FZ_EVENTS.support.eq('vasoactive_infusion')].copy()
va_ev['t'] = pd.to_datetime(va_ev.event_time, errors='coerce')
idx['pressor_time'] = idx.vis.map(va_ev[~va_ev.detail.astype(str).isin(['dobutamine', 'milrinone'])].groupby('visit_id').t.min())
idx['ventilation'] = fz_flag('imv_ever')
idx['vent_time'] = pd.to_datetime(idx.vis.map(FZ.imv_first_time), errors='coerce')
print('Frozen-dataset rules vs order-text rules: vasoactive', int(idx.vasoactive.sum()), 'vs', int(own.vasoactive.sum()),
      '| invasive ventilation', int(idx.ventilation.sum()), 'vs', int(own.ventilation.sum()))
txt = idx.dx_text
idx['status_epilepticus'] = txt.str.contains('癫痫持续状态|惊厥持续状态')
idx['septic_shock'] = txt.str.contains('脓毒性休克|感染性休克')
idx['shock_dx'] = txt.str.contains('休克')
idx['lactate_ge4'] = idx.lac_early >= 4
idx['age_lt1'] = idx.age_y < 1

# Cardiac arrest / CPR: dated procedures, undated diagnoses, or narrative mention.
idx['cpr_date'] = idx.vis.map(cpr_proc)
ARREST = re.compile(r'心肺复苏|心脏骤停|心跳骤停|呼吸心跳骤停|心搏骤停|心脏停搏')
HEDGE = re.compile(r'可能|随时|风险|警惕|预防|避免|防止|告知|交代|谈话|签字|若|如出现|一旦|拒绝|放弃|不予')

def arrest_documented(text):
    # Affirmative narrative mention of cardiac arrest or CPR; clauses with warnings, counseling, or refusal are ignored.
    for sent in re.split(r'[。；;\n！!？?]', str(text)):
        for c in re.split(r'[，,]', sent):
            m = ARREST.search(c)
            if m and not HEDGE.search(c) and not re.search(r'无|未|否认', c[max(0, m.start() - 4):m.start()]):
                return True
    return False

idx['cpr_any'] = idx.cpr_date.notna() | txt.str.contains('心脏停搏|呼吸心跳骤停|心跳骤停|心搏骤停') | idx.course.apply(arrest_documented)
# CPR orders and procedure codes from the frozen dataset are added; the earliest dated evidence is the CPR date.
idx['cpr_any'] = idx.cpr_any | fz_flag('cpr_ever')
idx['cpr_date'] = pd.concat([idx.cpr_date, pd.to_datetime(idx.vis.map(FZ.cpr_first_time), errors='coerce').dt.normalize()], axis=1).min(axis=1)
def arrest_before(first_time):
    # True when an arrest/CPR is documented and cannot be shown to occur on a later calendar day than first_time.
    later = idx.cpr_date.notna() & first_time.notna() & (first_time.dt.normalize() < idx.cpr_date)
    return idx.cpr_any & ~later
idx['arrest_before_tnt'] = arrest_before(idx.tnt_early_time)
idx['arrest_before_echo'] = arrest_before(idx.lvef_first_time)
idx['arrest_before_ecg'] = arrest_before(idx.qtcf_early_time)
idx['arrest_before_bnp'] = arrest_before(idx.bnp_early_time)
idx['arrest_before_tnt_first'] = arrest_before(idx.tnt_first_time)

idx['era'] = pd.cut(idx.admit, [pd.Timestamp('2000-01-01'), pd.Timestamp('2020-01-01'), pd.Timestamp('2022-12-01'),
                                pd.Timestamp('2023-02-01'), pd.Timestamp('2100-01-01')],
                    labels=['Pre-2020', '2020-Nov 2022', 'Dec 2022-Jan 2023', 'Feb 2023 onward'], right=False)
print(idx[['ventilation', 'vasoactive', 'status_epilepticus', 'septic_shock', 'cpr_any', 'methylprednisolone', 'ivig', 'tocilizumab']].sum())

## Part 2. ITES re-classification (2024 international consensus)

Each index admission is re-classified against the 2024 international consensus definitions of
infection-triggered encephalopathy syndromes (ITES; Sakuma et al., Dev Med Child Neurol 2024).
The algorithm only *proposes* a category from the exported evidence; every proposal is written to an
adjudication worksheet so that a pediatric neurologist can confirm or override it. If an adjudicated file
(`output_ites/ites_adjudicated.csv`, columns `vis` and `adjudicated_class`) is present, adjudicated
categories replace the algorithmic ones in all downstream analyses.

In [ ]:
# ITES evidence 1/4: neuroimaging features from MRI and head CT reports (clause-level negation, side-aware thalamic rule).
SENT = re.compile(r'[。；;\n！!？?]')
CLAUSE = re.compile(r'[，,]')
NEG_IMG = re.compile(r'未见|未发现|无明显|无异常|无确切|未显示|不明显|消失|吸收|未见确切|(?<!异常)正常|排除')
ABN = re.compile(r'异常信号|信号异常|异常密度|长T1|长T2|短T1|短T2|高信号|信号增高|信号灶|信号影|弥散受限|扩散受限|肿胀|肿大|水肿|坏死|出血'
                 r'|异常灶|病灶|低密度|密度减低|密度降低|异常改变')
DWI_POS = re.compile(r'弥散受限|扩散受限|弥散[^，,。；]{0,4}受限|DWI[^，,。；]{0,14}(?:高信号|信号增高|受限)|弥散加权[^，,。；]{0,8}高信号')
SIDE = re.compile(r'双侧|两侧|对称|左|右')
DIFFUSE_EDEMA = re.compile(   # global swelling, herniation or cisternal effacement
    r'全脑(?:肿胀|水肿)|弥漫性脑(?:肿胀|水肿)|广泛(?:性)?脑(?:肿胀|水肿)|脑组织弥漫性?肿胀|颅内广泛脑肿胀|全脑弥漫性肿胀'
    r'|大脑[、及和]?小脑半球[^。；]{0,24}肿胀|扁桃体[^，,。；]{0,6}(?:下疝|疝入)|脑疝形成|全脑.{0,6}脑疝'
    r'|脑池[^，,。；]{0,6}(?:闭塞|消失)|基底池[^，,。；]{0,6}(?:受压|闭塞|消失)'
    r'|脑沟[^，,。；]{0,4}(?:普遍|广泛|弥漫)[^，,。；]{0,4}(?:变浅|消失)|(?:普遍|广泛|弥漫)[^，,。；]{0,6}脑沟[^，,。；]{0,6}(?:变浅|消失)')
GWM_LOSS = re.compile(r'(?:双侧|两侧)大脑(?:半球)?[^。；]{0,16}灰白质(?:分界|界限)[^，,。；]{0,3}(?:不清|模糊|欠清)')
EFFACE = re.compile(r'脑沟[^，,。；]{0,6}(?:变浅|消失|受压)|脑池[^，,。；]{0,6}(?:受压|变窄|闭塞|消失)|脑室[^，,。；]{0,8}(?:变窄|变小|稍小|受压|缩小|显示不清)'
                    r'|外侧裂[^，,。；]{0,4}(?:闭塞|变窄)')
SUBCORT = re.compile(r'皮层下|皮质下|U形纤维|U纤维|皮髓质交界|灰白质交界')
LOBAR_WM = re.compile(r'(?:额|顶|颞|枕)叶?[^，,。；]{0,6}白质|大脑半球[^，,。；]{0,4}白质')
EXTENSIVE = re.compile(r'双侧|两侧|多发|弥漫|广泛|大脑半球|额顶|顶枕|额颞|颞顶|颞枕')
UNSURE = re.compile(r'似|可疑|[？?]')
HEMI = re.compile(r'(?<!双侧)(?<!两侧)(?:左|右)(?:侧)?大脑半球')
CHRONIC = re.compile(r'萎缩|软化|囊变|胶质增生|陈旧')


def clauses(text):
    for sent in SENT.split(str(text)):
        if sent.strip():
            yield sent, [c for c in CLAUSE.split(sent) if c.strip()]


def positive_clauses(text, region):
    # Clauses naming the region together with an abnormal descriptor and no negation.
    return [(sent, c) for sent, cls in clauses(text) for c in cls if re.search(region, c) and ABN.search(c) and not NEG_IMG.search(c)]


def thalamic_side(clause):
    # Nearest side marker before "丘脑" in its list item; otherwise inherited from the nearest preceding list item.
    items = clause.split('、')
    for i, it in enumerate(items):
        p = it.find('丘脑')
        if p < 0:
            continue
        marks = SIDE.findall(it[:p])
        if not marks:
            if '对称' in it[p:]:
                return 'bilateral'
            for prev in reversed(items[:i]):
                mm = SIDE.findall(prev)
                if mm:
                    marks = [mm[0]]
                    break
        if not marks:
            return 'unspecified'
        return 'unilateral' if marks[-1] in ('左', '右') else 'bilateral'
    return None


def find_thalamic(text, bilateral):
    for sent, c in positive_clauses(text, r'丘脑'):
        side = thalamic_side(c)
        if side is None:
            continue
        if side == 'unspecified' and re.search(r'对称|双侧丘脑|两侧丘脑', sent):
            side = 'bilateral'
        if not bilateral or side == 'bilateral':
            return c.strip()
    return ''


def find_splenial(text):
    for sent, cls in clauses(text):
        if '胼胝体' in sent and '压部' in sent:
            for c in cls:
                if '压部' in c and (ABN.search(c) or DWI_POS.search(c)) and not NEG_IMG.search(c):
                    return sent.strip()
    m = re.search(r'[^。；]*(?:可逆性胼胝体压部|胼胝体压部[^，,。；]{0,8}(?:综合征|脑病|病变))[^。；]*', str(text))
    if m and not re.search(r'未见|消失|吸收', m.group(0)):
        return m.group(0).strip()
    return ''


def find_subcortical_dwi(text, region=SUBCORT):
    # Extensive (bilateral or multilobar) white-matter lesions with restricted diffusion in the same sentence (bright-tree-type pattern).
    for sent, cls in clauses(text):
        if not region.search(sent) or CHRONIC.search(sent) or UNSURE.search(sent) or not EXTENSIVE.search(sent):
            continue
        reg = any(region.search(c) and (ABN.search(c) or DWI_POS.search(c)) and not NEG_IMG.search(c) for c in cls)
        dwi = any(DWI_POS.search(c) and not NEG_IMG.search(c) for c in cls)
        if reg and dwi:
            return sent.strip()
    return ''


def find_diffuse_edema(text):
    # Global swelling, herniation or cisternal effacement; bilateral loss of gray-white differentiation counts only with effacement.
    efface = any(EFFACE.search(c) and not re.search(r'未见|无明显|无|未', c) for _, cls in clauses(text) for c in cls)
    for sent, cls in clauses(text):
        for c in cls:
            m = DIFFUSE_EDEMA.search(c)
            if m and not re.search(r'未见|无明显|无|未|排除', c[max(0, m.start() - 8):m.start()]):
                return sent.strip()
        m = GWM_LOSS.search(sent)
        if m and efface and not re.search(r'未见|无明显|无|未', sent[max(0, m.start() - 8):m.start()]):
            return sent.strip()
    return ''


def find_hemispheric(text):
    for sent, cls in clauses(text):
        if re.search(r'双侧大脑半球|两侧大脑半球', sent) or CHRONIC.search(sent):
            continue
        if HEMI.search(sent) and (re.search(r'肿胀|水肿', sent) or DWI_POS.search(sent)) and not NEG_IMG.search(sent):
            return sent.strip()
    return ''


def imaging_table(sheet, head_only):
    # The supplement holds findings + conclusion; the main export holds the report name. Merge by admission and time.
    s = supp[sheet].copy(); s['vis'] = s['科研就诊编号'].astype(str); s['time'] = pd.to_datetime(s['检查时间'], errors='coerce')
    s['text'] = s['检查所见'].fillna('').astype(str) + '。' + s['检查结论'].fillna('').astype(str)
    s['concl'] = s['检查结论'].fillna('').astype(str)
    m = sheets[sheet].copy(); m['vis'] = m['科研就诊编号'].astype(str); m['time'] = pd.to_datetime(m['检查时间'], errors='coerce')
    m['name'] = m['报告名称'].fillna('').astype(str) if '报告名称' in m else ''
    m['concl_main'] = m['检查结论'].fillna('').astype(str)
    x = s.merge(m[['vis', 'time', 'name']].drop_duplicates(['vis', 'time']), on=['vis', 'time'], how='left')
    only_main = m[~m.set_index(['vis', 'time']).index.isin(x.set_index(['vis', 'time']).index)].copy()
    only_main['text'] = only_main['concl_main']; only_main['concl'] = only_main['concl_main']
    x = pd.concat([x[['vis', 'time', 'name', 'text', 'concl']], only_main[['vis', 'time', 'name', 'text', 'concl']]], ignore_index=True)
    x['name'] = x['name'].fillna('')
    x = x.merge(idx[['vis', 'admit', 'discharge']], on='vis')
    x = x[(x.time >= x.admit - pd.Timedelta(days=1)) & (x.time <= x.discharge + pd.Timedelta(days=1))].copy()
    if head_only:
        x = x[x.name.str.contains(r'头|颅|脑', regex=True) | (x.name == '') & x.text.str.contains(r'脑室|脑沟|基底节|丘脑|小脑|中线结构')]
    x['day'] = (x.time - x.admit).dt.total_seconds() / 86400
    return x.sort_values(['vis', 'time']).reset_index(drop=True)


MRI = imaging_table('MRI报告', head_only=False)
MRI = MRI[MRI.text.str.contains(r'脑|丘脑|胼胝体|基底节|小脑|中线结构')].copy()
CT = imaging_table('CT报告', head_only=True)
FEATURES = {'thal_bilat': lambda t: find_thalamic(t, True), 'thal_any': lambda t: find_thalamic(t, False),
            'diffuse_edema': find_diffuse_edema, 'hemispheric': find_hemispheric}
for X in (MRI, CT):
    for k, f in FEATURES.items():
        X[k + '_txt'] = X.text.apply(f)
        X[k] = X[k + '_txt'].ne('')
    X['brainstem'] = X.text.apply(lambda t: bool(positive_clauses(t, r'脑干|桥脑|脑桥|延髓|中脑|大脑脚')))
for k, f in {'splenial': find_splenial, 'subcortical_dwi': find_subcortical_dwi, 'lobar_wm_dwi': lambda t: find_subcortical_dwi(t, LOBAR_WM)}.items():
    MRI[k + '_txt'] = MRI.text.apply(f)
    MRI[k] = MRI[k + '_txt'].ne('')
MRI['any_dwi'] = MRI.text.apply(lambda t: any(DWI_POS.search(c) and not NEG_IMG.search(c) for _, cls in clauses(t) for c in cls))
MRI['normal'] = MRI.concl.str.contains(r'未见明显异常|未见异常') & ~MRI[['thal_any', 'splenial', 'subcortical_dwi', 'diffuse_edema', 'any_dwi']].any(axis=1)
MRI['imp_ane'] = MRI.concl.str.contains(r'坏死性脑病') & ~MRI.concl.str.contains(r'不支持|排除|除外')
MRI['imp_mers'] = MRI.concl.str.contains(r'可逆性胼胝体|胼胝体压部[^，,。；]{0,8}(?:综合征|脑病)')
MRI['imp_aesd'] = MRI.concl.str.contains(r'双相|双向|迟发性弥散|后期弥散|AESD')
MRI['imp_ane_txt'] = np.where(MRI.imp_ane, MRI.concl.str.replace(r'\s+', ' ', regex=True).str.slice(0, 120), '')


def per_admission(X, prefix, flags):
    out = X.groupby('vis').agg(**{f'{prefix}_n': ('text', 'size'), f'{prefix}_first_day': ('day', 'min')},
                               **{f'{prefix}_{c}': (c, 'any') for c in flags})
    for c in flags:
        if c + '_txt' in X:
            pos = X[X[c]]
            out[f'{prefix}_{c}_day'] = pos.groupby('vis').day.min()
            out[f'{prefix}_{c}_evidence'] = pos.groupby('vis').apply(lambda g: f"{prefix.upper()} day {g.day.iloc[0]:.0f}: {g[c + '_txt'].iloc[0][:160]}")
    return out


mri_v = per_admission(MRI, 'mri', ['thal_bilat', 'thal_any', 'brainstem', 'diffuse_edema', 'hemispheric', 'splenial', 'subcortical_dwi',
                                    'lobar_wm_dwi', 'any_dwi', 'normal', 'imp_ane', 'imp_mers', 'imp_aesd'])
for c in ['subcortical_dwi', 'lobar_wm_dwi']:
    mri_v[f'mri_{c}_day2plus'] = MRI[MRI.day >= 2].groupby('vis')[c].any()
    mri_v[f'mri_{c}_day2plus'] = mri_v[f'mri_{c}_day2plus'].fillna(False).astype(bool)
ct_v = per_admission(CT, 'ct', ['thal_bilat', 'thal_any', 'diffuse_edema', 'hemispheric'])
print('Index admissions with brain MRI:', mri_v.shape[0], '| with head CT:', ct_v.shape[0])
print({k: int(mri_v[k].sum()) for k in mri_v.columns if mri_v[k].dtype == bool})
print({k: int(ct_v[k].sum()) for k in ct_v.columns if ct_v[k].dtype == bool})

In [ ]:
# ITES evidence 2/4: EEG, CSF, fever, altered consciousness, seizures.
eeg = supp['脑电图检查'].copy(); eeg['vis'] = eeg['科研就诊编号'].astype(str)
eeg['text'] = (eeg['背景活动'].fillna('').astype(str) + '。' + eeg['印象'].fillna('').astype(str))
eeg = eeg[eeg.vis.isin(set(idx.vis))]
EEG_SLOW = re.compile(r'慢波活动|弥漫性慢波|广泛性慢波|弥漫性\s*\d|广泛性\s*\d|背景[^，,。；]{0,10}慢|慢化|δ波|δ活动|θ活动|θ波[^，,。；]{0,4}(?:增多|为主)'
                      r'|慢波为主|慢波增多|慢波较多|大量慢波|多量慢波|慢波明显')
EEG_SUPP = re.compile(r'电静息|暴发[-—]?抑制|爆发[-—]?抑制|低平抑制|抑制性脑波|低电压|电压低平|背景抑制')
EEG_SZ = re.compile(r'癫痫持续状态|电持续状态|非惊厥|NCSE|发作期')
eeg['slow'] = eeg.text.apply(lambda t: any(EEG_SLOW.search(c) and not re.search(r'未见|少量|无明显', c) for _, cls in clauses(t) for c in cls))
eeg['suppressed'] = eeg.text.str.contains(EEG_SUPP)
eeg['seizure_pattern'] = (eeg['印象'].fillna('').astype(str).str.contains(EEG_SZ)) | \
                         (eeg['发作期'].fillna('').astype(str).str.contains(r'监测到|发作') & ~eeg['发作期'].fillna('').astype(str).str.contains(r'未见特殊|未发生'))
eeg_v = eeg.groupby('vis').agg(eeg_n=('text', 'size'), eeg_slow=('slow', 'any'), eeg_suppressed=('suppressed', 'any'), eeg_seizure=('seizure_pattern', 'any'))

csf = pd.concat([sheets['实验室检查_脑脊液常规'], supp['实验室检查_脑脊液常规']]).drop_duplicates('报告编号')
csf['vis'] = csf['就诊编号'].astype(str); csf['wbc'] = parse_lab_value(csf['白细胞计数定量-定量结果'], qual_col(csf, '白细胞计数定量-定量结果'))[0]
csf_v = csf[csf.vis.isin(set(idx.vis))].groupby('vis').agg(csf_wbc_max=('wbc', 'max'))
cbio = pd.concat([sheets['实验室检查_脑脊液生化'], supp['实验室检查_脑脊液生化']]).drop_duplicates('报告编号')
cbio['vis'] = cbio['就诊编号'].astype(str)
prot = parse_lab_value(cbio['脑脊液蛋白定量-定量结果'], qual_col(cbio, '脑脊液蛋白定量-定量结果'))[0]
unit = cbio['脑脊液蛋白定量-定量结果单位'].astype(str).str.lower()
cbio['protein_gL'] = np.where(unit.str.contains('mg'), prot / 1000, prot)
csf_v = csf_v.join(cbio[cbio.vis.isin(set(idx.vis))].groupby('vis').protein_gL.max().rename('csf_protein_max'), how='outer')
vd = supp['实验室检查_脑脊液病毒DNA测定'].copy(); vd['vis'] = vd['就诊编号'].astype(str)
qual = [c for c in vd.columns if c.endswith('-定性结果')]
vd['positive'] = vd[qual].astype(str).apply(lambda r: r.str.contains('阳').any(), axis=1)
csf_v = csf_v.join(vd[vd.vis.isin(set(idx.vis))].groupby('vis').positive.any().rename('csf_virus_positive'), how='outer')

an = supp['儿科入院记录'].copy(); an['vis'] = an['科研就诊编号'].astype(str); an = an[an.vis.isin(set(idx.vis))]
an = an.groupby('vis').agg(cc=('主诉', lambda s: ' '.join(map(str, s.dropna()))), hpi=('现病史', lambda s: ' '.join(map(str, s.dropna()))),
                           pmh=('既往史', lambda s: ' '.join(map(str, s.dropna()))), pe=('体格检查', lambda s: ' '.join(map(str, s.dropna()))))
FEVER = re.compile(r'发热|发烧|高热|低热|中热|热峰|体温[^，,。；]{0,6}(?:升高|最高|达3[89]|达4)')
ENC = re.compile(r'昏迷|嗜睡|昏睡|意识障碍|意识不清|意识模糊|神志不清|神志模糊|神志恍惚|神志昏睡|神志嗜睡|神志浅昏迷|呼之不应|反应差|反应迟钝'
                 r'|反应低下|精神萎靡|精神反应差|精神差|谵妄|胡言乱语|烦躁不安|易激惹|GCS|格拉斯哥|淡漠|不认人|行为异常|精神行为异常|意识改变')
SEIZ = re.compile(r'抽搐|惊厥|抽动')
NEG_TXT = re.compile(r'无|不伴|否认|未|没有|不详')


def mentions(text, pat):
    return any(pat.search(c) and not NEG_TXT.search(c[:max(0, pat.search(c).start())][-6:]) for _, cls in clauses(text) for c in cls)


ev = idx[['vis', 'patient', 'subtype', 'age_y', 'admit', 'discharge', 'icu', 'death', 'poor', 'dx_text', 'course']].copy()
ev = ev.merge(an, left_on='vis', right_index=True, how='left')
for c in ['cc', 'hpi', 'pmh', 'pe']:
    ev[c] = ev[c].fillna('')
vit = supp['患者生命体征'].copy(); vit['vis'] = vit['科研就诊编号'].astype(str); vit = vit[vit.vis.isin(set(idx.vis))]
vit['T'] = pd.to_numeric(vit['体温(°C)'], errors='coerce'); vit['t'] = pd.to_datetime(vit['测量时间'], errors='coerce')
vit = vit.merge(idx[['vis', 'admit']], on='vis')
early = vit[(vit.t - vit.admit).dt.total_seconds().between(-86400, 3 * 86400)]
ev['tmax_72h'] = ev.vis.map(early.groupby('vis')['T'].max())
ev['fever_text'] = [mentions(a + '。' + b, FEVER) for a, b in zip(ev.cc, ev.hpi)]
ev['fever_course'] = ev.course.apply(lambda t: mentions(t[:400], FEVER))
ev['fever'] = ev.fever_text | ev.fever_course | (ev.tmax_72h >= 38.0)
ev['encephalopathy_text'] = [mentions(a + '。' + b + '。' + c, ENC) or mentions(d[:600], ENC) for a, b, c, d in zip(ev.cc, ev.hpi, ev.pe, ev.course)]
ev['seizure_text'] = [mentions(a + '。' + b, SEIZ) or mentions(c[:600], SEIZ) for a, b, c in zip(ev.cc, ev.hpi, ev.course)]
ev['biphasic_text'] = (ev.hpi + ev.course).str.contains(r'再次抽搐|再发抽搐|再次出现抽搐|又出现抽搐|再次惊厥|再次发作|双相|双向|第二次抽搐')
ev['prior_epilepsy'] = ev.pmh.apply(lambda t: mentions(t, re.compile(r'癫痫(?!持续)|婴儿痉挛'))) | \
                       ev.hpi.str.contains(r'诊断[“"]?癫痫|癫痫病史|既往.{0,8}癫痫')
ev['prior_neuro_disease'] = ev.pmh.apply(lambda t: mentions(t, re.compile(r'脑瘫|发育落后|发育迟缓|智力低下|遗传代谢|线粒体|脑积水')))
for t in (mri_v, ct_v, eeg_v, csf_v):
    ev = ev.merge(t, left_on='vis', right_index=True, how='left')
print('Fever documented:', int(ev.fever.sum()), '| altered consciousness documented:', int(ev.encephalopathy_text.sum()),
      '| EEG available:', int(ev.eeg_n.notna().sum()), '| CSF WBC available:', int(ev.csf_wbc_max.notna().sum()))

In [ ]:
# ITES evidence 3/4: shock and the six multi-organ items used for ASEM (first 72 h of admission).
def first72(sheet, col, how):
    parts = []
    for book in (sheets, supp):
        if sheet in book and col + '-定量结果' in book[sheet].columns:
            d = book[sheet]
            parts.append(pd.DataFrame({'report': d['报告编号'].astype(str), 'vis': d['就诊编号'].astype(str),
                                       'v': parse_lab_value(d[col + '-定量结果'], qual_col(d, col + '-定量结果'))[0],
                                       'uln': pd.to_numeric(d.get(col + '-正常高值'), errors='coerce'),
                                       'lln': pd.to_numeric(d.get(col + '-正常低值'), errors='coerce'),
                                       't': pd.to_datetime(d['报告时间'], errors='coerce')}))
    x = pd.concat(parts).drop_duplicates('report').dropna(subset=['v'])
    x = x.merge(idx[['vis', 'admit']], on='vis')
    x = x[(x.t - x.admit).dt.total_seconds().between(-12 * 3600, 72 * 3600)]
    return x.groupby('vis').apply(how)


ev['hb_min'] = ev.vis.map(first72('实验室检查_血常规', '血红蛋白浓度定量', lambda g: g.v.min()))
ev['plt_min'] = ev.vis.map(first72('实验室检查_血常规', '血小板计数定量', lambda g: g.v.min()))
ev['inr_max'] = ev.vis.map(first72('实验室检查_凝血功能检查', '国际标准比率定量', lambda g: g.v.max()))
ev['fib_low'] = ev.vis.map(first72('实验室检查_凝血功能检查', '纤维蛋白原检测定量', lambda g: bool((g.v < g.lln).any())))
ev['ph_min'] = ev.vis.map(first72('实验室检查_血气分析', '酸碱度定量', lambda g: g.v.min()))
ev['be_min'] = ev.vis.map(first72('实验室检查_血气分析', '碱剩余定量', lambda g: g.v.min()))
ev['alt_x_uln'] = ev.vis.map(first72('实验室检查_血液肝功能测定', '谷丙转氨酶定量', lambda g: (g.v / g.uln).max()))
ev['ast_x_uln'] = ev.vis.map(first72('实验室检查_血液肝功能测定', '谷草转氨酶定量', lambda g: (g.v / g.uln).max()))
ev['cr_x_uln'] = ev.vis.map(first72('实验室检查_血液肾功能测定', '肌酐定量', lambda g: (g.v / g.uln).max()))
ev['ammonia_max'] = ev.vis.map(first72('实验室检查_血氨AMMO', '血氨AMMO定量', lambda g: g.v.max()))
ev['na_max'] = ev.vis.map(first72('实验室检查_血液电解质测定', '钠定量', lambda g: g.v.max()))
dx = ev.dx_text.fillna('')
MOF = {
    'anemia': ev.hb_min < 100,                                                         # Hb < 100 g/L
    'thrombocytopenia': ev.plt_min < 100,                                              # platelets < 100 x 10^9/L
    'dic': ((ev.inr_max >= 1.5) & ev.fib_low.fillna(False).astype(bool)) | dx.str.contains(r'弥散性血管内凝血|DIC'),
    'acidosis': (ev.ph_min < 7.30) | (ev.be_min <= -8),
    'liver': (ev.alt_x_uln > 2) | (ev.ast_x_uln > 2),                                   # > 2x age-specific ULN
    'renal': (ev.cr_x_uln > 1) | dx.str.contains(r'急性肾损伤|急性肾衰|急性肾功能'),        # > age-specific ULN
}
for k, v in MOF.items():
    ev[f'mof_{k}'] = v.fillna(False).astype(bool)
ev['mof_count'] = ev[[f'mof_{k}' for k in MOF]].sum(axis=1)
vaso72 = idx.set_index('vis').vaso_time.sub(idx.set_index('vis').admit).dt.total_seconds().le(72 * 3600)
ev['shock'] = ev.vis.map(vaso72).fillna(False).astype(bool) | dx.str.contains(r'休克') & ~dx.str.contains(r'失血性休克|心源性休克')
ev['septic_shock'] = dx.str.contains(r'脓毒性休克|感染性休克|脓毒症休克')
ev['sepsis_dx'] = dx.str.contains(r'脓毒症|败血症|脓毒性休克|感染性休克')
ev['cpr'] = ev.vis.map(idx.set_index('vis').cpr_any).fillna(False).astype(bool)
print('MOF items:', {k: int(ev[f'mof_{k}'].sum()) for k in MOF}, '| shock:', int(ev.shock.sum()), '| septic shock dx:', int(ev.septic_shock.sum()))

In [ ]:
# ITES evidence 4/4: alternative diagnoses that the consensus requires to be excluded.
EXCLUSIONS = {   # category: (pattern on discharge diagnoses of the index admission, applies to imaging-defined subtypes too)
    'CNS infection (bacterial/TB/fungal) or specific viral encephalitis': (r'化脓性脑膜炎|细菌性脑膜炎|结核性脑膜炎|隐球菌|真菌性脑膜炎|脑脓肿|疱疹病毒性脑炎|单纯疱疹病毒性脑炎|单纯疱疹性脑炎|HSV脑炎|乙型脑炎|流行性乙型', True),
    'Autoimmune or demyelinating disease': (r'自身免疫性脑炎|自体免疫性脑炎|免疫介导性脑炎|NMDA|GABA|LGI1|MOG|ADEM|急性播散性脑脊髓炎|视神经脊髓炎|多发性硬化|桥本脑病', True),
    'Metabolic disease, Reye syndrome or IEM': (r'瑞氏|Reye|瑞夷|线粒体|Leigh|亚急性坏死性|甲基丙二酸|戊二酸|有机酸血症|尿素循环|遗传代谢|先天性代谢|代谢性脑病'
                                              r'|低血糖脑病|低血糖昏迷|肝性脑病|尿毒症|Wernicke|韦尼克|维生素B1缺乏', True),
    'Hypoxic-ischemic injury, drowning or asphyxia': (r'溺水|窒息|缺氧缺血性脑病|缺血缺氧性脑病|缺氧性脑病|缺氧缺血性脑损伤', True),
    'Trauma': (r'颅脑损伤|颅脑外伤|脑挫|硬膜外血肿|外伤性|颅骨骨折|虐待', True),
    'Poisoning': (r'一氧化碳|药物中毒|农药|有机磷|毒蕈|蘑菇中毒|鼠药|酒精中毒|乙醇中毒|铅中毒|汞中毒|百草枯|药物过量', True),
    'Cerebrovascular disease or tumor': (r'脑梗|静脉窦血栓|烟雾病|颅内动脉瘤|动静脉畸形|血管炎|脑肿瘤|母细胞瘤|胶质瘤|室管膜瘤|生殖细胞瘤|颅内占位', True),
    'PRES or hypertensive encephalopathy': (r'可逆性后部|PRES|高血压脑病', True),
    'Systemic inflammatory disease (HLH, Kawasaki disease, MIS-C, SLE)': (r'噬血|HLH|巨噬细胞活化|川崎|多系统炎症综合征|MIS-C|系统性红斑狼疮|狼疮', False),
    'Epilepsy or developmental and epileptic encephalopathy': (r'癫痫(?!持续)|婴儿痉挛|West综合征|Dravet|Lennox', False),
    'Malignancy or chemotherapy': (r'白血病|淋巴瘤|神经母细胞瘤|化疗', False),
}
UNCERTAIN = re.compile(r'[？?]|待排|待查|可能|不除外|疑似|可疑')
dx_items = ev.dx_text.fillna('').str.split(r'\s*\|\s*')


def excl_status(items, pat):
    # 'definite' when an item names the diagnosis without an uncertainty marker; 'possible' when only qualified mentions exist.
    hits = [x for x in items if re.search(pat, x)]
    if not hits:
        return ''
    return 'definite' if any(not UNCERTAIN.search(x) for x in hits) else 'possible'


for k, (p, _) in EXCLUSIONS.items():
    st_ = dx_items.apply(lambda L: excl_status(L, p))
    ev['excl: ' + k] = st_.eq('definite')
    ev['possible excl: ' + k] = st_.eq('possible')
ev['excl: Epilepsy or developmental and epileptic encephalopathy'] |= ev.prior_epilepsy
FIRES_LABEL = ev.subtype.eq('FIRES') | dx.str.contains(r'FIRES|热性感染相关|发热感染相关|感染相关性癫痫综合征')
ev.loc[FIRES_LABEL, 'excl: Epilepsy or developmental and epileptic encephalopathy'] = False   # epilepsy is the FIRES phenotype itself
ev['neonate'] = ev.age_y < 28 / 365
ev['csf_pleocytosis'] = ev.csf_wbc_max > 20
ev['csf_virus_positive'] = ev.csf_virus_positive.fillna(False).astype(bool)
HARD = ['excl: ' + k for k, (_, hard) in EXCLUSIONS.items() if hard]
SOFT = ['excl: ' + k for k, (_, hard) in EXCLUSIONS.items() if not hard]
print('Exclusion flags:', {k.replace('excl: ', ''): int(ev[k].sum()) for k in HARD + SOFT})
print('CSF WBC > 20 x10^6/L:', int(ev.csf_pleocytosis.sum()), '| CSF virus PCR positive:', int(ev.csf_virus_positive.sum()), '| neonates:', int(ev.neonate.sum()))

In [ ]:
# ITES assignment (hierarchical; every step is recorded in `ites_reason`).
def b(row, col):
    v = row.get(col)
    return bool(v) if pd.notna(v) else False


def assign(row):
    hard = [c.replace('excl: ', '') for c in HARD if row[c]]
    soft = [c.replace('excl: ', '') for c in SOFT if row[c]]
    thal = b(row, 'mri_thal_bilat') or b(row, 'ct_thal_bilat') or b(row, 'mri_imp_ane')
    thal_day = np.nanmin([row.get('mri_thal_bilat_day', np.nan), row.get('ct_thal_bilat_day', np.nan), np.nan if not b(row, 'mri_imp_ane') else row.get('mri_first_day', np.nan)])
    edema = b(row, 'mri_diffuse_edema') or b(row, 'ct_diffuse_edema')
    mers_img = b(row, 'mri_splenial') or b(row, 'mri_imp_mers')
    aesd_img = b(row, 'mri_subcortical_dwi') or b(row, 'mri_imp_aesd') or (b(row, 'mri_lobar_wm_dwi_day2plus') and not mers_img) or \
        (row.subtype == 'AESD' and b(row, 'mri_any_dwi'))
    hemi = b(row, 'mri_hemispheric') or b(row, 'ct_hemispheric')
    has_neuroimaging = pd.notna(row.get('mri_n')) or pd.notna(row.get('ct_n'))
    fires = row.subtype == 'FIRES' or FIRES_LABEL.loc[row.name]
    flag = '' if row.fever else ' [fever not documented]'
    if row.neonate:
        return 'Non-ITES: neonatal encephalopathy', 'Neonate (<28 days)'
    if hard:
        return 'Non-ITES: alternative diagnosis', '; '.join(hard)
    if row.csf_virus_positive or row.csf_pleocytosis:
        return 'Non-ITES: encephalitis/meningitis', f'CSF WBC max {row.csf_wbc_max:g} x10^6/L' if row.csf_pleocytosis else 'CSF viral PCR positive'
    if thal and not (fires and not thal_day <= 7):
        return 'ANE', ('Bilateral thalamic lesions' if (b(row, 'mri_thal_bilat') or b(row, 'ct_thal_bilat')) else 'Radiologist impression of ANE') + flag
    if fires:
        return 'FIRES', 'Clinician-diagnosed FIRES' + flag
    if edema:
        if row.septic_shock:
            return 'SAE', 'Diffuse cerebral edema with septic shock (septic shock excludes AFCE/ASEM)'
        if row.shock and row.mof_count >= 3:
            return 'ASEM', f'Diffuse cerebral edema, shock, {int(row.mof_count)} of 6 organ items' + flag
        return 'AFCE', 'Diffuse cerebral edema' + flag
    if aesd_img:
        late = b(row, 'mri_subcortical_dwi_day2plus') or b(row, 'mri_lobar_wm_dwi_day2plus')
        return 'AESD', 'White-matter restricted diffusion' + (' (MRI on day >= 2)' if late else ' (day of MRI < 2: check timing)') + flag
    if mers_img:
        if 'Systemic inflammatory disease (HLH, Kawasaki disease, MIS-C, SLE)' in soft:
            return 'Non-ITES: alternative diagnosis', 'Splenial lesion with systemic inflammatory disease (MERS exclusion)'
        return 'MERS', 'Splenial lesion' + (' with restricted diffusion' if b(row, 'mri_any_dwi') else ' (diffusion not stated)') + flag
    if hemi and row.seizure_text:
        return 'HHE', 'Unilateral hemispheric swelling after seizures' + flag
    if row.subtype in ('ANE', 'MERS', 'AESD') and not has_neuroimaging:
        return row.subtype, 'Clinician-diagnosed; no neuroimaging report in the export' + flag
    if row.septic_shock or row.subtype == 'Septic_encephalopathy':
        return 'SAE', 'Septic shock or clinician-labeled septic encephalopathy, no ITES-specific imaging'
    if soft:
        return 'Non-ITES: alternative diagnosis', '; '.join(soft)
    if not row.fever:
        return 'Unclassifiable', 'No documented febrile illness'
    eeg_enceph = b(row, 'eeg_slow') or b(row, 'eeg_suppressed')
    if eeg_enceph and row.encephalopathy_text:
        return 'Possible ITES', 'Fever, altered consciousness, encephalopathic EEG, no syndrome-specific imaging'
    if eeg_enceph:
        return 'Unclassifiable', 'Fever and encephalopathic EEG, but altered consciousness not documented; no syndrome-specific imaging'
    if pd.isna(row.get('eeg_n')) and not has_neuroimaging:
        return 'Unclassifiable', 'No EEG and no neuroimaging'
    return 'Unclassifiable', 'No syndrome-specific imaging and no encephalopathic EEG documented'


ev[['ites_class', 'ites_reason']] = ev.apply(lambda r: pd.Series(assign(r)), axis=1)
TIER = {'ANE': 'Definite ITES', 'ASEM': 'Definite ITES', 'AFCE': 'Definite ITES', 'AESD': 'Definite ITES', 'MERS': 'Definite ITES',
        'HHE': 'ITES-related', 'FIRES': 'ITES-related', 'Possible ITES': 'Possible ITES', 'SAE': 'Sepsis-associated encephalopathy',
        'Unclassifiable': 'Unclassifiable'}
ev['ites_tier'] = ev.ites_class.map(TIER).fillna('Non-ITES')
ev['ites_algorithmic'] = ev.ites_class
CLIN = {'ANE': 'ANE', 'MERS': 'MERS', 'AESD': 'AESD', 'FIRES': 'FIRES', 'Septic_encephalopathy': 'SAE'}
ev['clinician_label'] = ev.subtype.map(CLIN).fillna('Other/unspecified')
specific_img = ev.ites_reason.str.contains(r'thalamic|edema|white-matter|Splenial|hemispheric', case=False)
conflict_excl = ev.ites_class.str.startswith('Non-ITES') & (ev.clinician_label.isin(['ANE', 'MERS', 'AESD', 'FIRES']) |
                 (ev.mri_thal_bilat.fillna(False).astype(bool) | ev.mri_splenial.fillna(False).astype(bool)))
possible_excl = ev[[c for c in ev.columns if c.startswith('possible excl: ')]].any(axis=1)
ev['review_reasons'] = [
    '; '.join(r for r, f in [
        ('clinician label differs from algorithm', cl not in ('Other/unspecified',) and cl != ic),
        ('exclusion conflicts with ITES-type imaging or clinician ITES label', ce),
        ('fever not documented', 'fever not documented' in rs),
        ('AESD imaging before day 2', 'check timing' in rs),
        ('splenial lesion without stated restricted diffusion', 'diffusion not stated' in rs),
        ('clinician label without neuroimaging in export', 'no neuroimaging report' in rs),
        ('qualified (?) alternative diagnosis present', pe and not ic.startswith('Non-ITES')),
        ('cardiac arrest/CPR documented', cp and ic in ('ANE', 'ASEM', 'AFCE', 'AESD')),
        ('altered consciousness not documented', (not en) and ic in ('ANE', 'ASEM', 'AFCE', 'AESD', 'MERS', 'HHE', 'FIRES')),
    ] if f)
    for cl, ic, ce, rs, pe, cp, en in zip(ev.clinician_label, ev.ites_class, conflict_excl, ev.ites_reason, possible_excl, ev.cpr, ev.encephalopathy_text)]
ev['needs_review'] = ev.review_reasons.ne('')
ADJ = HERE / 'output_ites' / 'ites_adjudicated.csv'
if ADJ.exists():
    adj = pd.read_csv(ADJ, dtype={'vis': str}).dropna(subset=['adjudicated_class'])
    ev = ev.merge(adj[['vis', 'adjudicated_class']], on='vis', how='left')
    ev['ites_class'] = ev.adjudicated_class.fillna(ev.ites_class)
    ev['ites_tier'] = ev.ites_class.map(TIER).fillna('Non-ITES')
    print('Adjudicated categories applied:', int(ev.adjudicated_class.notna().sum()))
else:
    print('No adjudication file found: algorithmic categories are used.')
print(ev.ites_class.value_counts().to_string())

In [ ]:
# Operational criteria (English summary of every rule used above; exported for the supplement).
CRITERIA = pd.DataFrame([
    ('Population', 'Index admission', 'First acute-phase admission per patient with an acute encephalopathy diagnosis and a recorded discharge disposition.'),
    ('Core criterion', 'Febrile illness', 'Fever in the chief complaint or history of present illness (clause-level negation), in the first 400 characters of the hospital course, or a recorded temperature ≥ 38.0°C from 24 h before to 72 h after admission. Required for possible ITES; for syndromes defined by neuroimaging or FIRES, absence is flagged for adjudication rather than used to exclude, because the consensus criterion also accepts infection without fever and documentation is incomplete in retrospective records.'),
    ('Core criterion', 'Altered consciousness', 'Coma, stupor, somnolence, lethargy, confusion, delirium, irritability or GCS documentation in the admission note or hospital course (clause-level negation). Required for possible ITES; for syndromes defined by neuroimaging or FIRES, absence is flagged for adjudication.'),
    ('Exclusion', 'Neonate', 'Age < 28 days.'),
    ('Exclusion', 'Alternative diagnoses', 'Unqualified discharge diagnosis of: bacterial/tuberculous/fungal CNS infection or HSV/Japanese encephalitis; autoimmune encephalitis, ADEM or other demyelinating disease; metabolic encephalopathy, inborn error of metabolism, Reye syndrome, hepatic or uremic encephalopathy, HUS; hypoxic-ischemic injury, drowning or asphyxia; trauma; poisoning; cerebral infarction, venous sinus thrombosis, vasculitis or tumor; PRES/hypertensive encephalopathy. Diagnoses qualified by "?" or "possible" are flagged for review only.'),
    ('Exclusion', 'Encephalitis by CSF', 'CSF white-cell count > 20 × 10^6/L at any lumbar puncture during the admission, or a positive CSF viral PCR.'),
    ('Exclusion (soft)', 'Systemic inflammatory disease, epilepsy, malignancy', 'HLH/MAS, Kawasaki disease, MIS-C or SLE; known epilepsy or developmental and epileptic encephalopathy (except FIRES); leukemia/lymphoma/chemotherapy. Applied only when no syndrome-specific neuroimaging is present (Kawasaki disease also excludes MERS).'),
    ('Subtype', 'ANE', 'Bilateral thalamic lesions on MRI or CT (side-aware parsing of the report), or a radiologist impression of acute necrotizing encephalopathy.'),
    ('Subtype', 'FIRES', 'Clinician diagnosis of FIRES; takes precedence over thalamic lesions first seen after day 7.'),
    ('Subtype', 'ASEM', 'Diffuse cerebral edema plus shock (vasoactive infusion within 72 h or shock diagnosis, not hemorrhagic/cardiogenic) plus ≥ 3 of: Hb < 100 g/L; platelets < 100 × 10^9/L; DIC (INR ≥ 1.5 with fibrinogen below the reference limit, or DIC diagnosis); acidosis (pH < 7.30 or base excess ≤ −8 mmol/L); ALT or AST > 2x age-specific upper limit; creatinine above the age-specific upper limit or AKI diagnosis (first 72 h).'),
    ('Subtype', 'AFCE', 'Diffuse cerebral edema (global swelling, tonsillar herniation, cisternal effacement, or bilateral loss of gray-white differentiation with effacement) not meeting ASEM; septic shock diagnosis reassigns to SAE.'),
    ('Subtype', 'AESD', 'Bilateral or multilobar subcortical white-matter restricted diffusion (bright-tree-type), lobar white-matter restricted diffusion on MRI from day 2 without a splenial lesion, or clinician-diagnosed AESD with any restricted diffusion.'),
    ('Subtype', 'MERS', 'Splenial lesion of the corpus callosum on MRI (or radiologist impression of MERS/RESLES); restricted diffusion recorded when stated.'),
    ('Subtype', 'HHE', 'Unilateral hemispheric swelling or restricted diffusion after seizures.'),
    ('Non-ITES comparator', 'SAE', 'Septic shock diagnosis or clinician-labeled septic encephalopathy without ANE, FIRES, AESD, MERS, or HHE features; children with diffuse cerebral edema and septic shock are also classified as SAE because septic shock is an exclusion for AFCE and ASEM.'),
    ('Tier', 'Possible ITES', 'Febrile illness, altered consciousness and an encephalopathic EEG (diffuse slowing or suppression) without syndrome-specific imaging or exclusions.'),
    ('Tier', 'Unclassifiable', 'No documented febrile illness; fever and an encephalopathic EEG without documented altered consciousness; or neither syndrome-specific imaging nor an encephalopathic EEG.'),
    ('Review', 'Flags', 'Clinician label differs from algorithm; exclusion conflicting with ITES-type imaging or clinician ITES label; fever not documented; AESD imaging before day 2; splenial lesion without stated restricted diffusion; clinician label without neuroimaging in the export; qualified alternative diagnosis; cardiac arrest/CPR with ANE/AFCE/ASEM/AESD; altered consciousness not documented for an imaging-defined syndrome or FIRES.'),
], columns=['domain', 'item', 'operational definition'])
CRITERIA

In [ ]:
# Classification summary, agreement with clinician labels, era and outcome profile.
OUT_ITES = OUT
TIER_ORDER = ['Definite ITES', 'ITES-related', 'Possible ITES', 'Sepsis-associated encephalopathy', 'Non-ITES', 'Unclassifiable']
CLASS_ORDER = ['ANE', 'ASEM', 'AFCE', 'AESD', 'MERS', 'HHE', 'FIRES', 'Possible ITES', 'SAE', 'Non-ITES: alternative diagnosis',
               'Non-ITES: encephalitis/meningitis', 'Non-ITES: neonatal encephalopathy', 'Unclassifiable']
ev['era'] = pd.cut(ev.admit, [pd.Timestamp('2000-01-01'), pd.Timestamp('2020-01-01'), pd.Timestamp('2022-12-01'),
                              pd.Timestamp('2023-02-01'), pd.Timestamp('2100-01-01')],
                   labels=['Pre-2020', '2020-Nov 2022', 'Dec 2022-Jan 2023', 'Feb 2023 onward'], right=False)
ev['critical_ama'] = ev.vis.map(idx.set_index('vis').critical_ama)
summary = (ev.groupby('ites_class')
             .agg(tier=('ites_tier', 'first'), N=('vis', 'size'), age_median=('age_y', 'median'), icu=('icu', 'sum'),
                  deaths=('death', 'sum'), death_or_critical_ama=('poor', 'sum'), needs_review=('needs_review', 'sum'))
             .reindex([c for c in CLASS_ORDER if c in set(ev.ites_class)]))
summary['% of cohort'] = [float(rnd(100 * n / len(ev), 1)) for n in summary.N]
summary['age_median'] = [float(rnd(v, 1)) for v in summary.age_median]
agreement = pd.crosstab(ev.ites_class, ev.clinician_label, margins=True).reindex([c for c in CLASS_ORDER if c in set(ev.ites_class)] + ['All'])
by_era = pd.crosstab(ev.ites_class, ev.era).reindex([c for c in CLASS_ORDER if c in set(ev.ites_class)])
excl_counts = pd.DataFrame({'definite (unqualified) diagnosis': {k: int(ev['excl: ' + k].sum()) for k in EXCLUSIONS},
                            'qualified (?) diagnosis only': {k: int(ev['possible excl: ' + k].sum()) for k in EXCLUSIONS}})
lab_map = {'ANE': 'ANE', 'MERS': 'MERS', 'AESD': 'AESD', 'FIRES': 'FIRES', 'SAE': 'SAE'}
spec = ev[ev.clinician_label.isin(lab_map)]
print('Clinician-labeled specific syndromes:', len(spec), '| algorithm agrees:', int((spec.clinician_label == spec.ites_class).sum()))
print('Admissions flagged for review:', int(ev.needs_review.sum()), 'of', len(ev))
summary

In [ ]:
# ITES summary tables for the supplement (class summary, agreement with clinician labels, era, exclusions, criteria; no identifiers).
with pd.ExcelWriter(OUT / 'Tables_ITES.xlsx', engine='openpyxl') as xw:
    summary.reset_index().to_excel(xw, sheet_name='Class_summary', index=False)
    agreement.to_excel(xw, sheet_name='Agreement_vs_clinician')
    by_era.to_excel(xw, sheet_name='Class_by_era')
    excl_counts.to_excel(xw, sheet_name='Exclusion_diagnoses')
    CRITERIA.to_excel(xw, sheet_name='Operational_criteria', index=False)
print('Saved', (OUT / 'Tables_ITES.xlsx').relative_to(HERE))

## Part 3. Analytic cohorts, analyses, and figures

Figures are saved to `output_publication/figures/` as they are drawn (Figure 1, cardiac phenotype by syndrome group;
Figure 3, dose-response; Figure 2, forest plot; eFigure 1, neuroimaging and EEG correlates). The analysis tables and the
numbers quoted in the manuscript are saved to `output_publication/analysis/` at the end of this part.

In [ ]:
# Syndrome groups from the ITES re-classification (used in all tables, figures and models).
GROUP_ITES = {'ANE': 'ANE', 'MERS': 'MERS', 'AESD': 'Other ITES', 'AFCE': 'Other ITES', 'ASEM': 'Other ITES', 'HHE': 'Other ITES',
              'FIRES': 'Other ITES', 'Possible ITES': 'Possible ITES', 'SAE': 'SAE', 'Unclassifiable': 'Unclassifiable'}
ITES_COLS = ['vis', 'ites_class', 'ites_tier', 'ites_algorithmic', 'needs_review', 'mri_n', 'ct_n', 'eeg_n', 'mri_thal_bilat', 'ct_thal_bilat',
             'mri_brainstem', 'mri_diffuse_edema', 'ct_diffuse_edema', 'eeg_suppressed']
idx = idx.drop(columns=[c for c in ITES_COLS[1:] if c in idx.columns]).merge(ev[ITES_COLS], on='vis', how='left')
idx['group'] = idx.ites_class.map(GROUP_ITES).fillna('Non-ITES')
idx['group3'] = idx.group.where(idx.group.isin(['ANE', 'SAE']), 'Other')
idx['ites_any'] = idx.ites_tier.isin(['Definite ITES', 'ITES-related', 'Possible ITES'])
print(idx.group.value_counts().to_string())

In [ ]:
# Analytic populations and cohort flow.
icu = idx[idx.icu].copy()
P_TNT = icu[icu.tnt_early.notna() & ~icu.arrest_before_tnt].copy()
P_TNT['injury'] = P_TNT.tnt_early > HSCTN_URL
P_TNT['injury_alt'] = P_TNT.tnt_early > HSCTN_URL_ALT
P_TNT['log2_tnt'] = np.log2(P_TNT.tnt_early_ngL.clip(lower=1))
P_EF = icu[icu.lvef_early_min.notna() & ~icu.arrest_before_echo].copy()
P_EF['dysfunction'] = P_EF.lvef_early_min < 55
P_QT = icu[icu.qtcf_early.notna() & ~icu.arrest_before_ecg].copy()
P_QT['qt_prolonged'] = P_QT.qtcf_early >= 460
P_QT['qtb_prolonged'] = P_QT.qtcb_early >= 460
P_BNP = icu[icu.bnp_early.notna() & ~icu.arrest_before_bnp].copy()
P_BNP['log2_bnp'] = np.log2(P_BNP.bnp_early.clip(lower=1))

# Time anchoring at the first hs-cTn report: organ support already present vs started afterwards.
tt = P_TNT.tnt_early_time
P_TNT['vaso_before'] = P_TNT.vaso_time.notna() & (P_TNT.vaso_time <= tt)
P_TNT['vent_before'] = P_TNT.vent_time.notna() & (P_TNT.vent_time <= tt)
P_TNT['support_before'] = P_TNT.vaso_before | P_TNT.vent_before
P_TNT['vaso_after'] = P_TNT.vaso_time.notna() & (P_TNT.vaso_time > tt)
P_TNT['pressor_after'] = P_TNT.pressor_time.notna() & (P_TNT.pressor_time > tt)
P_TNT['vent_after'] = P_TNT.vent_time.notna() & (P_TNT.vent_time > tt)
P_DET = P_TNT[~P_TNT.support_before].copy()
WINDOW_H = 168   # primary definition: escalation within 7 days of the first hs-cTn report
def started_within(col, hours):
    h = (P_DET[col] - P_DET.tnt_early_time).dt.total_seconds() / 3600
    return (h > 0) & (h <= hours)
P_DET['vaso_7d'] = started_within('vaso_time', WINDOW_H)
P_DET['pressor_7d'] = started_within('pressor_time', WINDOW_H)
P_DET['vent_7d'] = started_within('vent_time', WINDOW_H)
hrs_to_discharge = (P_DET.discharge - P_DET.tnt_early_time).dt.total_seconds() / 3600   # death or critical discharge happens at discharge
P_DET['poor_7d'] = P_DET.poor.astype(bool) & (hrs_to_discharge <= WINDOW_H)
P_DET['poor_72h'] = P_DET.poor.astype(bool) & (hrs_to_discharge <= 72)
P_DET['deterioration'] = P_DET.vaso_7d | P_DET.vent_7d | P_DET.poor_7d
P_DET['deterioration_72h'] = started_within('vaso_time', 72) | started_within('vent_time', 72) | P_DET.poor_72h
P_DET['deterioration_any'] = P_DET.vaso_after | P_DET.vent_after | P_DET.poor.astype(bool)
esc = pd.concat([P_DET.vaso_time.where(P_DET.vaso_after), P_DET.vent_time.where(P_DET.vent_after)], axis=1).min(axis=1)
P_DET['hours_to_escalation'] = (esc - P_DET.tnt_early_time).dt.total_seconds() / 3600

flow = pd.DataFrame([
    ('AES patients (any admission)', len(coh)),
    ('Acute-phase index admission', int(coh.acute_index.sum())),
    ('Index admission with recorded disposition (analysis base)', len(idx)),
    ('ICU stay during index admission', len(icu)),
    ('ICU: hs-cTn within 48 h of admission', int(icu.tnt_early.notna().sum())),
    ('ICU: hs-cTn excluded for cardiac arrest before or on the day of measurement', int((icu.tnt_early.notna() & icu.arrest_before_tnt).sum())),
    ('Primary hs-cTn cohort', len(P_TNT)),
    ('LVEF cohort (echo within 72 h, arrest rule applied)', len(P_EF)),
    ('QTc cohort (ECG within 48 h, arrest rule applied)', len(P_QT)),
    ('NT-proBNP cohort (within 48 h, arrest rule applied)', len(P_BNP)),
    ('Primary cohort without organ support at the first hs-cTn report (deterioration cohort)', len(P_DET)),
], columns=['step', 'n'])
flow['events (death or critical AMA)'] = [np.nan, np.nan, int(idx.poor.sum()), int(icu.poor.sum()),
    int(icu[icu.tnt_early.notna()].poor.sum()), np.nan, int(P_TNT.poor.sum()), int(P_EF.poor.sum()),
    int(P_QT.poor.sum()), int(P_BNP.poor.sum()), int(P_DET.poor.sum())]
flow['events (deterioration within 7 days)'] = [np.nan] * 10 + [int(P_DET.deterioration.sum())]
flow

In [ ]:
# Table 1: primary hs-cTn cohort by myocardial injury (early hs-cTn above the URL).
def med_iqr(s, d=1):
    s = s.dropna()
    return f'{rnd(s.median(), d)} ({rnd(s.quantile(.25), d)}-{rnd(s.quantile(.75), d)})' if len(s) else ''

def n_pct(s):
    s = s.dropna().astype(bool)
    return f'{int(s.sum())} ({rnd(100 * s.mean())})' if len(s) else ''

def p_cont(a, b):
    a, b = a.dropna(), b.dropna()
    return stats.mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan

def p_bin(a, b):
    t = np.array([[a.sum(), (~a).sum()], [b.sum(), (~b).sum()]])
    return stats.fisher_exact(t)[1] if t.min() < 5 else stats.chi2_contingency(t)[1]

def table_by(df, flag, cont, binary, cats):
    g1, g0 = df[df[flag]], df[~df[flag]]
    rows = [('N', str(len(df)), str(len(g1)), str(len(g0)), np.nan)]
    for item in cont:
        lab, col, dec = item if len(item) == 3 else (*item, 1)   # decimals shown for this variable
        rows.append((lab + ', median (IQR)', med_iqr(df[col], dec), med_iqr(g1[col], dec), med_iqr(g0[col], dec), p_cont(g1[col], g0[col])))
    for lab, col in binary:
        rows.append((lab + ', n (%)', n_pct(df[col]), n_pct(g1[col]), n_pct(g0[col]), p_bin(g1[col].astype(bool), g0[col].astype(bool))))
    for lab, col in cats:
        cs, c1, c0 = df[col].astype(str), g1[col].astype(str), g0[col].astype(str)
        p = stats.chi2_contingency(pd.crosstab(cs, df[flag]))[1]
        rows.append((lab, '', '', '', p))
        for lvl in cs.value_counts().index:
            rows.append(('   ' + str(lvl), n_pct(cs == lvl), n_pct(c1 == lvl), n_pct(c0 == lvl), np.nan))
    return pd.DataFrame(rows, columns=['Characteristic', 'All', 'Exposed', 'Not exposed', 'P'])

CONT = [('Age, years', 'age_y', 1), ('Early hs-cTn, ng/L', 'tnt_early_ngL', 1), ('Early lactate, mmol/L', 'lac_early', 1),
        ('Early NT-proBNP, pg/mL', 'bnp_early', 0), ('LVEF (minimum within 72 h), %', 'lvef_early_min', 0), ('QTc Fridericia (first 48 h), ms', 'qtcf_early', 0)]
BIN = [('Male', 'male'), ('Age < 1 year', 'age_lt1'), ('Invasive ventilation', 'ventilation'), ('Vasoactive infusion', 'vasoactive'),
       ('Status epilepticus', 'status_epilepticus'), ('Septic shock (diagnosis)', 'septic_shock'), ('Lactate >= 4 mmol/L (first 48 h)', 'lactate_ge4'),
       ('Methylprednisolone', 'methylprednisolone'), ('IVIG', 'ivig'), ('Tocilizumab', 'tocilizumab'),
       ('In-hospital death', 'death'), ('Critical discharge against advice', 'critical_ama'), ('Death or critical discharge against advice', 'poor')]
CATS = [('Syndrome group (ITES re-classification)', 'group'), ('Era', 'era')]
table1 = table_by(P_TNT, 'injury', CONT, BIN, CATS).rename(columns={'Exposed': f'hs-cTn > {URL_NGL} ng/L', 'Not exposed': f'hs-cTn <= {URL_NGL} ng/L'})
table1

In [ ]:
# Supplementary: who was tested? ICU index admissions with versus without early hs-cTn.
icu['tested'] = icu.tnt_early.notna()
tab_tested = table_by(icu, 'tested', [('Age, years', 'age_y')],
                      [('Male', 'male'), ('Invasive ventilation', 'ventilation'), ('Vasoactive infusion', 'vasoactive'),
                       ('Status epilepticus', 'status_epilepticus'), ('Death or critical discharge against advice', 'poor')],
                      [('Syndrome group (ITES re-classification)', 'group')]).rename(columns={'Exposed': 'Tested', 'Not exposed': 'Not tested'})
tab_tested

In [ ]:
# Table 2: cardiac phenotype across syndrome groups (ICU index admissions; arrest rule applied per measure).
ORDER_G = ['ANE', 'MERS', 'Other ITES', 'Possible ITES', 'SAE', 'Non-ITES', 'Unclassifiable']
def frac(df, col):
    s = df[col].dropna().astype(bool)
    return int(s.sum()), len(s)
rows = []
for g in [g for g in ORDER_G if (icu.group == g).sum() > 0]:
    a, b, c, d = P_TNT[P_TNT.group == g], P_EF[P_EF.group == g], P_QT[P_QT.group == g], P_BNP[P_BNP.group == g]
    i1, n1 = frac(a, 'injury'); i2, n2 = frac(b, 'dysfunction'); i3, n3 = frac(c, 'qt_prolonged'); i4, n4 = frac(c, 'qtb_prolonged')
    rows.append({'group': g, 'ICU N': int((icu.group == g).sum()),
                 f'hs-cTn > {URL_NGL} ng/L': f'{i1}/{n1} ({rnd(100 * i1 / n1, 0)}%)' if n1 else '',
                 'hs-cTn median (IQR), ng/L': med_iqr(a.tnt_early_ngL),
                 'LVEF < 55%': f'{i2}/{n2} ({rnd(100 * i2 / n2, 0)}%)' if n2 else '',
                 'QTcF >= 460 ms': f'{i3}/{n3} ({rnd(100 * i3 / n3, 0)}%)' if n3 else '',
                 'QTcB >= 460 ms': f'{i4}/{n4} ({rnd(100 * i4 / n4, 0)}%)' if n4 else '',
                 'NT-proBNP median (IQR), pg/mL': med_iqr(d.bnp_early, 0),
                 '_inj': (i1, n1), '_dys': (i2, n2), '_qt': (i3, n3), '_qtb': (i4, n4)})
pheno = pd.DataFrame(rows)

def mc_chi2_p(groups, flags, reps=10000, seed=SEED):
    # Monte Carlo (permutation) p value for a k x 2 chi-square test; robust to sparse cells.
    codes = pd.Categorical(groups).codes; f = np.asarray(flags, float)
    def stat(ff):
        k = codes.max() + 1
        n = np.bincount(codes, minlength=k); n1 = np.bincount(codes, weights=ff, minlength=k)
        p = ff.mean(); e1, e0 = n * p, n * (1 - p)
        return np.sum((n1 - e1) ** 2 / e1 + ((n - n1) - e0) ** 2 / e0)
    obs = stat(f); r = np.random.default_rng(seed)
    hits = sum(stat(r.permutation(f)) >= obs - 1e-12 for _ in range(reps))
    return (hits + 1) / (reps + 1)

p_inj = stats.chi2_contingency(pd.crosstab(P_TNT.group, P_TNT.injury))[1]
p_tnt = stats.kruskal(*[P_TNT[P_TNT.group == g].tnt_early_ngL for g in ORDER_G if (P_TNT.group == g).sum() > 0]).pvalue
p_bnp = stats.kruskal(*[P_BNP[P_BNP.group == g].bnp_early.dropna() for g in ORDER_G if P_BNP[P_BNP.group == g].bnp_early.notna().sum() > 0]).pvalue
p_dys = stats.chi2_contingency(pd.crosstab(P_EF.group, P_EF.dysfunction))[1]
p_qt = stats.chi2_contingency(pd.crosstab(P_QT.group, P_QT.qt_prolonged))[1]
p_qtb = stats.chi2_contingency(pd.crosstab(P_QT.group, P_QT.qtb_prolonged))[1]
p_mc = {'injury': mc_chi2_p(P_TNT.group, P_TNT.injury), 'lvef': mc_chi2_p(P_EF.group, P_EF.dysfunction),
        'qtc': mc_chi2_p(P_QT.group, P_QT.qt_prolonged), 'qtcb': mc_chi2_p(P_QT.group, P_QT.qtb_prolonged)}
print(f'P across groups: injury {p_inj:.2g}; hs-cTn {p_tnt:.2g}; NT-proBNP {p_bnp:.2g}; LVEF<55 {p_dys:.2g}; QTcF>=460 {p_qt:.2g}; QTcB>=460 {p_qtb:.2g}')
print('Monte Carlo P (10,000 permutations):', {k: round(v, 4) for k, v in p_mc.items()})
table2 = pheno.drop(columns=['_inj', '_dys', '_qt', '_qtb'])
table2

In [ ]:
# Figure helpers (submission style: no titles, panel letters aligned with the y-axis title).
# Color figures (PCCM encourages color): validated categorical slots 1-2 (blue, orange; adjacent CVD and normal-vision
# checks pass on white) plus a neutral gray for "other"; every mark is opaque so that EPS export keeps it intact.
GREY_DARK, GREY_MID, GREY_LIGHT = '#404040', '#8c8c8c', '#d0d0d0'
BLUE, ORANGE, NEUTRAL = '#2a78d6', '#eb6834', '#898781'
BLUE_BAND = '#b7d3f6'                 # light step of the blue ramp, for confidence bands
REF_INK = '#52514e'                   # reference lines
TIER_COLOR = {'ANE': BLUE, 'MERS': BLUE, 'Other ITES': BLUE, 'Possible ITES': BLUE, 'SAE': ORANGE,
              'Non-ITES': NEUTRAL, 'Unclassifiable': NEUTRAL}
TIER_HANDLES = [Patch(facecolor=BLUE, label='ITES (definite, related, or possible)'),
                Patch(facecolor=ORANGE, label='Sepsis-associated encephalopathy'),
                Patch(facecolor=NEUTRAL, label='Non-ITES or unclassifiable')]

def panel_letters(fig, axes, letters):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    for ax, letter in zip(axes, letters):
        bbox = ax.yaxis.label.get_window_extent(renderer=renderer)
        x0 = inv.transform((bbox.x0, 0))[0]
        top = ax.get_position().y1
        fig.text(x0, top + 0.012, letter, fontsize=18, fontweight='bold', ha='left', va='bottom')

def save(fig, name):
    # EPS and PDF (vector) for submission; TIFF (LZW, TIFF_WIDTH_IN wide at TIFF_DPI) as the raster alternative; PNG preview.
    fig.savefig(FIG / f'{name}.png', dpi=600, bbox_inches='tight')
    fig.savefig(FIG / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(FIG / f'{name}.eps', bbox_inches='tight')
    sizes = {'EPS': (FIG / f'{name}.eps').stat().st_size / 1e6}
    if EXPORT_TIFF:
        with Image.open(FIG / f'{name}.png') as im:
            width_in = im.size[0] / 600
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=TIFF_DPI * TIFF_WIDTH_IN / width_in, bbox_inches='tight')
        buf.seek(0)
        with Image.open(buf) as im:
            im.convert('RGB').save(FIG / f'{name}.tif', compression='tiff_lzw', dpi=(TIFF_DPI, TIFF_DPI))
            tif_in = im.size[0] / TIFF_DPI
        sizes['TIFF'] = (FIG / f'{name}.tif').stat().st_size / 1e6
    note = f' (TIFF {tif_in:.2f} in wide at {TIFF_DPI} dpi)' if EXPORT_TIFF else ''
    flag = '  <-- above the 5 MB upload limit' if max(sizes.values()) > 5 else ''
    print(name + ': ' + ', '.join(f'{k} {v:.2f} MB' for k, v in sizes.items()) + note + flag)

def bar_fraction(ax, labels, fracs, ylabel, color=BLUE):
    pct = [100 * a / b if b else 0 for a, b in fracs]
    top = max(pct + [10])
    ymax = 112 if top >= 90 else top * 1.25
    colors = [TIER_COLOR.get(l, color) for l in labels]
    bars = ax.bar(range(len(labels)), pct, color=colors, edgecolor='none', width=0.65)
    for i, (a, b) in enumerate(fracs):
        ax.text(i, pct[i] + 0.015 * ymax, f'{a}/{b}', ha='center', va='bottom', fontsize=11)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel(ylabel); ax.set_ylim(0, ymax)
    if top >= 90:
        ax.set_yticks(range(0, 101, 20))
    return bars

def plain_log_ticks(axis, ticks, lo, hi):
    ticks = [t for t in ticks if lo <= t <= hi]
    axis.set_major_locator(FixedLocator(ticks))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    axis.set_minor_locator(NullLocator())

In [ ]:
# Figure 1: cardiac phenotype across syndrome groups.
fig, axs = plt.subplots(2, 2, figsize=(12, 9.5))
fig.subplots_adjust(hspace=0.55, wspace=0.32)
groups = [g for g in ORDER_G if (P_TNT.group == g).sum() > 0]
bar_fraction(axs[0, 0], groups, [tuple(pheno.set_index('group').loc[g, '_inj']) for g in groups], f'hs-cTnT > {URL_NGL} ng/L (%)')

ax = axs[0, 1]
data = [P_TNT[P_TNT.group == g].tnt_early_ngL.clip(lower=1).values for g in groups]
for i, (g, d) in enumerate(zip(groups, data)):
    ax.scatter(i + rng.uniform(-0.18, 0.18, len(d)), d, s=9, color=TIER_COLOR[g], linewidths=0, zorder=2)
ax.boxplot(data, positions=range(len(groups)), widths=0.55, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor='none', edgecolor='black'), medianprops=dict(color='black', linewidth=1.8),
           whiskerprops=dict(color='black'), capprops=dict(color='black'), zorder=3)
ax.set_yscale('log')
ylo, yhi = ax.get_ylim()
plain_log_ticks(ax.yaxis, [1, 3, 10, 30, 100, 300, 1000, 3000], ylo, yhi)
ref = ax.axhline(URL_NGL, color=REF_INK, linestyle='--', linewidth=1.1, zorder=1)
ax.set_xticks(range(len(groups))); ax.set_xticklabels(groups, rotation=30, ha='right')
ax.set_ylabel('hs-cTnT (ng/L)')
ax.legend([ref], [f'Upper reference limit ({URL_NGL} ng/L)'], loc='upper right')

gq = [g for g in ORDER_G if (P_EF.group == g).sum() > 0]
bar_fraction(axs[1, 0], gq, [tuple(pheno.set_index('group').loc[g, '_dys']) for g in gq], 'LVEF < 55% (%)')
gq = [g for g in ORDER_G if (P_QT.group == g).sum() > 0]
bar_fraction(axs[1, 1], gq, [tuple(pheno.set_index('group').loc[g, '_qt']) for g in gq], 'QTc (Fridericia) ≥ 460 ms (%)')
fig.legend(handles=TIER_HANDLES, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.04), fontsize=12)
panel_letters(fig, [axs[0, 0], axs[0, 1], axs[1, 0], axs[1, 1]], 'ABCD')
save(fig, 'Figure1')
plt.show()

In [ ]:
# Association with death or critical discharge against advice: modified Poisson regression (robust SE).
def mpoisson(df, formula):
    m = smf.glm(formula, data=df, family=sm.families.Poisson()).fit(cov_type='HC0')
    return m

def rr_row(m, term):
    b, se = m.params[term], m.bse[term]
    return np.exp(b), np.exp(b - 1.96 * se), np.exp(b + 1.96 * se), m.pvalues[term]

for d in (P_TNT, P_EF, P_QT, P_BNP):
    for c in ['poor', 'male', 'age_lt1', 'ventilation', 'vasoactive', 'status_epilepticus']:
        d[c] = d[c].astype(int)
    d['group_ane'] = (d.group3 == 'ANE').astype(int)
    d['group_sae'] = (d.group3 == 'SAE').astype(int)

M1 = ' + age_y + male'
M2 = M1 + ' + group_ane + group_sae'
M3 = M2 + ' + ventilation + vasoactive + status_epilepticus'
EXPOSURES = [
    (f'hs-cTn > {URL_NGL} ng/L', P_TNT, 'injury'), ('hs-cTn, per doubling', P_TNT, 'log2_tnt'),
    ('LVEF < 55%', P_EF, 'dysfunction'), ('NT-proBNP, per doubling', P_BNP, 'log2_bnp'),
    ('QTc (Fridericia) >= 460 ms', P_QT, 'qt_prolonged'),
]
res = []
for lab, d, var in EXPOSURES:
    d = d.copy(); d[var] = d[var].astype(float)
    row = {'exposure': lab, 'N': len(d), 'events': int(d.poor.sum())}
    if d[var].nunique() == 2:
        e1, n1 = int(d[d[var] == 1].poor.sum()), int((d[var] == 1).sum())
        e0, n0 = int(d[d[var] == 0].poor.sum()), int((d[var] == 0).sum())
        row['events/N exposed'] = f'{e1}/{n1}'; row['events/N unexposed'] = f'{e0}/{n0}'
    for mname, rhs in [('Model 1', M1), ('Model 2', M2), ('Model 3', M3)]:
        try:
            m = mpoisson(d, f'poor ~ {var}{rhs}')
            rr, lo, hi, p = rr_row(m, var)
            row[mname] = f'{rnd(rr, 2)} ({rnd(lo, 2)}-{rnd(hi, 2)})'; row[mname + ' P'] = p
            row[mname + '_num'] = (rr, lo, hi)
        except Exception as e:
            row[mname] = f'not estimable ({type(e).__name__})'
    res.append(row)
assoc = pd.DataFrame(res)
table3 = assoc[[c for c in assoc.columns if not c.endswith('_num')]]
table3

In [ ]:
# Overlap-weighted estimate for myocardial injury (propensity model carries the covariates; outcome model does not).
PS_COVS = ['age_y', 'age_lt1', 'male', 'group_ane', 'group_sae', 'ventilation', 'vasoactive', 'status_epilepticus', 'septic_shock_i', 'lac_ge4_i', 'lac_missing']
def prep_ps(d):
    d = d.copy()
    d['septic_shock_i'] = d.septic_shock.astype(int)
    d['lac_missing'] = d.lac_early.isna().astype(int)
    d['lac_ge4_i'] = (d.lac_early >= 4).fillna(False).astype(int)
    d['injury_i'] = d.injury.astype(int)
    return d

def overlap_rr(d):
    X = sm.add_constant(d[PS_COVS].astype(float))
    ps = sm.Logit(d.injury_i, X).fit(disp=0, method='bfgs', maxiter=500).predict(X)
    w = np.where(d.injury_i == 1, 1 - ps, ps)
    r1 = np.average(d.poor[d.injury_i == 1], weights=w[d.injury_i == 1])
    r0 = np.average(d.poor[d.injury_i == 0], weights=w[d.injury_i == 0])
    return r1, r0, w, ps

def smd(d, w, col):
    a, b = d[d.injury_i == 1], d[d.injury_i == 0]
    wa, wb = (None, None) if w is None else (w[d.injury_i == 1], w[d.injury_i == 0])
    ma, mb = np.average(a[col], weights=wa), np.average(b[col], weights=wb)
    va = np.average((a[col] - ma) ** 2, weights=wa); vb = np.average((b[col] - mb) ** 2, weights=wb)
    return (ma - mb) / np.sqrt((va + vb) / 2) if (va + vb) > 0 else 0.0

D = prep_ps(P_TNT)
r1, r0, w, ps = overlap_rr(D)
boot = []
for _ in range(1000):
    s = D.sample(len(D), replace=True, random_state=int(rng.integers(1e9)))
    if s.injury_i.nunique() < 2 or s.poor.sum() == 0:
        continue
    try:
        b1, b0, _, _ = overlap_rr(s)
        if b0 > 0:
            boot.append((b1 / b0, b1 - b0))
    except Exception:
        pass
boot = np.array(boot)
ow = {'risk_injury': r1, 'risk_no_injury': r0, 'RR': r1 / r0, 'RR_lo': np.percentile(boot[:, 0], 2.5), 'RR_hi': np.percentile(boot[:, 0], 97.5),
      'RD': r1 - r0, 'RD_lo': np.percentile(boot[:, 1], 2.5), 'RD_hi': np.percentile(boot[:, 1], 97.5), 'bootstrap_reps': len(boot)}
balance = pd.DataFrame({'covariate': PS_COVS, 'SMD unweighted': [smd(D, None, c) for c in PS_COVS],
                        'SMD overlap-weighted': [smd(D, w, c) for c in PS_COVS]}).round(3)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in ow.items()})
balance

In [ ]:
# Time-anchored analyses at the first hs-cTn report.
def prep_det(d):
    d = d.copy()
    for c in ['poor', 'male', 'status_epilepticus', 'injury', 'injury_alt', 'vaso_before', 'vent_before', 'deterioration', 'deterioration_72h', 'deterioration_any']:
        if c in d:
            d[c] = d[c].astype(int)
    d['group_ane'] = (d.group3 == 'ANE').astype(int)
    d['group_sae'] = (d.group3 == 'SAE').astype(int)
    d['log2_tnt'] = np.log2(d.tnt_early_ngL.clip(lower=1))
    d['lac_ge4'] = (d.lac_early >= 4).fillna(False).astype(int)
    d['lac_missing'] = d.lac_early.isna().astype(int)
    return d

BASE_DET = ' + age_y + male + group_ane + group_sae + status_epilepticus'
A_ = prep_det(P_TNT)
rr_pre = rr_row(mpoisson(A_, 'poor ~ injury' + BASE_DET + ' + vaso_before + vent_before'), 'injury')
Dt = prep_det(P_DET)
Dt['det_pressor'] = (P_DET.pressor_7d | P_DET.vent_7d | P_DET.poor_7d).astype(int)
Dt['det_no_vent'] = (P_DET.vaso_7d | P_DET.poor_7d).astype(int)

def det_row(label, d, outcome, var, rhs):
    m = mpoisson(d, f'{outcome} ~ {var}{rhs}')
    rr, lo, hi, p = rr_row(m, var)
    row = {'analysis': label, 'N': len(d), 'events': int(d[outcome].sum())}
    if d[var].nunique() == 2:
        row['events/N, injury'] = f'{int(d[d[var] == 1][outcome].sum())}/{int((d[var] == 1).sum())}'
        row['events/N, no injury'] = f'{int(d[d[var] == 0][outcome].sum())}/{int((d[var] == 0).sum())}'
    row.update({'RR (95% CI)': f'{rnd(rr, 2)} ({rnd(lo, 2)}-{rnd(hi, 2)})', 'P': p, '_num': (rr, lo, hi)})
    return row

t5 = [
    {'analysis': 'Death or critical AMA, adjusted for organ support present before testing', 'N': len(A_), 'events': int(A_.poor.sum()),
     'RR (95% CI)': f'{rnd(rr_pre[0], 2)} ({rnd(rr_pre[1], 2)}-{rnd(rr_pre[2], 2)})', 'P': rr_pre[3], '_num': rr_pre[:3]},
    det_row('Subsequent deterioration within 7 days (primary definition)', Dt, 'deterioration', 'injury', BASE_DET),
    det_row('Subsequent deterioration, additionally adjusted for lactate', Dt, 'deterioration', 'injury', BASE_DET + ' + lac_ge4 + lac_missing'),
    det_row('Subsequent deterioration, hs-cTn per doubling', Dt, 'deterioration', 'log2_tnt', BASE_DET),
    det_row('Deterioration defined with vasopressors only (dobutamine/milrinone excluded)', Dt, 'det_pressor', 'injury', BASE_DET),
    det_row('Deterioration without ventilation (vasoactive start or death/critical AMA)', Dt, 'det_no_vent', 'injury', BASE_DET),
    det_row('Subsequent deterioration, age >= 1 year', Dt[Dt.age_y >= 1], 'deterioration', 'injury', BASE_DET),
    det_row('Deterioration with escalation within 72 h', Dt, 'deterioration_72h', 'injury', BASE_DET),
    det_row('Deterioration with escalation at any time during admission', Dt, 'deterioration_any', 'injury', BASE_DET),
    det_row(f'Subsequent deterioration, injury defined as hs-cTn > {ALT_NGL} ng/L', Dt, 'deterioration', 'injury_alt', BASE_DET),
    det_row('Subsequent deterioration, consensus ITES only (definite, related or possible)', Dt[Dt.ites_any.astype(bool)], 'deterioration', 'injury', BASE_DET.replace(' + group_sae', '')),
    det_row('Subsequent deterioration, excluding non-ITES and unclassifiable', Dt[Dt.ites_any.astype(bool) | Dt.group.eq('SAE')], 'deterioration', 'injury', BASE_DET),
]
table5 = pd.DataFrame(t5)
# Effect modification by consensus ITES status (definite, related or possible ITES vs all other diagnoses).
Dt['ites_i'] = Dt.ites_any.astype(int)
m_int = mpoisson(Dt, 'deterioration ~ injury * ites_i' + BASE_DET)
p_interaction = float(m_int.pvalues['injury:ites_i'])
sub_rows = []
for lab, d in [('Consensus ITES (definite, related or possible)', Dt[Dt.ites_i == 1]),
               ('Other diagnoses (SAE, non-ITES, unclassifiable)', Dt[Dt.ites_i == 0]), ('All', Dt)]:
    e1, n1 = int(d[d.injury == 1].deterioration.sum()), int((d.injury == 1).sum())
    e0, n0 = int(d[d.injury == 0].deterioration.sum()), int((d.injury == 0).sum())
    sub_rows.append({'subgroup': lab, 'N': len(d), 'deterioration, injury': f'{e1}/{n1} ({rnd(100 * e1 / n1)}%)',
                     'deterioration, no injury': f'{e0}/{n0} ({rnd(100 * e0 / n0)}%)',
                     'unadjusted RR': float(rnd((e1 / n1) / (e0 / n0), 2)) if e0 else np.nan,
                     'risk difference, %': float(rnd(100 * (e1 / n1 - e0 / n0), 1))})
det_subgroups = pd.DataFrame(sub_rows)
# Fragility index (unadjusted Fisher exact test): children in the lower-risk group whose outcome would have to change
# from no event to event before the association lost statistical significance (p >= 0.05).
def fragility_index(e1, n1, e0, n0, alpha=0.05):
    tab = lambda a, b: [[a, n1 - a], [b, n0 - b]]
    p0 = stats.fisher_exact(tab(e1, e0))[1]
    if p0 >= alpha:
        return 0, p0
    # Events are added to the lower-risk group (Walsh et al.; with unequal group sizes, the group with the lower event rate).
    fi, add_to_0 = 0, e0 / n0 <= e1 / n1
    while True:
        if (add_to_0 and e0 >= n0) or (not add_to_0 and e1 >= n1):
            return np.nan, p0
        if add_to_0:
            e0 += 1
        else:
            e1 += 1
        fi += 1
        if stats.fisher_exact(tab(e1, e0))[1] >= alpha:
            return fi, p0
fi_e1, fi_n1 = int(Dt[Dt.injury == 1].deterioration.sum()), int((Dt.injury == 1).sum())
fi_e0, fi_n0 = int(Dt[Dt.injury == 0].deterioration.sum()), int((Dt.injury == 0).sum())
frag_index, frag_fisher_p = fragility_index(fi_e1, fi_n1, fi_e0, fi_n0)
print(f'Fragility index (deterioration, Fisher exact): {frag_index} (unadjusted Fisher p = {frag_fisher_p:.4f})')
print(f'P for interaction (myocardial injury x consensus ITES): {p_interaction:.3f}')
print(det_subgroups.to_string(index=False))
h_inj = P_DET[P_DET.injury].hours_to_escalation.dropna(); h_no = P_DET[~P_DET.injury].hours_to_escalation.dropna()
print('Hours from first hs-cTn report to escalation: median (IQR)', f'{P_DET.hours_to_escalation.median():.1f}',
      P_DET.hours_to_escalation.quantile([.25, .75]).round(1).tolist(), '| n =', int(P_DET.hours_to_escalation.notna().sum()))
print('Escalation within 7 days: vasoactive', int(P_DET.vaso_7d.sum()), '| ventilation', int(P_DET.vent_7d.sum()),
      '| death or critical AMA within 7 days', int(P_DET.poor_7d.sum()), '| during the admission', int(P_DET.poor.sum()))
table5.drop(columns=['_num'])

In [ ]:
# Supplementary: deterioration cohort (no organ support at the first hs-cTn report) by myocardial injury.
table_det = table_by(P_DET, 'injury', CONT, BIN, CATS).rename(columns={'Exposed': f'hs-cTn > {URL_NGL} ng/L', 'Not exposed': f'hs-cTn <= {URL_NGL} ng/L'})
table_det

In [ ]:
# Dose-response: event rates across early hs-cTn categories (multiples of the URL) and age-adjusted restricted cubic splines.
bins = [0, URL_NGL, 5 * URL_NGL, 10 * URL_NGL, np.inf]
labels_c = [f'≤{URL_NGL}', f'{URL_NGL + 1}–{5 * URL_NGL}', f'{5 * URL_NGL + 1}–{10 * URL_NGL}', f'>{10 * URL_NGL}']

def trend_table(d, outcome):
    d = d.copy()
    d['cat'] = pd.cut(d.tnt_early_ngL, bins=bins, labels=labels_c, right=True)
    t = d.groupby('cat', observed=False).agg(N=(outcome, 'size'), events=(outcome, 'sum'))
    t['rate %'] = [float(rnd(100 * e / n, 1)) if n else np.nan for e, n in zip(t.events, t.N)]
    k = np.arange(len(t)); n = t.N.values.astype(float); e = t.events.values.astype(float); pbar = e.sum() / n.sum()
    T = np.sum(k * (e - n * pbar)); V = pbar * (1 - pbar) * (np.sum(n * k ** 2) - np.sum(n * k) ** 2 / n.sum())
    return t, 2 * (1 - stats.norm.cdf(abs(T / np.sqrt(V))))   # Cochran-Armitage test for trend

dose, p_trend = trend_table(prep_det(P_TNT), 'poor')
dose_det, p_trend_det = trend_table(Dt, 'deterioration')
print(dose, f'\nP for trend (death or critical AMA): {p_trend:.3g}\n')
print(dose_det, f'\nP for trend (subsequent deterioration): {p_trend_det:.3g}')

def rcs3(x, knots):
    t1, t2, t3 = knots
    pos = lambda u: np.clip(u, 0, None) ** 3
    return (pos(x - t1) - pos(x - t2) * (t3 - t1) / (t3 - t2) + pos(x - t3) * (t2 - t1) / (t3 - t2)) / (t3 - t1) ** 2

def spline_fit(d, outcome):
    x = np.log2(d.tnt_early_ngL.clip(lower=1)).values
    knots = np.percentile(x, [10, 50, 90])
    def design(xx, age):
        xx = np.asarray(xx, float)
        return np.column_stack([np.ones_like(xx), xx, rcs3(xx, knots), np.asarray(age, float) * np.ones_like(xx)])
    X = design(x, d.age_y.values); y = d[outcome].astype(float).values
    spl = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    lin = sm.GLM(y, X[:, [0, 1, 3]], family=sm.families.Binomial()).fit()
    p_nl = stats.chi2.sf(2 * (spl.llf - lin.llf), 1)
    gx = np.linspace(np.log2(3), np.log2(d.tnt_early_ngL.quantile(0.98)), 120)
    age_med = d.age_y.median()
    diff = design(gx, age_med) - design(np.array([np.log2(URL_NGL)]), age_med)
    lp = diff @ spl.params
    se = np.sqrt(np.einsum('ij,jk,ik->i', diff, spl.cov_params(), diff))
    curve = pd.DataFrame({'tnt_ngL': 2 ** gx, 'OR': np.exp(lp), 'lo': np.exp(lp - 1.96 * se), 'hi': np.exp(lp + 1.96 * se)})
    return curve, p_nl, 2 ** knots

curve_poor, p_nonlin, knots_poor = spline_fit(P_TNT, 'poor')
curve_det, p_nonlin_det, knots_det = spline_fit(Dt, 'deterioration')
print(f'\nSpline knots (ng/L), deterioration model: {np.round(knots_det, 1)}; P for non-linearity: {p_nonlin_det:.3g}')
print(f'Spline knots (ng/L), death or critical AMA model: {np.round(knots_poor, 1)}; P for non-linearity: {p_nonlin:.3g}')

In [ ]:
# Figure 3: hs-cTn and outcomes (dose-response).
fig, axs = plt.subplots(1, 3, figsize=(17, 4.9))
fig.subplots_adjust(wspace=0.36)
for ax, t, ylab in [(axs[0], dose, 'Death or critical discharge (%)'), (axs[1], dose_det, 'Deterioration within 7 days (%)')]:
    ax.bar(range(len(t)), t['rate %'], color=BLUE, edgecolor='none', width=0.65)
    ymax = max(t['rate %'].max() * 1.3, 10)
    for i, (n_ev, nn, r) in enumerate(zip(t.events, t.N, t['rate %'])):
        ax.text(i, r + 0.015 * ymax, f'{int(n_ev)}/{int(nn)}', ha='center', va='bottom', fontsize=11)
    ax.set_xticks(range(len(t))); ax.set_xticklabels(labels_c)
    ax.set_xlabel('Early hs-cTnT (ng/L)'); ax.set_ylabel(ylab)
    ax.set_ylim(0, ymax)
ax = axs[2]
band = ax.fill_between(curve_det.tnt_ngL, curve_det.lo, curve_det.hi, color=BLUE_BAND, linewidth=0)
line, = ax.plot(curve_det.tnt_ngL, curve_det.OR, color=BLUE, linewidth=2.2)
h1 = ax.axhline(1, color=REF_INK, linestyle=':', linewidth=1.1)
v14 = Line2D([], [], color=REF_INK, linestyle='--', linewidth=1.1)   # legend entry; the line is drawn after the legend is placed
ax.set_xscale('log'); ax.set_yscale('log')
plain_log_ticks(ax.xaxis, [3, 10, 30, 100, 300, 1000], curve_det.tnt_ngL.min(), curve_det.tnt_ngL.max())
ylo, yhi = ax.get_ylim()
plain_log_ticks(ax.yaxis, [0.1, 0.25, 0.5, 1, 2, 4, 8, 16, 32], ylo, yhi)
ax.set_xlabel('Early hs-cTnT (ng/L)'); ax.set_ylabel('Odds ratio, deterioration')
leg = ax.legend([line, band, h1, v14], [f'Age-adjusted odds ratio (reference {URL_NGL} ng/L)', '95% confidence interval', 'Odds ratio = 1',
                                        f'Upper reference limit ({URL_NGL} ng/L)'], loc='upper left', fontsize=9.5)
# Dashed reference line at the URL: stop it just below the legend so the two do not overlap.
xlim_c, ylim_c = ax.get_xlim(), ax.get_ylim()
fig.canvas.draw()
leg_box = leg.get_window_extent().transformed(ax.transAxes.inverted())
x_url = ax.transAxes.inverted().transform(ax.transData.transform((URL_NGL, 1)))[0]
top_url = leg_box.y0 - 0.03 if leg_box.x0 <= x_url <= leg_box.x1 else 1.0
ax.plot([URL_NGL, URL_NGL], [0, top_url], transform=ax.get_xaxis_transform(), color=REF_INK, linestyle='--', linewidth=1.1)
ax.set_xlim(xlim_c); ax.set_ylim(ylim_c)
panel_letters(fig, axs, 'ABC')
save(fig, 'Figure3')
plt.show()

In [ ]:
# Incremental prognostic value of hs-cTn (apparent and optimism-corrected AUC; 1,000 bootstrap samples).
def fit_auc(df, f, outcome):
    m = smf.glm(f, data=df, family=sm.families.Binomial()).fit()
    return m, roc_auc_score(df[outcome], m.predict(df))

def incremental(df, base, outcome, reps=1000):
    m0, a0 = fit_auc(df, base, outcome)
    m1, a1 = fit_auc(df, base + ' + log2_tnt', outcome)
    lr = stats.chi2.sf(2 * (m1.llf - m0.llf), 1)
    deltas, o0, o1 = [], [], []
    for _ in range(reps):
        s_ = df.sample(len(df), replace=True, random_state=int(rng.integers(1e9)))
        if s_[outcome].nunique() < 2:
            continue
        try:
            b0, x0 = fit_auc(s_, base, outcome); b1, x1 = fit_auc(s_, base + ' + log2_tnt', outcome)
            deltas.append(x1 - x0)
            o0.append(x0 - roc_auc_score(df[outcome], b0.predict(df)))
            o1.append(x1 - roc_auc_score(df[outcome], b1.predict(df)))
        except Exception:
            pass
    return {'AUC base': a0, 'AUC base (optimism-corrected)': a0 - np.mean(o0), 'AUC + hs-cTn': a1,
            'AUC + hs-cTn (optimism-corrected)': a1 - np.mean(o1), 'delta AUC': a1 - a0,
            'delta lo': np.percentile(deltas, 2.5), 'delta hi': np.percentile(deltas, 97.5), 'LR test P': lr, 'reps': len(deltas)}

inc_poor = incremental(prep_det(P_TNT), 'poor ~ age_y + male + group_ane + group_sae + ventilation + vasoactive + status_epilepticus', 'poor')
inc_det = incremental(Dt, 'deterioration ~ age_y + male + group_ane + group_sae + status_epilepticus', 'deterioration')
table4 = pd.DataFrame([
    dict(outcome='Death or critical AMA (primary cohort)', base='age, sex, syndrome group, ventilation, vasoactive infusion, status epilepticus', **inc_poor),
    dict(outcome='Deterioration within 7 days (no organ support at testing)', base='age, sex, syndrome group, status epilepticus', **inc_det),
])
num_cols = table4.select_dtypes('number').columns.drop(['reps'])
table4[num_cols] = table4[num_cols].round(4)
table4

In [ ]:
# Figure 2: forest plot of adjusted relative risks.
def num(row):
    return tuple(row['_num'])
inj = assoc.set_index('exposure')
sections = [
    ('Death or critical discharge', [
        ('Myocardial injury, model 1', inj.loc[f'hs-cTn > {URL_NGL} ng/L', 'Model 1_num']),
        ('Myocardial injury, model 2', inj.loc[f'hs-cTn > {URL_NGL} ng/L', 'Model 2_num']),
        ('Myocardial injury, model 3', inj.loc[f'hs-cTn > {URL_NGL} ng/L', 'Model 3_num']),
        ('Myocardial injury, support before testing', tuple(rr_pre[:3])),
        ('Myocardial injury, overlap weighting', (ow['RR'], ow['RR_lo'], ow['RR_hi'])),
        ('LVEF < 55%, model 3', inj.loc['LVEF < 55%', 'Model 3_num']),
        ('NT-proBNP per doubling, model 3', inj.loc['NT-proBNP, per doubling', 'Model 3_num']),
        ('QTcF ≥ 460 ms, model 3', inj.loc['QTc (Fridericia) >= 460 ms', 'Model 3_num']),
    ]),
    ('Deterioration within 7 days, no organ support at testing', [
        ('Myocardial injury, adjusted', num(table5.iloc[1])),
        ('Myocardial injury, adjusted incl. lactate', num(table5.iloc[2])),
        ('hs-cTnT per doubling, adjusted', num(table5.iloc[3])),
    ]),
]
rows_fp = []
for head, items in sections:
    rows_fp.append((head, None))
    rows_fp += [(lab, v) for lab, v in items]
fig, ax = plt.subplots(figsize=(10, 0.5 * len(rows_fp) + 1.0))
ys = np.arange(len(rows_fp))[::-1]
for y, (lab, v) in zip(ys, rows_fp):
    if v is None:
        continue
    rr, lo, hi = v
    ax.plot([lo, hi], [y, y], color=BLUE, linewidth=1.6)
    ax.plot(rr, y, marker='s', color=BLUE, markersize=8)
    ax.annotate(f'{rnd(rr, 2)} ({rnd(lo, 2)}–{rnd(hi, 2)})', xy=(1.02, y), xycoords=('axes fraction', 'data'), va='center', fontsize=11)
ax.set_yticks(ys)
ax.set_yticklabels([lab for lab, _ in rows_fp])
for tick, (lab, v) in zip(ax.get_yticklabels(), rows_fp):
    if v is None:
        tick.set_fontweight('bold')
ax.set_xscale('log')
ref = ax.axvline(1, color=REF_INK, linestyle='--', linewidth=1.1)
xlo, xhi = ax.get_xlim()
plain_log_ticks(ax.xaxis, [0.25, 0.5, 1, 2, 4, 8], xlo, xhi)
ax.set_xlabel('Relative risk (95% CI)')
ax.legend([ref], ['Relative risk = 1'], loc='lower right')
save(fig, 'Figure2')
plt.show()

In [ ]:
# Exploratory brain-heart correlates: neuroimaging and EEG features from the ITES evidence (index admission).
BH = P_TNT.copy()
BH['thalamus'] = np.where(BH.mri_n.notna() | BH.ct_n.notna(), (BH.mri_thal_bilat.fillna(False).astype(bool) | BH.ct_thal_bilat.fillna(False).astype(bool)), np.nan)
BH['brainstem'] = np.where(BH.mri_n.notna(), BH.mri_brainstem.fillna(False).astype(bool), np.nan)
BH['diffuse'] = np.where(BH.mri_n.notna() | BH.ct_n.notna(), (BH.mri_diffuse_edema.fillna(False).astype(bool) | BH.ct_diffuse_edema.fillna(False).astype(bool)), np.nan)
BH['eeg_suppressed'] = np.where(BH.eeg_n.notna(), BH.eeg_suppressed.fillna(False).astype(bool), np.nan)
bh_rows = []
for lab, col in [('Bilateral thalamic lesions', 'thalamus'), ('Brainstem lesion', 'brainstem'), ('Diffuse cerebral edema', 'diffuse'),
                 ('EEG background suppression', 'eeg_suppressed')]:
    d = BH[BH[col].notna()].copy(); d[col] = d[col].astype(int); d['inj'] = d.injury.astype(int)
    a1, n1 = int(d[d[col] == 1].inj.sum()), int((d[col] == 1).sum())
    a0, n0 = int(d[d[col] == 0].inj.sum()), int((d[col] == 0).sum())
    try:
        m = smf.glm(f'inj ~ {col} + age_y + vasoactive', data=d, family=sm.families.Poisson()).fit(cov_type='HC0')
        rr, lo, hi, p = rr_row(m, col)
        est = f'{rnd(rr, 2)} ({rnd(lo, 2)}-{rnd(hi, 2)})'
    except Exception:
        est, p = 'not estimable', np.nan
    bh_rows.append({'feature': lab, 'N with study': len(d), 'injury if present': f'{a1}/{n1}', 'injury if absent': f'{a0}/{n0}',
                    'adjusted RR (age, vasoactive)': est, 'P': p, '_p1': a1 / n1 if n1 else np.nan, '_p0': a0 / n0 if n0 else np.nan,
                    '_n': (a1, n1, a0, n0)})
brain_heart = pd.DataFrame(bh_rows)
brain_heart.drop(columns=['_p1', '_p0', '_n'])

In [ ]:
# eFigure 1 (exploratory): prevalence of myocardial injury by neuroimaging and EEG findings.
fig, ax = plt.subplots(figsize=(10, 5))
xpos = np.arange(len(brain_heart))
w_ = 0.36
b1 = ax.bar(xpos - w_ / 2, 100 * brain_heart._p1, width=w_, color=BLUE, edgecolor='white', linewidth=1.5)
b0 = ax.bar(xpos + w_ / 2, 100 * brain_heart._p0, width=w_, color=ORANGE, edgecolor='white', linewidth=1.5)
for i, (a1, n1, a0, n0) in enumerate(brain_heart._n):
    ax.text(i - w_ / 2, 100 * a1 / n1 + 1.2 if n1 else 1, f'{a1}/{n1}', ha='center', va='bottom', fontsize=10)
    ax.text(i + w_ / 2, 100 * a0 / n0 + 1.2 if n0 else 1, f'{a0}/{n0}', ha='center', va='bottom', fontsize=10)
ax.set_xticks(xpos); ax.set_xticklabels(['Bilateral thalamic\nlesions', 'Brainstem\nlesion', 'Diffuse cerebral\nedema', 'EEG\nsuppression'])
ax.set_ylabel(f'hs-cTnT > {URL_NGL} ng/L (%)')
ax.set_ylim(0, np.nanmax(100 * np.r_[brain_heart._p1, brain_heart._p0]) * 1.3)
ax.legend([b1, b0], ['Finding present', 'Finding absent'], loc='upper right')
save(fig, 'eFigure1')
plt.show()

In [ ]:
# Sensitivity analyses for myocardial injury (Model 3 unless stated).
def model3_rr(d, var='injury', outcome='poor', rhs=M3):
    d = d.copy(); d[var] = d[var].astype(int); d[outcome] = d[outcome].astype(int)
    m = smf.glm(f'{outcome} ~ {var}{rhs}', data=d, family=sm.families.Poisson()).fit(cov_type='HC0')
    return rr_row(m, var)

def prep(d):
    d = d.copy()
    for c in ['male', 'age_lt1', 'ventilation', 'vasoactive', 'status_epilepticus']:
        d[c] = d[c].astype(int)
    d['group_ane'] = (d.group3 == 'ANE').astype(int); d['group_sae'] = (d.group3 == 'SAE').astype(int)
    return d

sens = []
def add(lab, d, **kw):
    try:
        rr, lo, hi, p = model3_rr(d, **kw)
        sens.append((lab, len(d), int(d[kw.get('outcome', 'poor')].sum()), f'{rnd(rr, 2)} ({rnd(lo, 2)}-{rnd(hi, 2)})', p))
    except Exception as e:
        sens.append((lab, len(d), int(d[kw.get('outcome', 'poor')].sum()), f'not estimable ({type(e).__name__})', np.nan))

add('Primary (early hs-cTn, ICU, arrest rule)', P_TNT)
pk = prep(icu[icu.tnt_peak.notna() & ~icu.arrest_before_tnt_first])
pk['injury'] = pk.tnt_peak > HSCTN_URL
add('Peak hs-cTn during admission', pk)
add('Excluding every patient with any cardiac arrest/CPR', P_TNT[~P_TNT.cpr_any])
add('Age >= 1 year only', P_TNT[P_TNT.age_y >= 1])
d = P_TNT.copy(); d['death_any_ama'] = (d.death | d.ama).astype(int)
add('Outcome: death or any discharge against advice', d, outcome='death_any_ama')
add(f'Injury defined as hs-cTn > {ALT_NGL} ng/L', P_TNT.assign(injury=P_TNT.injury_alt))
d = P_TNT.copy(); d['poor_rule'] = (d.death.astype(bool) | d.critical_ama_rule.astype(bool)).astype(int)
add('Outcome: critical discharge by the documentation rule (no clinician review)', d, outcome='poor_rule')
add('Consensus ITES only (definite, related or possible)', P_TNT[P_TNT.ites_any.astype(bool)], rhs=M3.replace(' + group_sae', ''))
dd = P_TNT.copy(); dd['death_i'] = dd.death.astype(int)
e1, n1 = int(dd[dd.injury].death_i.sum()), int(dd.injury.sum()); e0, n0 = int(dd[~dd.injury].death_i.sum()), int((~dd.injury).sum())
sens.append(('Outcome: in-hospital death only (Fisher exact, unadjusted)', len(dd), int(dd.death_i.sum()),
             f'{e1}/{n1} vs {e0}/{n0}', stats.fisher_exact([[e1, n1 - e1], [e0, n0 - e0]])[1]))
lg = smf.glm('poor ~ injury' + M3, data=P_TNT.assign(injury=P_TNT.injury.astype(int)), family=sm.families.Binomial()).fit()
sens.append(('Logistic regression (odds ratio), Model 3', len(P_TNT), int(P_TNT.poor.sum()),
             f'{rnd(np.exp(lg.params["injury"]), 2)} ({rnd(np.exp(lg.conf_int().loc["injury", 0]), 2)}-{rnd(np.exp(lg.conf_int().loc["injury", 1]), 2)})', lg.pvalues['injury']))
sensitivity = pd.DataFrame(sens, columns=['analysis', 'N', 'events', 'estimate (95% CI)', 'P'])
sensitivity

In [ ]:
# Supplementary: CK-MB activity versus hs-cTn within the primary cohort.
C = P_TNT[P_TNT.ckmb_early.notna()].copy()
C['ckmb_high'] = C.ckmb_early > 25
disc = pd.crosstab(C.ckmb_high.map({True: 'CK-MB activity > 25 U/L', False: 'CK-MB activity <= 25 U/L'}),
                   C.injury.map({True: f'hs-cTn > {URL_NGL} ng/L', False: f'hs-cTn <= {URL_NGL} ng/L'}))
auc_ckmb = roc_auc_score(C.poor, C.ckmb_early); auc_tnt_c = roc_auc_score(C.poor, C.tnt_early)
print(f'N={len(C)}; AUC for death or critical AMA: CK-MB activity {auc_ckmb:.3f} vs hs-cTn {auc_tnt_c:.3f}')
disc

In [ ]:
# Numbers quoted in the manuscript (computed here so that every number in the text can be traced and re-checked).
def frac_txt(a, b):
    return f'{int(a)}/{int(b)} ({rnd(100 * a / b)}%)'

def miqr(x, d=1):
    x = pd.Series(x).dropna()
    return f'{rnd(x.median(), d)} ({rnd(x.quantile(.25), d)}-{rnd(x.quantile(.75), d)})'

hrs_first = (P_TNT.tnt_early_time - P_TNT.admit).dt.total_seconds() / 3600
esc7 = P_DET.hours_to_escalation[(P_DET.hours_to_escalation > 0) & (P_DET.hours_to_escalation <= WINDOW_H)]
ites_icu = idx[idx.icu].ites_class.value_counts().to_dict()
MS = {
    'cohort': {'aes_patients': len(coh), 'acute_index': int(coh.acute_index.sum()), 'analysis_base': len(idx),
               'base_age': miqr(idx.age_y), 'base_male': frac_txt(idx.male.sum(), len(idx)), 'base_icu': frac_txt(idx.icu.sum(), len(idx)),
               'base_deaths': int(idx.death.sum()), 'base_critical_ama': int(idx.critical_ama.sum()), 'base_poor': int(idx.poor.sum()),
               'admission_years': [int(idx.admit.dt.year.min()), int(idx.admit.dt.year.max())],
               'icu_tested_48h': frac_txt(icu.tnt_early.notna().sum(), len(icu)),
               'excluded_arrest': int((icu.tnt_early.notna() & icu.arrest_before_tnt).sum()),
               'primary': len(P_TNT), 'primary_age': miqr(P_TNT.age_y), 'primary_age_lt1': frac_txt(P_TNT.age_lt1.sum(), len(P_TNT)),
               'primary_male': frac_txt(P_TNT.male.sum(), len(P_TNT)), 'primary_poor': frac_txt(P_TNT.poor.sum(), len(P_TNT)),
               'primary_deaths': int(P_TNT.death.sum()), 'primary_critical_ama': int(P_TNT.critical_ama.sum()),
               'primary_ventilation': frac_txt(P_TNT.ventilation.sum(), len(P_TNT)), 'primary_vasoactive': frac_txt(P_TNT.vasoactive.sum(), len(P_TNT)),
               'hours_admission_to_first_hsctn': miqr(hrs_first)},
    'ites_all': ev.ites_class.value_counts().to_dict(), 'ites_tier_all': ev.ites_tier.value_counts().to_dict(),
    'ites_icu': ites_icu, 'ites_primary': P_TNT.ites_class.value_counts().to_dict(), 'groups_primary': P_TNT.group.value_counts().to_dict(),
    'ites_review_flags': int(ev.needs_review.sum()),
    'injury': {'primary': frac_txt(P_TNT.injury.sum(), len(P_TNT)), 'alt_threshold': frac_txt(P_TNT.injury_alt.sum(), len(P_TNT)),
               'hsctn_median': miqr(P_TNT.tnt_early_ngL), 'by_group': {g: list(v) for g, v in pheno.set_index('group')['_inj'].items()},
               'lvef_lt55': frac_txt(P_EF.dysfunction.sum(), len(P_EF)), 'lvef_by_group': {g: list(v) for g, v in pheno.set_index('group')['_dys'].items()},
               'qtcf_ge460': frac_txt(P_QT.qt_prolonged.sum(), len(P_QT)), 'qt_by_group': {g: list(v) for g, v in pheno.set_index('group')['_qt'].items()},
               'bnp_median': miqr(P_BNP.bnp_early, 0), 'p_groups': {'injury': p_inj, 'hsctn': p_tnt, 'lvef': p_dys, 'qtc': p_qt, 'bnp': p_bnp}},
    'death_or_critical_ama': {'injured': frac_txt(P_TNT[P_TNT.injury].poor.sum(), P_TNT.injury.sum()),
                              'not_injured': frac_txt(P_TNT[~P_TNT.injury].poor.sum(), (~P_TNT.injury).sum()),
                              'models': assoc[['exposure', 'Model 1', 'Model 1 P', 'Model 2', 'Model 2 P', 'Model 3', 'Model 3 P']].to_dict('records'),
                              'overlap_weighting': {k: (float(v) if not isinstance(v, int) else v) for k, v in ow.items()},
                              'adjusted_support_before_testing': [float(v) for v in rr_pre],
                              'dose': dose.reset_index().astype({'cat': str}).to_dict('records'), 'p_trend': p_trend},
    'deterioration': {'n': len(P_DET), 'events': int(P_DET.deterioration.sum()), 'vaso_7d': int(P_DET.vaso_7d.sum()),
                      'vent_7d': int(P_DET.vent_7d.sum()), 'death_or_critical_ama': int(P_DET.poor.sum()),
                      'injured': frac_txt(P_DET[P_DET.injury].deterioration.sum(), P_DET.injury.sum()),
                      'not_injured': frac_txt(P_DET[~P_DET.injury].deterioration.sum(), (~P_DET.injury).sum()),
                      'hours_to_escalation_within_7d': miqr(esc7), 'n_escalations_within_7d': int(len(esc7)),
                      'table5': table5.drop(columns=['_num']).to_dict('records'), 'p_interaction_ites': p_interaction,
                      'subgroups': det_subgroups.to_dict('records'), 'dose': dose_det.reset_index().astype({'cat': str}).to_dict('records'),
                      'p_trend': p_trend_det, 'p_nonlinearity': p_nonlin_det, 'incremental': inc_det},
    'incremental_poor': inc_poor,
    'brain_heart': brain_heart.drop(columns=['_p1', '_p0', '_n']).to_dict('records'),
    'ckmb': {'n': len(C), 'auc_ckmb': auc_ckmb, 'auc_hsctn': auc_tnt_c, 'crosstab': disc.to_dict()},
    'sensitivity_poor': sensitivity.to_dict('records'),
    'measured_n_primary': {c: int(P_TNT[c].notna().sum()) for c in ['lac_early', 'bnp_early', 'lvef_early_min', 'qtcf_early']},
    'measured_n_primary_by_injury': {c: [int(P_TNT[P_TNT.injury][c].notna().sum()), int(P_TNT[~P_TNT.injury][c].notna().sum())]
                                     for c in ['lac_early', 'bnp_early', 'lvef_early_min', 'qtcf_early']},
    'measured_n_det': {c: int(P_DET[c].notna().sum()) for c in ['lac_early', 'bnp_early', 'lvef_early_min', 'qtcf_early']},
    'admission_range': [idx.admit.min().strftime('%Y-%m'), idx.admit.max().strftime('%Y-%m')],   # months only: no exact admission dates in outputs
    'qtcb_ge460': frac_txt(P_QT.qtb_prolonged.sum(), len(P_QT)), 'qtb_by_group': {g: list(v) for g, v in pheno.set_index('group')['_qtb'].items()},
    'p_qtcb_groups': p_qtb, 'p_groups_monte_carlo': p_mc,
    'deterioration_poor_7d': int(P_DET.poor_7d.sum()), 'deterioration_poor_any': int(P_DET.poor.astype(bool).sum()),
    'asem_candidates': ev[(ev.mri_diffuse_edema.fillna(False).astype(bool) | ev.ct_diffuse_edema.fillna(False).astype(bool)) & ev.shock & (ev.mof_count >= 3)]
                       .ites_class.value_counts().to_dict(),
    'definite_ites_without_fever': int((ev.ites_tier.eq('Definite ITES') & ~ev.fever).sum()),
    'definite_ites_without_altered_consciousness': int((ev.ites_tier.eq('Definite ITES') & ~ev.encephalopathy_text).sum()),
    'arrest_excluded_tnt': int((icu.tnt_early.notna() & icu.arrest_before_tnt).sum()),
    'fragility_index_deterioration': int(frag_index), 'fisher_p_deterioration': float(frag_fisher_p),
    'dama_review_index': idx[idx.ama].vis.map(FZ.dama_review_decision).fillna('not reviewed').value_counts().to_dict(),
    'critical_ama_final_vs_rule': [int(idx.critical_ama.sum()), int(idx.critical_ama_rule.sum())],
    'kruskal_note': 'hs-cTn across groups: Kruskal-Wallis; proportions: chi-square',
}
(OUT / 'manuscript_numbers.json').write_text(json.dumps(MS, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print('Hours from admission to first hs-cTn:', MS['cohort']['hours_admission_to_first_hsctn'], '| escalation within 7 days, hours:', MS['deterioration']['hours_to_escalation_within_7d'])

In [ ]:
# Export the analysis tables and key numbers (no identifiers).
with pd.ExcelWriter(OUT / 'Tables_AES_cardiac.xlsx') as xw:
    flow.to_excel(xw, sheet_name='Cohort_flow', index=False)
    table1.to_excel(xw, sheet_name='Table1_by_injury', index=False)
    table2.to_excel(xw, sheet_name='Table2_phenotype', index=False)
    table3.to_excel(xw, sheet_name='Table3_associations', index=False)
    table4.to_excel(xw, sheet_name='Table4_incremental_AUC', index=False)
    table5.drop(columns=['_num']).to_excel(xw, sheet_name='Table5_time_anchored', index=False)
    pd.DataFrame([ow]).to_excel(xw, sheet_name='Overlap_weighting', index=False)
    balance.to_excel(xw, sheet_name='S1_PS_balance', index=False)
    tab_tested.to_excel(xw, sheet_name='S2_tested_vs_not', index=False)
    sensitivity.to_excel(xw, sheet_name='S3_sensitivity', index=False)
    brain_heart.drop(columns=['_p1', '_p0', '_n']).to_excel(xw, sheet_name='S4_brain_heart', index=False)
    disc.to_excel(xw, sheet_name='S5_CKMB_vs_hsTn')
    dose.reset_index().to_excel(xw, sheet_name='S6_dose_death_critAMA', index=False)
    dose_det.reset_index().to_excel(xw, sheet_name='S7_dose_deterioration', index=False)
    table_det.to_excel(xw, sheet_name='S8_deterioration_cohort', index=False)
    det_subgroups.to_excel(xw, sheet_name='S9_deterioration_by_ITES', index=False)
key = {'hsctn_url_ngL': URL_NGL, 'hsctn_alt_ngL': ALT_NGL, 'index_admissions': len(idx), 'icu': len(icu), 'primary_cohort': len(P_TNT), 'primary_events': int(P_TNT.poor.sum()),
       'injury_n': int(P_TNT.injury.sum()), 'deterioration_cohort': len(P_DET), 'deterioration_events': int(P_DET.deterioration.sum()),
       'p_trend_poor': p_trend, 'p_trend_deterioration': p_trend_det, 'p_nonlinearity_poor': p_nonlin, 'p_nonlinearity_det': p_nonlin_det,
       'overlap_weighting': ow, 'rr_adjusted_for_support_before_testing': list(rr_pre),
       'incremental_poor': inc_poor, 'incremental_deterioration': inc_det,
       'hours_to_escalation_median_any_time': float(P_DET.hours_to_escalation.median()),
       'hours_to_escalation_iqr_any_time': [float(v) for v in P_DET.hours_to_escalation.quantile([.25, .75])],
       'hours_to_escalation_within_7d': MS['deterioration']['hours_to_escalation_within_7d'],
       'p_interaction_injury_x_ites': p_interaction,
       'p_across_groups': {'injury': p_inj, 'tnt': p_tnt, 'lvef': p_dys, 'qtc': p_qt}}
(OUT / 'key_numbers.json').write_text(json.dumps(key, indent=2, default=float), encoding='utf-8')
print('Saved to', OUT.relative_to(HERE))

## Part 4. Publication tables and figure legends

Formatted as in the submission to *Pediatric Critical Care Medicine*: Times New Roman, three-line tables with the
header row repeated on each page, footnote markers as italic superscript letters, and abbreviations defined in the
footnotes. Main tables are written one per Word file (`Table1.docx`, `Table2.docx`); eTables 1-13 and eFigure 1 are
written to one Word file; every table is also written to `Publication_tables.xlsx`. The STROBE checklist and the
eMethods text of the Supplemental Digital Content are part of the manuscript files and are not generated here.

In [ ]:
# Publication tables 1/4: analysis results and display conventions.
# The publication layer reads only the analysis files written above (manuscript_numbers.json, Tables_AES_cardiac.xlsx,
# Tables_ITES.xlsx, key_numbers.json), so every table cell traces back to an exported analysis result.
# Displayed numbers use half-up rounding; p values have two decimals, three below 0.01, and '< 0.001' below 0.001.
try:
    import docx  # python-docx, for the Word tables
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'python-docx'])
    import docx

MSJ = json.loads((OUT / 'manuscript_numbers.json').read_text(encoding='utf-8'))
TA = pd.read_excel(OUT / 'Tables_AES_cardiac.xlsx', sheet_name=None)
TI = pd.read_excel(OUT / 'Tables_ITES.xlsx', sheet_name=None)
KEYJ = json.loads((OUT / 'key_numbers.json').read_text(encoding='utf-8'))
URL, ALT = KEYJ['hsctn_url_ngL'], KEYJ['hsctn_alt_ngL']
MONTHS = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

def month_year(s):
    t = pd.Timestamp(s)
    return f'{MONTHS[t.month - 1]} {t.year}'

def p_fmt(p):
    p = float(p)
    if p < 0.001:
        return '< 0.001'
    if p < 0.01 and float(rnd(p, 3)) < 0.01:
        return rnd(p, 3)
    two = rnd(p, 2)
    if p < 0.05 and two == '0.05':          # never let rounding turn p < 0.05 into 0.05
        d = 3
        while float(rnd(p, d)) >= 0.05 and d < 6:
            d += 1
        return rnd(p, d)
    if two == '1.00':
        return '> 0.99'
    return two

def p_cell(p):
    if p is None or (isinstance(p, float) and pd.isna(p)) or p == '':
        return ''
    return p_fmt(p)

def p_txt(p, lead='p'):
    v = p_fmt(p)
    return f'{lead} {v}' if v.startswith('<') else f'{lead} = {v}'

def ci_parts(s):
    m = re.match(r'\s*([\d.]+) \(([\d.]+)-([\d.]+)\)', str(s))
    return m.groups() if m else None

def rr_cell(s):
    parts = ci_parts(s)
    return f'{parts[0]} ({parts[1]}–{parts[2]})' if parts else str(s)

def frac_parts(s):
    return re.match(r'(\d+)/(\d+) \(([\d.]+)%\)', s).groups()

def minus(x, d=3):
    return rnd(x, d).replace('-', '−')

def pct_txt(a, b, d=0):
    return f'{rnd(100 * a / b, d)}%'

def cell(v):
    return '' if v is None or (isinstance(v, float) and pd.isna(v)) else str(v)

ev_cell = cell

def clean_iqr(v):
    v = str(v)
    m = re.match(r'^([\d.]+) \(([\d.]+)-([\d.]+)\)$', v)
    return f'{m.group(1)} ({m.group(2)}–{m.group(3)})' if m else v

def _strings(obj):
    if isinstance(obj, str):
        yield obj
    elif isinstance(obj, dict):
        for v in obj.values():
            yield from _strings(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            yield from _strings(v)

def assert_no_placeholders(obj):
    # Unformatted f-string fields (for example "{det_inj[0]}") must never reach the documents; ^{...}/_{...} are markup.
    bad = sorted({m.group(0) for t in _strings(obj) for m in re.finditer(r'(?<![\^_])\{[^{}]*\}', t)})
    if bad:
        raise ValueError(f'Unformatted placeholders found: {bad[:10]}')

COH, INJ, DOA, DET = MSJ['cohort'], MSJ['injury'], MSJ['death_or_critical_ama'], MSJ['deterioration']
INJ_KEY = f'hs-cTn > {URL} ng/L'
T5 = {r['analysis']: r for r in DET['table5']}
SENS = {r['analysis']: r for r in MSJ['sensitivity_poor']}
OW, INC, INCP = DOA['overlap_weighting'], DET['incremental'], MSJ['incremental_poor']
PG, PMC = INJ['p_groups'], MSJ['p_groups_monte_carlo']
DOSE_P, DOSE_D = DOA['dose'], DET['dose']
for _d in DOSE_P + DOSE_D:   # displayed rates re-derived from the counts with half-up rounding
    _d['rate %'] = rnd(100 * _d['events'] / _d['N'], 1) if _d['N'] else ''
_spec = ev[ev.clinician_label.isin(['ANE', 'MERS', 'AESD', 'FIRES', 'SAE'])]
AGREE = (int((_spec.clinician_label == _spec.ites_class).sum()), len(_spec))
start_month, end_month = month_year(MSJ['admission_range'][0]), month_year(MSJ['admission_range'][1])
pre = DOA['adjusted_support_before_testing']
det_primary = T5['Subsequent deterioration within 7 days (primary definition)']
det_lac = T5['Subsequent deterioration, additionally adjusted for lactate']
det_dbl = T5['Subsequent deterioration, hs-cTn per doubling']
det_alt = T5[f'Subsequent deterioration, injury defined as hs-cTn > {ALT} ng/L']
det_ites = T5['Subsequent deterioration, consensus ITES only (definite, related or possible)']
det_inj, det_no = frac_parts(DET['injured']), frac_parts(DET['not_injured'])
poor_inj, poor_no = frac_parts(DOA['injured']), frac_parts(DOA['not_injured'])
primary_poor = frac_parts(COH['primary_poor'])
sens_ites = SENS['Consensus ITES only (definite, related or possible)']
FRAG = MSJ['fragility_index_deterioration']
mn = MSJ['measured_n_primary']
a3 = TA['Table3_associations'].set_index('exposure')
ERA_NAMES = {'Pre-2020': f'{start_month} to December 2019', '2020-Nov 2022': 'January 2020 to November 2022',
             'Dec 2022-Jan 2023': 'December 2022 to January 2023', 'Feb 2023 onward': f'February 2023 to {end_month}'}
GROUP_ORDER = ['ANE', 'MERS', 'Other ITES', 'Possible ITES', 'SAE', 'Non-ITES', 'Unclassifiable']
ABBREV_T1 = ('ANE = acute necrotizing encephalopathy, hs-cTn = high-sensitivity cardiac troponin, IQR = interquartile range, '
             'ITES = infection-triggered encephalopathy syndromes, '
             'IV = intravenous, LVEF = left ventricular ejection fraction, MERS = mild encephalopathy with a reversible splenial lesion, NT-proBNP = '
             'N-terminal pro-B-type natriuretic peptide, QTcF = QT interval corrected with the Fridericia formula, SAE = sepsis-associated encephalopathy')
OTHER_ITES = ('Other ITES comprises acute encephalopathy with biphasic seizures and late reduced diffusion, acute fulminant cerebral edema, '
              'hemiconvulsion-hemiplegia-epilepsy syndrome, and febrile infection-related epilepsy syndrome.')
ABBREV_S1 = ('AESD = acute encephalopathy with biphasic seizures and late reduced diffusion, AFCE = acute fulminant cerebral edema, ANE = acute '
             'necrotizing encephalopathy, ASEM = acute shock with encephalopathy and multiorgan failure, FIRES = febrile infection-related epilepsy '
             'syndrome, HHE = hemiconvulsion-hemiplegia-epilepsy syndrome, ITES = infection-triggered encephalopathy syndromes, MERS = mild '
             'encephalopathy with a reversible splenial lesion, SAE = sepsis-associated encephalopathy')
print(f'Analysis base {COH["analysis_base"]}; primary cohort {COH["primary"]} ({COH["primary_poor"]} with death or critical discharge); '
      f'deterioration cohort {DET["n"]} ({DET["events"]} events); fragility index {FRAG}')

In [ ]:
# Publication tables 2/4: main tables (Table 1 and Table 2, one Word file each) and figure legends.
t1 = TA['Table1_by_injury'].copy()
cols1 = list(t1.columns)
lab_all, lab_inj, lab_no = cols1[1], cols1[2], cols1[3]

def t1_row(char):
    r = t1[t1.Characteristic == char]
    if r.empty:
        raise KeyError(char)
    r = r.iloc[0]
    return [r[lab_all], r[lab_inj], r[lab_no], p_cell(r['P'])]

n_inj = t1_row('N')
table1_rows = [['Characteristic', f'All (n = {n_inj[0]})', f'hs-cTn > {URL} ng/L (n = {n_inj[1]})', f'hs-cTn ≤ {URL} ng/L (n = {n_inj[2]})', 'p']]
sections1 = [
    ('Demographics', [('Age, yr, median (IQR)', 'Age, years, median (IQR)'), ('Age < 1 yr, n (%)', 'Age < 1 year, n (%)'), ('Male, n (%)', 'Male, n (%)')]),
    ('Cardiac and laboratory measures', [
        ('Early hs-cTn, ng/L, median (IQR)', 'Early hs-cTn, ng/L, median (IQR)'),
        ('Early lactate, mmol/L, median (IQR)^{a}', 'Early lactate, mmol/L, median (IQR)'),
        ('Early NT-proBNP, pg/mL, median (IQR)^{a}', 'Early NT-proBNP, pg/mL, median (IQR)'),
        ('Lowest LVEF within 72 hours, %, median (IQR)^{a}', 'LVEF (minimum within 72 h), %, median (IQR)'),
        ('QTcF within 48 hours, ms, median (IQR)^{a}', 'QTc Fridericia (first 48 h), ms, median (IQR)'),
        ('Lactate ≥ 4 mmol/L within 48 hours, n (%)^{b}', 'Lactate >= 4 mmol/L (first 48 h), n (%)')]),
    ('Organ support and complications', [('Invasive ventilation, n (%)^{c}', 'Invasive ventilation, n (%)'),
                                         ('Vasoactive infusion, n (%)^{c}', 'Vasoactive infusion, n (%)'),
                                         ('Status epilepticus, n (%)', 'Status epilepticus, n (%)'), ('Septic shock, n (%)', 'Septic shock (diagnosis), n (%)')]),
    ('Treatment', [('Methylprednisolone, n (%)', 'Methylprednisolone, n (%)'), ('IV immunoglobulin, n (%)', 'IVIG, n (%)'),
                   ('Tocilizumab, n (%)', 'Tocilizumab, n (%)')]),
    ('Outcomes', [('In-hospital death, n (%)', 'In-hospital death, n (%)'), ('Critical discharge, n (%)', 'Critical discharge against advice, n (%)'),
                  ('Death or critical discharge, n (%)', 'Death or critical discharge against advice, n (%)')]),
]
for sec, items in sections1:
    table1_rows.append([f'**{sec}**', '', '', '', ''])
    for lab, key in items:
        vals = t1_row(key)
        table1_rows.append([lab] + [clean_iqr(v) for v in vals[:3]] + [vals[3]])
grp_start = t1.index[t1.Characteristic.astype(str).str.startswith('Syndrome group')][0]
era_start = t1.index[t1.Characteristic.astype(str).str.startswith('Era')][0]
table1_rows.append(['**Syndrome group, n (%)**', '', '', '', p_cell(t1.loc[grp_start, 'P'])])
grp_rows = {str(t1.loc[i, 'Characteristic']).strip(): t1.loc[i] for i in range(grp_start + 1, era_start)}
for gname in GROUP_ORDER:
    if gname in grp_rows:
        r = grp_rows[gname]
        table1_rows.append([f'   {gname}', r[lab_all], r[lab_inj], r[lab_no], ''])
table1_foot = (f'{ABBREV_T1}. {OTHER_ITES} Myocardial injury = early hs-cTn above {URL} ng/L. Critical discharge = discharge against medical advice in '
               'critical condition. Mann-Whitney U test for continuous variables; chi-square test (Yates correction for 2 × 2 tables) or Fisher exact test '
               'for categorical variables. '
               f'^{{a}}Among children with the measurement: lactate, n = {mn["lac_early"]}; NT-proBNP, n = {mn["bnp_early"]}; LVEF, n = '
               f'{mn["lvef_early_min"]}; QTcF, n = {mn["qtcf_early"]}. ^{{b}}Children without a lactate measurement are counted as below 4 mmol/L. '
               '^{c}At any time during the admission.')

T2_ROWS = [('Myocardial injury', det_primary), ('Myocardial injury, additionally adjusted for lactate', det_lac), ('hs-cTn, per doubling', det_dbl),
           (f'Myocardial injury defined as hs-cTn > {ALT} ng/L', det_alt), ('Myocardial injury, definite, related, or possible ITES only^{c}', det_ites)]
table2_rows = [['Outcome and analysis', 'N', 'Events', 'Events/N, injury', 'Events/N, no injury', 'Adjusted RR (95% CI)', 'p']]
table2_rows.append(['**Death or critical discharge (primary cohort)**', '', '', '', '', '', ''])
table2_rows.append(['Myocardial injury, adjusted for organ support started before testing^{a}', str(COH['primary']), primary_poor[0],
                    f'{poor_inj[0]}/{poor_inj[1]}', f'{poor_no[0]}/{poor_no[1]}', f'{rnd(pre[0], 2)} ({rnd(pre[1], 2)}–{rnd(pre[2], 2)})', p_cell(pre[3])])
table2_rows.append(['**Deterioration within 7 days (no organ support at the first hs-cTn report)^{b}**', '', '', '', '', '', ''])
for lab, row in T2_ROWS:
    table2_rows.append([lab, str(row['N']), str(row['events']), ev_cell(row.get('events/N, injury')), ev_cell(row.get('events/N, no injury')),
                        rr_cell(row['RR (95% CI)']), p_cell(row['P'])])
table2_foot = (f'Myocardial injury = early high-sensitivity cardiac troponin (hs-cTn) above {URL} ng/L. Relative risks (RRs) from modified Poisson '
               'regression with robust variance. Critical discharge = discharge against medical advice in critical condition. ^{a}Adjusted for age, sex, '
               'syndrome group (acute necrotizing encephalopathy [ANE] and sepsis-associated encephalopathy [SAE] vs all others), status epilepticus, and '
               'vasoactive infusion and invasive ventilation started before the first hs-cTn report. ^{b}Children receiving neither vasoactive infusion nor '
               'invasive ventilation when the first hs-cTn result was reported; deterioration = start of either, or death or critical discharge, within 7 '
               'days after the report. Models adjusted for age, sex, syndrome group, and status epilepticus (and lactate ≥ 4 mmol/L or not measured, where '
               'stated). ^{c}Definite infection-triggered encephalopathy syndromes (ITES), febrile infection-related epilepsy syndrome or '
               'hemiconvulsion-hemiplegia-epilepsy syndrome, or possible ITES; the SAE term was omitted (no child in this subset had SAE) and the ANE term '
               'was kept. Further sensitivity analyses are given in eTable 11.')

TABLES = [
    {'name': 'Table1', 'title': 'Table 1. Characteristics of the Primary Cohort by Early Myocardial Injury', 'rows': table1_rows,
     'footnote': table1_foot, 'widths': [3900, 1450, 1450, 1450, 700]},
    {'name': 'Table2', 'title': 'Table 2. Early Myocardial Injury Before Organ Support and Subsequent Outcomes', 'rows': table2_rows,
     'footnote': table2_foot, 'widths': [4700, 700, 800, 1300, 1400, 1950, 800], 'landscape': True},
]

n_lvef, n_bnp, n_qt = int(a3.loc['LVEF < 55%', 'N']), int(a3.loc['NT-proBNP, per doubling', 'N']), int(a3.loc['QTc (Fridericia) >= 460 ms', 'N'])
FIGURES = [
    {'name': 'Figure1', 'legend':
        f'**Figure 1.** Cardiac involvement across syndrome groups in children with acute encephalopathy in the PICU. **A**, Early high-sensitivity '
        f'cardiac troponin above the upper reference limit of {URL} ng/L ({p_txt(PG["injury"])}, chi-square test; Monte Carlo {p_txt(PMC["injury"])}). '
        f'**B**, Early high-sensitivity cardiac troponin concentrations (log scale); boxes show medians and interquartile ranges, whiskers extend to 1.5 '
        f'times the interquartile range, points are individual children, and the dashed line marks the upper reference limit ({p_txt(PG["hsctn"])}, '
        f'Kruskal-Wallis test). **C**, Left ventricular ejection fraction below 55% within 72 hours of admission ({p_txt(PG["lvef"])}; Monte Carlo '
        f'{p_txt(PMC["lvef"])}). **D**, QT interval corrected with the Fridericia formula of 460 ms or longer within 48 hours ({p_txt(PG["qtc"])}; Monte '
        f'Carlo {p_txt(PMC["qtc"])}). Numbers above bars are children with the finding/children with the measurement; Monte Carlo p values are from 10,000 '
        f'permutations. Colors denote consensus categories: blue, infection-triggered encephalopathy syndromes (ITES; definite, related, or possible); '
        f'orange, sepsis-associated encephalopathy (SAE); gray, non-ITES or unclassifiable. ANE = acute necrotizing encephalopathy, hs-cTn = '
        f'high-sensitivity cardiac troponin, LVEF = left ventricular ejection fraction, MERS = mild encephalopathy with a reversible splenial lesion, '
        f'QTc = corrected QT interval. {OTHER_ITES}'},
    {'name': 'Figure2', 'legend':
        f'**Figure 2.** Adjusted relative risks of death or critical discharge (in-hospital death or discharge against medical advice in critical '
        f'condition) and of deterioration within 7 days. Squares are relative risks and horizontal lines 95% CIs (log scale); the dashed line marks a '
        f'relative risk of 1. Myocardial injury = early high-sensitivity cardiac troponin (hs-cTn) above {URL} ng/L. Myocardial injury rows for death or '
        f'critical discharge are from the primary cohort (n = {COH["primary"]}): model 1 is adjusted for age and sex, model 2 also for syndrome group, and '
        f'model 3 also for invasive ventilation, vasoactive infusion, and status epilepticus; "support before testing" replaces organ support at any time '
        f'with organ support started before the first hs-cTn report, and overlap weighting uses a propensity model (eTable 8). Rows for left ventricular '
        f'ejection fraction (LVEF), N-terminal pro-B-type natriuretic peptide (NT-proBNP), and the QT interval corrected with the Fridericia formula '
        f'(QTcF) are from PICU children with the measurement (n = {n_lvef}, {n_bnp}, and {n_qt}). Deterioration (start of vasoactive infusion or invasive '
        f'ventilation, or death or critical discharge, within 7 days after the first hs-cTn report) was analyzed in children without organ support at the '
        f'report (n = {DET["n"]}), adjusted for age, sex, syndrome group, and status epilepticus (and lactate where stated). Estimates are given in '
        f'Table 2 and eTable 7.'},
    {'name': 'Figure3', 'legend':
        f'**Figure 3.** Early high-sensitivity cardiac troponin (hs-cTn) concentration and outcomes. **A**, Death or critical discharge (in-hospital '
        f'death or discharge against medical advice in critical condition) across hs-cTn categories defined by multiples of the upper reference limit, '
        f'in the primary cohort (n = {COH["primary"]}; {p_txt(DOA["p_trend"])} for trend, Cochran-Armitage test). **B**, Deterioration (start of vasoactive '
        f'infusion or invasive ventilation, or death or critical discharge, within 7 days after the first hs-cTn report) in children without organ support '
        f'at the report (n = {DET["n"]}; {p_txt(DET["p_trend"])} for trend). Numbers above bars are events/children. **C**, Age-adjusted odds ratio of '
        f'deterioration across early hs-cTn (restricted cubic spline of log2 hs-cTn with knots at the 10th, 50th, and 90th percentiles; reference, {URL} '
        f'ng/L; shaded band, 95% CI; {p_txt(DET["p_nonlinearity"])} for nonlinearity). The dashed vertical line marks the upper reference limit and the '
        f'dotted horizontal line an odds ratio of 1.'},
]
for f in FIGURES:
    for ext in ('eps', 'pdf', 'png'):
        if not (FIG / f'{f["name"]}.{ext}').exists():
            raise FileNotFoundError(FIG / f'{f["name"]}.{ext}')
assert_no_placeholders(TABLES)
assert_no_placeholders(FIGURES)
print('Main tables:', [t['name'] for t in TABLES], '| figure legends:', [f['name'] for f in FIGURES])

In [ ]:
# Publication tables 3/4: Supplemental Digital Content tables (eTables 1-13) and eFigure 1.
SUPP_ITEMS = []

def supp_table(title, rows, footnote='', widths=None, landscape=False):
    SUPP_ITEMS.append({'type': 'table', 'title': title, 'rows': rows, 'footnote': footnote, 'widths': widths or [], 'landscape': landscape})

crit = TI['Operational_criteria']
supp_table('eTable 1. Operational Criteria Used to Apply the 2024 Consensus Definitions of Infection-Triggered Encephalopathy Syndromes',
           [['Domain', 'Item', 'Operational definition']] + [[cell(r['domain']), cell(r['item']), cell(r['operational definition'])] for _, r in crit.iterrows()],
           f'Definitions follow Sakuma et al, *Dev Med Child Neurol* 2025;67:195–207. {ABBREV_S1}.', [1500, 2200, 5660])

flow_t = TA['Cohort_flow']
flow_lab = {'AES patients (any admission)': 'Children with an acute encephalopathy diagnosis (any admission)',
            'Acute-phase index admission': 'Acute-phase index admission',
            'Index admission with recorded disposition (analysis base)': 'Index admission with recorded discharge disposition',
            'ICU stay during index admission': 'PICU stay during the index admission',
            'ICU: hs-cTn within 48 h of admission': 'PICU: hs-cTn within 48 hours of admission',
            'ICU: hs-cTn excluded for cardiac arrest before or on the day of measurement':
                'PICU: excluded for documented cardiac arrest not dated after the day of hs-cTn measurement',
            'Primary hs-cTn cohort': 'Primary cohort',
            'LVEF cohort (echo within 72 h, arrest rule applied)': 'PICU: LVEF within 72 hours (cardiac arrest rule applied)',
            'QTc cohort (ECG within 48 h, arrest rule applied)': 'PICU: QTc within 48 hours (cardiac arrest rule applied)',
            'NT-proBNP cohort (within 48 h, arrest rule applied)': 'PICU: NT-proBNP within 48 hours (cardiac arrest rule applied)',
            'Primary cohort without organ support at the first hs-cTn report (deterioration cohort)': 'Primary cohort without organ support at the first hs-cTn report'}
unmapped = [s for s in flow_t.step if s not in flow_lab]
if unmapped:
    raise KeyError(f'Unmapped cohort-flow steps: {unmapped}')
rows = [['Step', 'N', 'Death or critical discharge, n', 'Deterioration within 7 days, n']]
for _, r in flow_t.iterrows():
    rows.append([flow_lab[r.step], cell(int(r.n)), '' if pd.isna(r.iloc[2]) else str(int(r.iloc[2])), '' if pd.isna(r.iloc[3]) else str(int(r.iloc[3]))])
supp_table('eTable 2. Cohort Derivation', rows, 'Cardiac arrest rule: children with documented cardiac arrest or cardiopulmonary resuscitation were '
           'excluded from a measurement-specific analysis unless the arrest was dated to a later calendar day than the measurement (undated arrests were '
           'excluded). hs-cTn = high-sensitivity cardiac troponin, LVEF = left ventricular ejection fraction, NT-proBNP = N-terminal pro-B-type natriuretic '
           'peptide, QTc = corrected QT interval.', [5400, 900, 1600, 1600])


def char_table(df, title, foot, c1, c2):
    rename = {'Age, years': 'Age, yr', 'Age < 1 year': 'Age < 1 yr', 'IVIG': 'IV immunoglobulin',
              'Death or critical discharge against advice': 'Death or critical discharge', 'Critical discharge against advice': 'Critical discharge',
              'Lactate >= 4 mmol/L (first 48 h)': 'Lactate ≥ 4 mmol/L within 48 hours', 'LVEF (minimum within 72 h)': 'Lowest LVEF within 72 hours',
              'QTc Fridericia (first 48 h)': 'QTcF within 48 hours', 'Septic shock (diagnosis)': 'Septic shock',
              'Syndrome group (ITES re-classification)': 'Syndrome group, n (%)', 'Era': 'Admission period, n (%)'}
    out = [['Characteristic', 'All', c1, c2, 'p']]
    block, pending = None, []

    def flush():
        order = GROUP_ORDER if block == 'group' else list(ERA_NAMES)
        pending.sort(key=lambda x: order.index(x[0]) if x[0] in order else 99)
        for name, vals in pending:
            out.append(['   ' + (ERA_NAMES.get(name, name) if block == 'era' else name)] + vals)
        pending.clear()

    for _, r in df.iterrows():
        raw = str(r.Characteristic)
        is_level = raw.startswith('   ')
        name = raw.strip()
        vals = [cell(r.iloc[k]) for k in (1, 2, 3)]
        if 'median' in name:
            vals = [clean_iqr(v) for v in vals]
        if is_level and block:
            pending.append((name, vals + ['']))
            continue
        if pending:
            flush()
        for a, b in rename.items():
            if name.startswith(a):
                name = name.replace(a, b, 1)
                break
        block = 'group' if name.startswith('Syndrome group') else ('era' if name.startswith('Admission period') else None)
        out.append([name] + (['', '', ''] if block else vals) + [p_cell(r.P)])
    if pending:
        flush()
    supp_table(title, out, foot, [3600, 1500, 1500, 1500, 800])


char_table(TA['S2_tested_vs_not'], 'eTable 3. PICU Children With and Without High-Sensitivity Cardiac Troponin T Measured Within 48 Hours of Admission',
           'Organ support at any time during the admission. Abbreviations as in Table 1 of the main text.', 'Tested', 'Not tested')

cs = TI['Class_summary']
rows = [['Category', 'Tier', 'N (% of cohort)', 'Age, yr, median', 'PICU stay, n', 'Deaths, n', 'Death or critical discharge, n', 'Flagged for review, n']]
for _, r in cs.iterrows():
    rows.append([cell(r.ites_class), cell(r.tier), f'{int(r.N)} ({rnd(100 * int(r.N) / COH["analysis_base"], 1)})', rnd(r.age_median, 1), cell(int(r.icu)),
                 cell(int(r.deaths)), cell(int(r.death_or_critical_ama)), cell(int(r.needs_review))])
ASEM_C = MSJ['asem_candidates']
ASEM_PHRASE = {'ANE': 'had bilateral thalamic lesions (ANE)', 'SAE': 'had septic shock (SAE)',
               'Non-ITES: alternative diagnosis': 'had an alternative diagnosis', 'Non-ITES: encephalitis/meningitis': 'had cerebrospinal fluid pleocytosis',
               'Non-ITES: neonatal encephalopathy': 'were neonates'}
_asem = [f'{v} {ASEM_PHRASE.get(k, "were classified as " + k)}' for k, v in sorted(ASEM_C.items(), key=lambda kv: -kv[1])]
if not _asem:
    asem_note = 'No child was classified as ASEM.'
else:
    _joined = ', '.join(_asem[:-1]) + (', and ' if len(_asem) > 2 else ' and ') + _asem[-1] if len(_asem) > 1 else _asem[0]
    asem_note = (f'No child was classified as ASEM: of {sum(ASEM_C.values())} children with diffuse cerebral edema, shock, and at least three organ '
                 f'dysfunction items, {_joined}.')
supp_table(f'eTable 4. Reclassification of {COH["analysis_base"]} Index Admissions With the 2024 Consensus Definitions', rows,
           f'{asem_note} Clinician diagnosis and algorithm agreed for {AGREE[0]} of {AGREE[1]} children with a specific clinician '
           f'diagnosis ({pct_txt(AGREE[0], AGREE[1], 1)}). Abbreviations as in eTable 1.', [2600, 1900, 1000, 800, 800, 700, 1000, 900], landscape=True)
ag = TI['Agreement_vs_clinician']
cols = [c for c in ag.columns if c != 'ites_class']
rows = [['Algorithmic category'] + [c.replace('Other/unspecified', 'Other or unspecified') for c in cols]]
for _, r in ag.iterrows():
    rows.append([cell(r.ites_class)] + [cell(int(r[c])) for c in cols])
supp_table('eTable 5. Algorithmic Category Versus the Clinician\'s Discharge Diagnosis', rows,
           'Columns show the encephalopathy syndrome named in the discharge diagnosis (septic encephalopathy shown as SAE). Abbreviations as in eTable 1.',
           [3200] + [900] * len(cols), landscape=True)

t2 = TA['Table2_phenotype'].copy()
rows = [['Syndrome group', 'PICU children, n', f'hs-cTn > {URL} ng/L, n/N (%)', 'hs-cTn, ng/L, median (IQR)', 'LVEF < 55%, n/N (%)',
         'QTcF ≥ 460 ms, n/N (%)', 'QTcB ≥ 460 ms, n/N (%)', 'NT-proBNP, pg/mL, median (IQR)']]
for _, r in t2.iterrows():
    rows.append([r['group'], str(int(r['ICU N'])), cell(r[f'hs-cTn > {URL} ng/L']), clean_iqr(r['hs-cTn median (IQR), ng/L']), cell(r['LVEF < 55%']),
                 cell(r['QTcF >= 460 ms']), cell(r['QTcB >= 460 ms']), clean_iqr(r['NT-proBNP median (IQR), pg/mL'])])
rows.append(['p, across groups^{a}', '', p_cell(PG['injury']), p_cell(PG['hsctn']), p_cell(PG['lvef']), p_cell(PG['qtc']), p_cell(MSJ['p_qtcb_groups']),
             p_cell(PG['bnp'])])
rows.append(['p, Monte Carlo^{b}', '', p_cell(PMC['injury']), '', p_cell(PMC['lvef']), p_cell(PMC['qtc']), p_cell(PMC['qtcb']), ''])
supp_table('eTable 6. Cardiac Involvement Across Syndrome Groups in PICU Children', rows,
           'Each measure was analyzed in PICU children with the measurement in its window (hs-cTn, QTc, and NT-proBNP within 48 hours of admission; LVEF '
           'within 72 hours), after excluding children with documented cardiac arrest not dated after the day of the measurement; denominators therefore '
           'differ between columns. QTcB = QT interval corrected with the Bazett formula; other abbreviations as in Table 1 of the main text. ^{a}Chi-square '
           'test for proportions and Kruskal-Wallis test for concentrations. ^{b}Monte Carlo p value from 10,000 permutations of the chi-square statistic.',
           [1500, 950, 1550, 1550, 1450, 1450, 1450, 1650], landscape=True)

T3_ROWS = [(f'hs-cTn > {URL} ng/L', INJ_KEY), ('hs-cTn, per doubling', 'hs-cTn, per doubling'), ('LVEF < 55%', 'LVEF < 55%'),
           ('NT-proBNP, per doubling', 'NT-proBNP, per doubling'), ('QTcF ≥ 460 ms', 'QTc (Fridericia) >= 460 ms')]
rows = [['Exposure', 'N', 'Events', 'Events/N, exposed', 'Events/N, unexposed', 'Model 1, RR (95% CI)', 'p', 'Model 2, RR (95% CI)', 'p',
         'Model 3, RR (95% CI)', 'p']]
for lab, key in T3_ROWS:
    r = a3.loc[key]
    row = [lab + ('^{a}' if key in (INJ_KEY, 'hs-cTn, per doubling') else '^{b}'), str(int(r['N'])), str(int(r['events'])),
           ev_cell(r.get('events/N exposed')), ev_cell(r.get('events/N unexposed'))]
    for mname in ('Model 1', 'Model 2', 'Model 3'):
        row += [rr_cell(r[mname]), p_cell(r[mname + ' P'])]
    rows.append(row)
supp_table('eTable 7. Cardiac Measures and Death or Critical Discharge', rows,
           'Death or critical discharge = in-hospital death or discharge against medical advice in critical condition. Relative risks (RRs) from modified '
           'Poisson regression with robust variance. Model 1: age and sex. Model 2: model 1 plus syndrome group (ANE and SAE vs all others). Model 3: model 2 '
           'plus invasive ventilation and vasoactive infusion (each at any time during the admission) and status epilepticus. ^{a}Primary cohort. ^{b}PICU '
           'children with the measurement in its window (LVEF within 72 hours; NT-proBNP and QTcF within 48 hours of admission), after the cardiac arrest '
           'rule. Abbreviations as in Table 1 of the main text.', [2050, 550, 700, 950, 1050, 1500, 650, 1500, 650, 1500, 650], landscape=True)

bal = TA['S1_PS_balance']
cov_lab = {'age_y': 'Age, yr', 'age_lt1': 'Age < 1 yr', 'male': 'Male', 'group_ane': 'ANE', 'group_sae': 'SAE', 'ventilation': 'Invasive ventilation',
           'vasoactive': 'Vasoactive infusion', 'status_epilepticus': 'Status epilepticus', 'septic_shock_i': 'Septic shock', 'lac_ge4_i': 'Lactate ≥ 4 mmol/L',
           'lac_missing': 'Lactate not measured'}
rows = [['Covariate', 'Standardized mean difference, unweighted', 'Standardized mean difference, overlap weighted']]
for _, r in bal.iterrows():
    rows.append([cov_lab.get(r.covariate, r.covariate), minus(r.iloc[1], 3), rnd(abs(r.iloc[2]), 3)])
rows.append(['**Weighted estimates**', '', ''])
rows.append(['Risk of death or critical discharge, myocardial injury', f'{rnd(100 * OW["risk_injury"], 1)}%', ''])
rows.append(['Risk of death or critical discharge, no myocardial injury', f'{rnd(100 * OW["risk_no_injury"], 1)}%', ''])
rows.append(['Relative risk (95% CI)', f'{rnd(OW["RR"], 2)} ({rnd(OW["RR_lo"], 2)}–{rnd(OW["RR_hi"], 2)})', ''])
rows.append(['Risk difference, percentage points (95% CI)',
             f'{minus(100 * OW["RD"], 1)} ({minus(100 * OW["RD_lo"], 1)} to {minus(100 * OW["RD_hi"], 1)})', ''])
supp_table('eTable 8. Overlap Weighting for Myocardial Injury and Death or Critical Discharge (Primary Cohort)', rows,
           'Overlap weights balance the covariates of the propensity model exactly. The propensity model included age, age < 1 year, sex, syndrome group '
           '(ANE and SAE), invasive ventilation, vasoactive infusion, status epilepticus, septic shock, and lactate (≥ 4 mmol/L, or not measured). '
           f'95% CIs from {int(OW["bootstrap_reps"]):,} bootstrap samples. Abbreviations as in Table 1 of the main text.', [4400, 2500, 2500])

sens_t = TA['S3_sensitivity']
sens_lab = {'Primary (early hs-cTn, ICU, arrest rule)': 'Primary analysis (model 3)', 'Peak hs-cTn during admission': 'Peak hs-cTn during the admission',
            'Excluding every patient with any cardiac arrest/CPR': 'Excluding every child with documented cardiac arrest',
            'Age >= 1 year only': 'Age ≥ 1 yr', 'Outcome: death or any discharge against advice': 'Outcome: death or any discharge against medical advice',
            f'Injury defined as hs-cTn > {ALT} ng/L': f'Myocardial injury defined as hs-cTn > {ALT} ng/L',
            'Outcome: critical discharge by the documentation rule (no clinician review)': 'Outcome: critical discharge by an earlier, stricter documentation rule without clinician review',
            'Consensus ITES only (definite, related or possible)': 'Definite, related, or possible ITES only^{a}',
            'Outcome: in-hospital death only (Fisher exact, unadjusted)': 'Outcome: in-hospital death (Fisher exact test, unadjusted)^{b}',
            'Logistic regression (odds ratio), Model 3': 'Logistic regression, model 3 (odds ratio)'}
unmapped = [s for s in sens_t.analysis if s not in sens_lab]
if unmapped:
    raise KeyError(f'Unmapped sensitivity rows: {unmapped}')
rows = [['Analysis', 'N', 'Events', 'Estimate (95% CI)', 'p']]
for _, r in sens_t.iterrows():
    est = cell(r.iloc[3])
    rows.append([sens_lab[r.analysis], cell(int(r.N)), cell(int(r.events)), rr_cell(est) if ci_parts(est) else est, p_cell(r.P)])
rows.append(['**Event rates by early hs-cTn category**', '', '', '', ''])
for d in DOSE_P:
    rows.append([f'   hs-cTn {d["cat"]} ng/L', cell(d['N']), cell(d['events']), f'{d["rate %"]}%', ''])
rows.append(['   p for trend (Cochran-Armitage)', '', '', '', p_cell(DOA['p_trend'])])
rows.append(['**Incremental discrimination**^{c}', '', '', '', ''])
rows.append(['   AUC, base model vs base model + log2 hs-cTn', '', '', f'{rnd(INCP["AUC base"], 3)} vs {rnd(INCP["AUC + hs-cTn"], 3)}', p_cell(INCP['LR test P'])])
rows.append(['   Optimism-corrected AUC', '', '', f'{rnd(INCP["AUC base (optimism-corrected)"], 3)} vs {rnd(INCP["AUC + hs-cTn (optimism-corrected)"], 3)}', ''])
n_cov_ites = 6
supp_table('eTable 9. Sensitivity Analyses and Incremental Discrimination for Death or Critical Discharge', rows,
           'Estimates are relative risks from model 3 (age, sex, syndrome group, invasive ventilation, vasoactive infusion, and status epilepticus) unless '
           'stated. ^{a}The SAE term was omitted (no child in this subset had SAE) and the ANE term kept. ^{b}Estimate column shows deaths/children with '
           'vs without myocardial injury. '
           '^{c}Base model: age, sex, syndrome group, invasive ventilation, vasoactive infusion, and status epilepticus; p value from the likelihood ratio '
           'test. AUC = area under the receiver operating characteristic curve; other abbreviations as in Table 1 of the main text.', [4600, 700, 800, 1900, 800])

char_table(TA['S8_deterioration_cohort'], 'eTable 10. Children Without Organ Support at the First Troponin T Report, by Myocardial Injury',
           'Invasive ventilation and vasoactive infusion in this table refer to any time during the admission (that is, after the first hs-cTn report). '
           'Abbreviations as in Table 1 of the main text.', f'hs-cTn > {URL} ng/L', f'hs-cTn ≤ {URL} ng/L')

t5_t = TA['Table5_time_anchored']
t5_lab = {'Death or critical AMA, adjusted for organ support present before testing': 'Death or critical discharge, adjusted for support before testing',
          'Subsequent deterioration within 7 days (primary definition)': 'Deterioration within 7 days (primary)',
          'Subsequent deterioration, additionally adjusted for lactate': 'Deterioration, additionally adjusted for lactate',
          'Subsequent deterioration, hs-cTn per doubling': 'Deterioration, hs-cTn per doubling',
          'Deterioration defined with vasopressors only (dobutamine/milrinone excluded)': 'Deterioration defined with vasopressors only',
          'Deterioration without ventilation (vasoactive start or death/critical AMA)': 'Deterioration without ventilation in the definition',
          'Subsequent deterioration, age >= 1 year': 'Deterioration, age ≥ 1 yr',
          'Deterioration with escalation within 72 h': 'Deterioration, all components within 72 hours',
          'Deterioration with escalation at any time during admission': 'Deterioration, all components at any time during the admission',
          f'Subsequent deterioration, injury defined as hs-cTn > {ALT} ng/L': f'Deterioration, myocardial injury defined as hs-cTn > {ALT} ng/L',
          'Subsequent deterioration, consensus ITES only (definite, related or possible)': 'Deterioration, definite, related, or possible ITES only^{a}',
          'Subsequent deterioration, excluding non-ITES and unclassifiable': 'Deterioration, excluding non-ITES and unclassifiable children'}
unmapped = [s for s in t5_t.analysis if s not in t5_lab]
if unmapped:
    raise KeyError(f'Unmapped time-anchored rows: {unmapped}')
rows = [['Analysis', 'N', 'Events', 'Events/N, injury', 'Events/N, no injury', 'RR (95% CI)', 'p']]
for _, r in t5_t.iterrows():
    rows.append([t5_lab[r.analysis], cell(int(r.N)), cell(int(r.events)), cell(r['events/N, injury']), cell(r['events/N, no injury']),
                 rr_cell(r['RR (95% CI)']), p_cell(r.P)])
rows.append(['**Deterioration by early hs-cTn category**', '', '', '', '', '', ''])
for d in DOSE_D:
    rows.append([f'   hs-cTn {d["cat"]} ng/L', cell(d['N']), cell(d['events']), '', '', f'{d["rate %"]}%', ''])
rows.append(['   p for trend (Cochran-Armitage)', '', '', '', '', '', p_cell(DET['p_trend'])])
rows.append(['**Incremental discrimination for deterioration**', '', '', '', '', '', ''])
rows.append(['   AUC, base model vs base model + log2 hs-cTn', '', '', '', '', f'{rnd(INC["AUC base"], 3)} vs {rnd(INC["AUC + hs-cTn"], 3)}',
             p_cell(INC['LR test P'])])
rows.append(['   AUC difference (95% CI)', '', '', '', '', f'{rnd(INC["delta AUC"], 3)} ({minus(INC["delta lo"], 3)} to {minus(INC["delta hi"], 3)})', ''])
rows.append(['   Optimism-corrected AUC', '', '', '', '',
             f'{rnd(INC["AUC base (optimism-corrected)"], 3)} vs {rnd(INC["AUC + hs-cTn (optimism-corrected)"], 3)}', ''])
rows.append(['**Fragility of the primary association**', '', '', '', '', '', ''])
rows.append(['   Fragility index (unadjusted Fisher exact test)', '', '', f'{det_inj[0]}/{det_inj[1]}', f'{det_no[0]}/{det_no[1]}', str(FRAG),
             p_cell(MSJ['fisher_p_deterioration'])])
supp_table('eTable 11. Time-Anchored Sensitivity Analyses, Dose-Response, and Incremental Discrimination for Deterioration', rows,
           'Deterioration as in Table 2 of the main text (within 72 hours or at any time where stated); models adjusted for age, sex, syndrome group, and '
           'status epilepticus. ^{a}SAE term omitted (no child in this subset had SAE); ANE term kept. Discrimination base model: age, sex, syndrome group, '
           'and status epilepticus; p value from the likelihood ratio test. Fragility index: number of children without myocardial injury whose outcome would '
           'have to change before the unadjusted Fisher exact p value reached 0.05 (eMethods). AUC = area under the receiver operating characteristic curve, '
           'RR = relative risk.', [4700, 600, 700, 1000, 1100, 1900, 700], landscape=True)


def frac_cell(s):
    a, b = re.match(r'(\d+)/(\d+)', s).groups()
    a, b = int(a), int(b)
    return f'{a}/{b} ({rnd(100 * a / b, 1)}%)', a / b


rows = [['Subgroup', 'N', 'Deterioration, myocardial injury', 'Deterioration, no myocardial injury', 'Unadjusted RR', 'Risk difference, percentage points']]
for r in DET['subgroups']:
    s1, p1 = frac_cell(r['deterioration, injury'])
    s0, p0 = frac_cell(r['deterioration, no injury'])
    rows.append([r['subgroup'].replace('(definite, related or possible)', '(definite, related, or possible)'), cell(r['N']), s1, s0,
                 rnd(p1 / p0, 2) if p0 else '', minus(100 * (p1 - p0), 1)])
supp_table('eTable 12. Deterioration by Myocardial Injury Within Consensus Syndromes and Other Diagnoses', rows,
           f'p for interaction between myocardial injury and consensus ITES status (modified Poisson model with age, sex, syndrome group, and status '
           f'epilepticus) = {p_fmt(DET["p_interaction_ites"])}. Deterioration as in eTable 11. Abbreviations as in Table 1 of the main text.',
           [3100, 600, 1600, 1700, 1200, 1300])

rows = [['Finding', 'Children with the study, n', 'Myocardial injury if present', 'Myocardial injury if absent', 'Adjusted RR (95% CI)', 'p']]
for r in MSJ['brain_heart']:
    rows.append([r['feature'], cell(r['N with study']), r['injury if present'], r['injury if absent'], rr_cell(r['adjusted RR (age, vasoactive)']), p_cell(r['P'])])
supp_table('eTable 13. Neuroimaging and Electroencephalography Findings and Myocardial Injury (Primary Cohort)', rows,
           'Findings from MRI or CT (thalamic lesions and diffuse cerebral edema), MRI (brainstem lesions), and EEG at any time during the index admission. '
           'Myocardial injury if present or absent: children with injury/children with or without the finding. Relative risks adjusted for age and vasoactive '
           'infusion. EEG = electroencephalography, RR = relative risk.', [2700, 1200, 1400, 1400, 1700, 700])

EFIG = {'type': 'figure', 'title': 'eFigure 1. Myocardial Injury by Neuroimaging and Electroencephalography Findings', 'path': 'figures/eFigure1.png',
        'legend': f'**eFigure 1.** Prevalence of myocardial injury (high-sensitivity cardiac troponin above {URL} ng/L) by neuroimaging and '
                  'electroencephalography (EEG) findings in the primary cohort. Blue bars, finding present; orange bars, finding absent; numbers are children '
                  'with injury/children in the group. Estimates are given in eTable 13.'}
SUPP_ITEMS.append(EFIG)
assert_no_placeholders(SUPP_ITEMS)
print('Supplement items:', len(SUPP_ITEMS), '|', ', '.join(it['title'].split('.')[0] for it in SUPP_ITEMS))

In [ ]:
# Publication tables 4/4: Word files (Table1.docx and Table2.docx, one table per file; eTables 1-13 and eFigure 1 in one
# file), figure legends, an Excel copy of every table, and a JSON record of the table contents.
from docx import Document
from docx.enum.section import WD_ORIENT, WD_SECTION
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, Twips
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, Side
from openpyxl.utils import get_column_letter

def tnt(obj):
    # The assay is high-sensitivity cardiac troponin T: displayed text says 'hs-cTnT' (analysis keys keep the generic 'hs-cTn').
    if isinstance(obj, str):
        s_ = re.sub(r'([Hh])igh-sensitivity cardiac troponin(?! [TI]\b)', r'\1igh-sensitivity cardiac troponin T', obj)
        return re.sub(r'hs-cTn(?![TI])', 'hs-cTnT', s_)
    if isinstance(obj, list):
        return [tnt(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(tnt(x) for x in obj)
    if isinstance(obj, dict):
        return {k: tnt(v) for k, v in obj.items()}
    return obj

TABLES, SUPP_ITEMS, FIGURES = tnt(TABLES), tnt(SUPP_ITEMS), tnt(FIGURES)
DOC_FONT = 'Times New Roman'
MARKUP = re.compile(r'(\*\*[^*]+\*\*|\*[^*]+\*|\^\{[^}]*\}|_\{[^}]*\})')   # **bold**, *italic*, ^{superscript}, _{subscript}

def new_document():
    doc = Document()
    st = doc.styles['Normal']
    st.font.name, st.font.size = DOC_FONT, Pt(10)
    st.element.get_or_add_rPr().get_or_add_rFonts().set(qn('w:eastAsia'), DOC_FONT)
    cp = doc.core_properties
    cp.author = cp.last_modified_by = cp.comments = ''
    cp.created = cp.modified = pd.Timestamp.now().to_pydatetime().replace(microsecond=0)
    return doc

def set_page(section, landscape=False):
    section.orientation = WD_ORIENT.LANDSCAPE if landscape else WD_ORIENT.PORTRAIT
    w, h = Inches(8.5), Inches(11)
    section.page_width, section.page_height = (h, w) if landscape else (w, h)
    section.left_margin = section.right_margin = section.top_margin = section.bottom_margin = Inches(1)

def add_run(par, text, size, bold=False, italic=False, sup=False, sub=False):
    r = par.add_run(text)
    r.font.name, r.font.size = DOC_FONT, Pt(size)
    r._element.get_or_add_rPr().get_or_add_rFonts().set(qn('w:eastAsia'), DOC_FONT)
    r.font.bold = True if bold else None
    r.font.italic = True if italic else None
    if sup:
        r.font.superscript = True
    if sub:
        r.font.subscript = True

def add_marked(par, text, size, bold=False, italic=False):
    # Superscript lowercase letters (table footnote markers) are set in italics, as the journal requires.
    pos = 0
    for m in MARKUP.finditer(text):
        if m.start() > pos:
            add_run(par, text[pos:m.start()], size, bold, italic)
        tok = m.group(0)
        if tok.startswith('**'):
            add_marked(par, tok[2:-2], size, True, italic)
        elif tok.startswith('*'):
            add_marked(par, tok[1:-1], size, bold, True)
        elif tok.startswith('^{'):
            add_run(par, tok[2:-1], size, bold, italic or bool(re.fullmatch(r'[a-z]+', tok[2:-1])), sup=True)
        else:
            add_run(par, tok[2:-1], size, bold, italic, sub=True)
        pos = m.end()
    if pos < len(text):
        add_run(par, text[pos:], size, bold, italic)

def add_par(container, text, size, bold=False, align=None, before=0, after=0, keep_next=False):
    p = container.add_paragraph()
    pf = p.paragraph_format
    pf.space_before, pf.space_after, pf.line_spacing, pf.keep_with_next = Pt(before), Pt(after), 1.0, keep_next
    if align is not None:
        p.alignment = align
    add_marked(p, text, size, bold)
    return p

def _edge(tag, on):
    el = OxmlElement(f'w:{tag}')
    if on:
        for k, v in (('val', 'single'), ('sz', '8'), ('space', '0'), ('color', '000000')):
            el.set(qn(f'w:{k}'), v)
    else:
        el.set(qn('w:val'), 'nil')
    return el

def add_table(doc, rows, widths, total, size, left_align_all):
    # Three-line table: rules above and below the header row and below the last row; the header row repeats on each page.
    ncol = len(rows[0])
    w = widths if widths and len(widths) == ncol else [1] * ncol
    tw = [int(x * total / sum(w)) for x in w]
    tw[-1] += total - sum(tw)
    tbl = doc.add_table(rows=len(rows), cols=ncol)
    tbl.autofit = False
    tblPr = tbl._tbl.tblPr
    tblW = tblPr.find(qn('w:tblW'))
    if tblW is None:
        tblW = OxmlElement('w:tblW')
        tblPr.append(tblW)
    tblW.set(qn('w:w'), str(total))
    tblW.set(qn('w:type'), 'dxa')
    layout = OxmlElement('w:tblLayout')
    layout.set(qn('w:type'), 'fixed')
    mar = OxmlElement('w:tblCellMar')
    for side, v in (('top', 12), ('left', 60), ('bottom', 12), ('right', 60)):
        e = OxmlElement(f'w:{side}')
        e.set(qn('w:w'), str(v))
        e.set(qn('w:type'), 'dxa')
        mar.append(e)
    look = tblPr.find(qn('w:tblLook'))
    for el in (layout, mar):
        if look is not None:
            look.addprevious(el)
        else:
            tblPr.append(el)
    for j, col in enumerate(tbl.columns):
        col.width = Twips(tw[j])
    last = len(rows) - 1
    for i, values in enumerate(rows):
        row = tbl.rows[i]
        trPr = row._tr.get_or_add_trPr()
        trPr.append(OxmlElement('w:cantSplit'))
        if i == 0:
            trPr.append(OxmlElement('w:tblHeader'))
        for j, val in enumerate(values):
            c = row.cells[j]
            c.width = Twips(tw[j])
            borders = OxmlElement('w:tcBorders')
            for tag, on in (('top', i == 0), ('left', False), ('bottom', i in (0, last)), ('right', False)):
                borders.append(_edge(tag, on))
            c._tc.get_or_add_tcPr().append(borders)
            c.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            p = c.paragraphs[0]
            p.paragraph_format.space_before = p.paragraph_format.space_after = Pt(0)
            p.paragraph_format.line_spacing = 1.0
            p.alignment = WD_ALIGN_PARAGRAPH.LEFT if (left_align_all or j == 0) else WD_ALIGN_PARAGRAPH.CENTER
            add_marked(p, '' if val is None else str(val), size, bold=(i == 0))
    return tbl

def add_page_number(section):
    p = section.footer.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    for kind in ('begin', 'instr', 'end'):
        r = p.add_run()
        r.font.name, r.font.size = DOC_FONT, Pt(10)
        if kind == 'instr':
            el = OxmlElement('w:instrText')
            el.set(qn('xml:space'), 'preserve')
            el.text = 'PAGE'
        else:
            el = OxmlElement('w:fldChar')
            el.set(qn('w:fldCharType'), kind)
        r._r.append(el)

def write_table_docx(t, path):
    # Journal table file: no center alignment, 10-point text, title above and footnotes below the table.
    doc = new_document()
    set_page(doc.sections[0], t.get('landscape', False))
    add_par(doc, t['title'], 11, bold=True, after=6, keep_next=True)
    add_table(doc, t['rows'], t.get('widths'), 12960 if t.get('landscape') else 9360, 10, left_align_all=True)
    add_par(doc, t.get('footnote', ''), 9, before=6)
    doc.save(path)

def write_supplement_docx(items, path):
    doc = new_document()
    set_page(doc.sections[0])
    add_page_number(doc.sections[0])
    add_par(doc, 'Supplemental Digital Content', 14, bold=True, align=WD_ALIGN_PARAGRAPH.CENTER, after=12)
    add_par(doc, 'Contents', 11, bold=True, after=6)
    for it in items:
        add_par(doc, it['title'], 11, after=3)
    for it in items:
        sec = doc.add_section(WD_SECTION.NEW_PAGE)
        set_page(sec, it.get('landscape', False))
        add_par(doc, it['title'], 11, bold=True, after=6, keep_next=True)
        if it['type'] == 'table':
            add_table(doc, it['rows'], it.get('widths'), 12960 if it.get('landscape') else 9360, 9, left_align_all=False)
            add_par(doc, it.get('footnote', ''), 9, before=6)
        else:
            doc.add_picture(str(PUB / it['path']), width=Inches(6.5))
            doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_par(doc, it['legend'], 10, before=6)
    doc.save(path)

def write_legends_docx(legends, path):
    doc = new_document()
    set_page(doc.sections[0])
    add_par(doc, 'FIGURE LEGENDS', 12, bold=True, after=12)
    for text in legends:
        add_par(doc, text, 12, after=12)
    doc.save(path)

SUP_MAP = str.maketrans('0123456789abcdefghijklmnoprstuvwxyz+-=()', '⁰¹²³⁴⁵⁶⁷⁸⁹ᵃᵇᶜᵈᵉᶠᵍʰⁱʲᵏˡᵐⁿᵒᵖʳˢᵗᵘᵛʷˣʸᶻ⁺⁻⁼⁽⁾')
SUB_MAP = str.maketrans('0123456789+-=()', '₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎')

def plain(text):
    # Markup to plain text for Excel: superscripts and subscripts as Unicode characters, emphasis markers removed.
    text = re.sub(r'\^\{([^}]*)\}', lambda m: m.group(1).translate(SUP_MAP), str(text))
    text = re.sub(r'_\{([^}]*)\}', lambda m: m.group(1).translate(SUB_MAP), text)
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    return re.sub(r'\*([^*]+)\*', r'\1', text)

def write_tables_xlsx(tables, path):
    wb = Workbook()
    wb.remove(wb.active)
    thin, none = Side(style='thin', color='000000'), Side()
    for t in tables:
        ws = wb.create_sheet(t['sheet'])
        rows, ncol = t['rows'], len(t['rows'][0])
        ws.cell(1, 1, plain(t['title'])).font = Font(bold=True, size=11)
        for i, values in enumerate(rows):
            for j, v in enumerate(values):
                txt = '' if v is None else str(v)
                bold = i == 0 or (txt.startswith('**') and txt.rstrip().endswith('**'))
                c = ws.cell(3 + i, 1 + j, plain(txt.strip()) if j == 0 else plain(txt))
                c.font = Font(bold=bold)
                c.alignment = Alignment(wrap_text=True, vertical='center', horizontal='left' if j == 0 else 'center',
                                        indent=2 if (j == 0 and txt.startswith('   ')) else 0)
                c.border = Border(top=thin if i == 0 else none, bottom=thin if i in (0, len(rows) - 1) else none)
        widths = t['widths'] if t.get('widths') and len(t['widths']) == ncol else [1500] * ncol
        for j, w in enumerate(widths):
            ws.column_dimensions[get_column_letter(j + 1)].width = max(8, w / 110)
        foot = plain(t.get('footnote', ''))
        foot_row = 3 + len(rows) + 1
        ws.cell(foot_row, 1, foot).alignment = Alignment(wrap_text=True, vertical='top')
        ws.merge_cells(start_row=foot_row, start_column=1, end_row=foot_row, end_column=ncol)
        chars_per_line = max(40, int(sum(w / 110 for w in widths) * 1.1))
        ws.row_dimensions[foot_row].height = min(409, 15 * (1 + len(foot) // chars_per_line))
        ws.freeze_panes = 'A4'
    wb.save(path)

for t in TABLES:
    write_table_docx(t, TAB / f'{t["name"]}.docx')
write_supplement_docx(SUPP_ITEMS, TAB / 'Supplemental_eTables_eFigure.docx')
LEGENDS = [f['legend'] for f in FIGURES] + [EFIG['legend']]
write_legends_docx(LEGENDS, TAB / 'Figure_legends.docx')
xl_items = [dict(t, sheet=t['name']) for t in TABLES] + \
           [dict(it, sheet=it['title'].split('.')[0].replace(' ', '')) for it in SUPP_ITEMS if it['type'] == 'table']
write_tables_xlsx(xl_items, TAB / 'Publication_tables.xlsx')
record = {'main_tables': TABLES, 'supplement_items': SUPP_ITEMS, 'figure_legends': LEGENDS}
(OUT / 'publication_tables.json').write_text(json.dumps(record, indent=1, ensure_ascii=False), encoding='utf-8')
print('Written:', ', '.join(p.name for p in sorted(TAB.glob('*')) if p.is_file()))
pd.DataFrame([[plain(c) for c in r] for r in table1_rows[1:]], columns=[plain(c) for c in table1_rows[0]])   # preview of Table 1

## Part 5. Cleaned-data workbook (internal use only)

One row per child (index admission) with every cleaned and derived analysis variable, the ITES evidence behind each
category, every hs-cTn, CK-MB, NT-proBNP, lactate, ECG, and LVEF result of the index admissions in long format, the
cohort flow, a data dictionary, and a descriptive summary by analytic population. Times are given as hours from
admission. Free text (discharge summaries, report text) is never exported. **The workbook contains research patient and
visit identifiers and admission dates: keep it within the study team.**

In [ ]:
# Cleaned data 1/2: analysis dataset (one row per index admission), ITES evidence, and repeated measurements (long format).
def hours_from_admission(t):
    return ((t - idx.admit).dt.total_seconds() / 3600).round(1)

def in_pop(df):
    return idx.vis.isin(set(df.vis))

def pop_value(df, col):
    # Value of df[col] for children in the analytic population df; missing for everyone else.
    return idx.vis.map(df.set_index('vis')[col])

def as_bool_na(s):
    return s.astype('boolean')

DISPOSITION_EN = {'医嘱离院': 'Discharged on medical advice', '非医嘱离院': 'Discharged against medical advice', '死亡': 'Died',
                  '医嘱转院': 'Transferred to another hospital', '医嘱转社区卫生服务机构/乡镇卫生院': 'Transferred to a community or township health center',
                  '其他': 'Other'}

# Repeated measurements of the index admissions (same sources and cleaning rules as the analysis).
LAB_SPECS = [('hs-cTn', 'ng/L', 1000, '实验室检查_血液心肌酶谱测定', '超敏心肌钙蛋白定量-定量结果'),
             ('CK-MB activity', 'U/L', 1, '实验室检查_血液心肌酶谱测定', '肌酸激酶同工酶-ＭＢ定量-定量结果'),
             ('NT-proBNP', 'pg/mL', 1, '实验室检查_血液N末端B型钠尿肽前体测定', 'N末端B型利钠肽原定量-定量结果'),
             ('Lactate', 'mmol/L', 1, '实验室检查_血液乳酸测定', '乳酸定量-定量结果')]
parts = []
for name_, unit_, factor_, sheet_, col_ in LAB_SPECS:
    x_ = lab_long(sheet_, col_)
    parts.append(pd.DataFrame({'vis': x_.vis.values, 'measure': name_, 'unit': unit_, 'value': x_.value.values * factor_, 'time': x_.time.values}))
ecg_t = ecg.dropna(subset=['time'])
for name_, col_, unit_ in [('ECG heart rate', 'HR', 'beats/min'), ('QT interval', 'QT', 'ms'), ('QTcF', 'value', 'ms'), ('QTcB', 'QTcB', 'ms')]:
    parts.append(pd.DataFrame({'vis': ecg_t.vis.values, 'measure': name_, 'unit': unit_, 'value': ecg_t[col_].values, 'time': ecg_t.time.values}))
parts.append(pd.DataFrame({'vis': echo.vis.values, 'measure': 'LVEF', 'unit': '%', 'value': echo.value.values, 'time': echo.time.values}))
meas = pd.concat(parts, ignore_index=True).merge(idx[['vis', 'patient', 'admit']], on='vis')
h_ = (meas.time - meas.admit).dt.total_seconds() / 3600
is_echo = meas.measure.eq('LVEF')
meas['in_early_window'] = np.where(is_echo, h_.between(-24, 72), h_.between(-12, 48))   # windows used for the early values
meas['hours_from_admission'] = h_.round(1)
meas['value'] = meas.value.round(4)
meas = (meas.sort_values(['patient', 'vis', 'measure', 'hours_from_admission'])
            [['patient', 'vis', 'measure', 'unit', 'value', 'hours_from_admission', 'in_early_window']].reset_index(drop=True))
mstat = meas.groupby(['measure', 'vis']).value.agg(['max', 'min', 'size'])

def adm_stat(measure, what):
    return idx.vis.map(mstat.loc[measure][what]) if measure in mstat.index.get_level_values(0) else pd.Series(np.nan, index=idx.index)

A = pd.DataFrame({'patient': idx.patient.astype(str), 'vis': idx.vis.astype(str)})
A['admission_date'] = idx.admit.dt.date
A['discharge_date'] = idx.discharge.dt.date
A['admission_year'] = idx.admit.dt.year
A['admission_period'] = idx.era.astype(str)
A['length_of_stay_days'] = ((idx.discharge - idx.admit).dt.total_seconds() / 86400).round(1)
A['age_years'] = idx.age_y
A['age_lt_1y'] = idx.age_lt1.astype(bool)
A['male'] = idx.male.astype(bool)
A['picu_stay'] = idx.icu.astype(bool)
A['diagnosis_label'] = idx.subtype
evi = ev.set_index('vis')
A['clinician_label'] = idx.vis.map(evi.clinician_label)
A['ites_class'] = idx.ites_class
A['ites_class_algorithmic'] = idx.ites_algorithmic
A['ites_tier'] = idx.ites_tier
A['ites_reason'] = idx.vis.map(evi.ites_reason)
A['ites_needs_review'] = idx.needs_review.astype(bool)
A['ites_review_reasons'] = idx.vis.map(evi.review_reasons)
A['syndrome_group'] = idx.group
A['ites_any'] = idx.ites_any.astype(bool)
A['disposition'] = idx.disposition.map(DISPOSITION_EN).fillna(idx.disposition)
A['in_hospital_death'] = idx.death.astype(bool)
A['discharge_against_advice'] = idx.ama.astype(bool)
A['critical_discharge_against_advice'] = idx.critical_ama.astype(bool)
A['critical_discharge_rule_only'] = idx.critical_ama_rule.astype(bool)
A['death_or_critical_discharge'] = idx.poor.astype(bool)
A['hsctn_early_ngL'] = idx.tnt_early_ngL.round(4)
A['hsctn_early_hours'] = hours_from_admission(idx.tnt_early_time)
A['hsctn_max_ngL'] = adm_stat('hs-cTn', 'max')
A['hsctn_n_reports'] = adm_stat('hs-cTn', 'size').fillna(0).astype(int)
A['ckmb_early_U_L'] = idx.ckmb_early
A['ckmb_max_U_L'] = adm_stat('CK-MB activity', 'max')
A['ntprobnp_early_pg_mL'] = idx.bnp_early
A['ntprobnp_early_hours'] = hours_from_admission(idx.bnp_early_time)
A['ntprobnp_max_pg_mL'] = adm_stat('NT-proBNP', 'max')
A['lactate_early_mmol_L'] = idx.lac_early
A['lactate_early_hours'] = hours_from_admission(idx.lac_early_time)
A['lactate_max_mmol_L'] = adm_stat('Lactate', 'max')
A['lactate_ge_4'] = idx.lactate_ge4.astype(bool)
A['qtcf_early_ms'] = idx.qtcf_early.round(1)
A['qtcb_early_ms'] = idx.qtcb_early.round(1)
A['ecg_early_hours'] = hours_from_admission(idx.qtcf_early_time)
A['qtcb_max_ms'] = adm_stat('QTcB', 'max').round(1)
A['lvef_early_min_pct'] = idx.lvef_early_min
A['lvef_min_pct'] = idx.lvef_min
A['invasive_ventilation'] = idx.ventilation.astype(bool)
A['ventilation_start_hours'] = hours_from_admission(idx.vent_time)
A['vasoactive_infusion'] = idx.vasoactive.astype(bool)
A['vasoactive_start_hours'] = hours_from_admission(idx.vaso_time)
A['vasopressor_start_hours'] = hours_from_admission(idx.pressor_time)
A['status_epilepticus'] = idx.status_epilepticus.astype(bool)
A['septic_shock'] = idx.septic_shock.astype(bool)
A['shock_diagnosis'] = idx.shock_dx.astype(bool)
A['cardiac_arrest_documented'] = idx.cpr_any.astype(bool)
A['cardiac_arrest_procedure_day'] = (idx.cpr_date - idx.admit.dt.normalize()).dt.days.astype('Int64')
A['methylprednisolone'] = idx.methylprednisolone.astype(bool)
A['iv_immunoglobulin'] = idx.ivig.astype(bool)
A['tocilizumab'] = idx.tocilizumab.astype(bool)
A['in_primary_cohort'] = in_pop(P_TNT)
A['excluded_arrest_rule_hsctn'] = (idx.icu & idx.tnt_early.notna() & idx.arrest_before_tnt).astype(bool)
A['myocardial_injury'] = as_bool_na(pop_value(P_TNT, 'injury'))
A['myocardial_injury_2x_url'] = as_bool_na(pop_value(P_TNT, 'injury_alt'))
A['in_lvef_cohort'] = in_pop(P_EF)
A['lvef_lt_55'] = as_bool_na(pop_value(P_EF, 'dysfunction'))
A['in_qtc_cohort'] = in_pop(P_QT)
A['qtcf_ge_460'] = as_bool_na(pop_value(P_QT, 'qt_prolonged'))
A['qtcb_ge_460'] = as_bool_na(pop_value(P_QT, 'qtb_prolonged'))
A['in_ntprobnp_cohort'] = in_pop(P_BNP)
A['vasoactive_before_hsctn'] = as_bool_na(pop_value(P_TNT, 'vaso_before'))
A['ventilation_before_hsctn'] = as_bool_na(pop_value(P_TNT, 'vent_before'))
A['in_deterioration_cohort'] = in_pop(P_DET)
A['vasoactive_start_within_7d'] = as_bool_na(pop_value(P_DET, 'vaso_7d'))
A['ventilation_start_within_7d'] = as_bool_na(pop_value(P_DET, 'vent_7d'))
A['death_or_critical_discharge_within_7d'] = as_bool_na(pop_value(P_DET, 'poor_7d'))
A['deterioration_7d'] = as_bool_na(pop_value(P_DET, 'deterioration'))
A['deterioration_72h'] = as_bool_na(pop_value(P_DET, 'deterioration_72h'))
A['deterioration_any_time'] = as_bool_na(pop_value(P_DET, 'deterioration_any'))
A['hours_hsctn_to_escalation'] = pop_value(P_DET, 'hours_to_escalation').astype(float).round(1)
A = A.sort_values(['patient', 'vis']).reset_index(drop=True)

# Consistency with the analysis (the cleaned dataset must reproduce every analytic population and count).
has = idx.tnt_early.notna()
assert np.allclose(adm_stat('hs-cTn', 'max')[has], idx.tnt_peak_ngL[has])
assert int(A.in_primary_cohort.sum()) == len(P_TNT) and int(A.myocardial_injury.sum()) == int(P_TNT.injury.sum())
assert int(A.in_deterioration_cohort.sum()) == len(P_DET) and int(A.deterioration_7d.sum()) == int(P_DET.deterioration.sum())
assert int(A.death_or_critical_discharge[A.in_primary_cohort].sum()) == int(P_TNT.poor.sum())

# ITES evidence: structured flags and values only (free text and report excerpts are never exported).
FREE_TEXT = {'cc', 'hpi', 'pmh', 'pe', 'course', 'dx_text'}
IN_DATASET = {'patient', 'vis', 'admit', 'discharge', 'age_y', 'icu', 'death', 'poor', 'subtype', 'era', 'critical_ama'}
ev_keep = [c for c in ev.columns if c not in FREE_TEXT | IN_DATASET and not c.endswith('_evidence') and not c.startswith(('excl: ', 'possible excl: '))]
EVS = ev[['patient', 'vis'] + ev_keep].copy()
EVS['patient'], EVS['vis'] = EVS.patient.astype(str), EVS.vis.astype(str)
EVS['exclusion_diagnoses'] = ev[[c for c in ev.columns if c.startswith('excl: ')]].apply(
    lambda r: '; '.join(k[len('excl: '):] for k, v in r.items() if v), axis=1)
EVS['qualified_exclusion_diagnoses'] = ev[[c for c in ev.columns if c.startswith('possible excl: ')]].apply(
    lambda r: '; '.join(k[len('possible excl: '):] for k, v in r.items() if v), axis=1)
for c in EVS.columns:
    if EVS[c].dtype == object and set(EVS[c].dropna().map(type)) <= {bool, np.bool_} and EVS[c].notna().any():
        EVS[c] = EVS[c].astype('boolean')
    elif EVS[c].dtype == float:
        EVS[c] = EVS[c].round(3)
FIRST = ['patient', 'vis', 'clinician_label', 'ites_algorithmic', 'adjudicated_class', 'ites_class', 'ites_tier', 'ites_reason', 'needs_review',
         'review_reasons']
EVS = EVS[[c for c in FIRST if c in EVS.columns] + [c for c in EVS.columns if c not in FIRST]]
EVS = EVS.sort_values(['patient', 'vis']).reset_index(drop=True)

TEXT_OK = {'patient', 'vis', 'admission_date', 'discharge_date', 'admission_period', 'diagnosis_label', 'clinician_label', 'ites_class',
           'ites_class_algorithmic', 'ites_algorithmic', 'ites_tier', 'ites_reason', 'ites_review_reasons', 'review_reasons', 'syndrome_group',
           'disposition', 'exclusion_diagnoses', 'qualified_exclusion_diagnoses', 'adjudicated_class', 'measure', 'unit'}
for name_, df_ in [('Analysis_dataset', A), ('ITES_evidence', EVS), ('Measurements_long', meas)]:
    leak = [c for c in df_.columns if df_[c].dtype == object and c not in TEXT_OK]
    if leak:
        raise ValueError(f'{name_}: unexpected text columns {leak}')
print(f'Analysis dataset: {A.shape[0]} children x {A.shape[1]} variables | ITES evidence: {EVS.shape[1]} variables | '
      f'repeated measurements: {len(meas):,} results ({meas.measure.nunique()} measures)')

In [ ]:
# Cleaned data 2/2: data dictionary, descriptive summary by analytic population, and the workbook (plus CSV copies).
from openpyxl.styles import Alignment as XAlign, Font as XFont

S_EXP = 'Main and supplementary exports'
DD_A = {
    'patient': ('Research patient identifier', 'text', '科研患者编号'),
    'vis': ('Research visit identifier of the index admission', 'text', '科研就诊编号'),
    'admission_date': ('Admission date of the index admission', 'date', '入院日期'),
    'discharge_date': ('Discharge date of the index admission', 'date', '出院日期'),
    'admission_year': ('Calendar year of admission', 'year', 'Derived from admission date'),
    'admission_period': ('Admission period', 'Pre-2020; 2020-Nov 2022; Dec 2022-Jan 2023; Feb 2023 onward', 'Derived from admission date'),
    'length_of_stay_days': ('Length of stay', 'days', 'Discharge minus admission time'),
    'age_years': ('Age at admission', 'years', '年龄（岁）'),
    'age_lt_1y': ('Age younger than 1 year', 'TRUE/FALSE', 'Derived'),
    'male': ('Male sex', 'TRUE/FALSE', '性别'),
    'picu_stay': ('Stay in either PICU: critical care department in the admission, transfer, or discharge department fields of the front page, or a '
                  'transfer record into or out of a PICU during the admission', 'TRUE/FALSE', '入院科别; 转科科别; 出院科别; 患者转科信息'),
    'diagnosis_label': ('Acute encephalopathy label parsed from the discharge diagnoses (first matching pattern in a fixed order)',
                        'ANE; HSES_ASEM; Reye; AESD; FIRES; MERS; Septic_encephalopathy; Pertussis_encephalopathy; Influenza_encephalopathy; '
                        'Toxic_encephalopathy; AES_unspecified', 'Discharge record diagnoses (出院诊断); coded diagnoses when no discharge record exists'),
    'clinician_label': ('Specific syndrome named in the discharge diagnosis (used for agreement with the algorithm)', 'ANE; MERS; AESD; FIRES; SAE; Other/unspecified',
                        'Derived from diagnosis_label'),
    'ites_class': ('Consensus category used in all analyses (adjudicated category when an adjudication file is present, otherwise algorithmic)',
                   'ANE; AESD; AFCE; ASEM; MERS; HHE; FIRES; Possible ITES; SAE; Non-ITES (3 types); Unclassifiable', 'ITES re-classification (eTable 1)'),
    'ites_class_algorithmic': ('Category proposed by the algorithm before adjudication', 'as ites_class', 'ITES re-classification'),
    'ites_tier': ('Consensus tier of ites_class', 'Definite ITES; ITES-related; Possible ITES; Sepsis-associated encephalopathy; Non-ITES; Unclassifiable',
                  'ITES re-classification'),
    'ites_reason': ('Rule that assigned the algorithmic category', 'text generated by the algorithm', 'ITES re-classification'),
    'ites_needs_review': ('Flagged for adjudication', 'TRUE/FALSE', 'ITES re-classification'),
    'ites_review_reasons': ('Reasons for the adjudication flag', 'text generated by the algorithm', 'ITES re-classification'),
    'syndrome_group': ('Syndrome group used in tables, figures, and models', 'ANE; MERS; Other ITES (AESD, AFCE, ASEM, HHE, FIRES); Possible ITES; SAE; '
                       'Non-ITES; Unclassifiable', 'Derived from ites_class'),
    'ites_any': ('Definite ITES, ITES-related syndrome, or possible ITES', 'TRUE/FALSE', 'Derived from ites_tier'),
    'disposition': ('Discharge disposition', 'Discharged on medical advice; Discharged against medical advice; Died; Transferred to another hospital; '
                    'Transferred to a community or township health center; Other', '离院方式 (supplementary export)'),
    'in_hospital_death': ('In-hospital death', 'TRUE/FALSE', '离院方式 or 转归情况 = 死亡'),
    'discharge_against_advice': ('Discharge against medical advice', 'TRUE/FALSE', '离院方式 = 非医嘱离院'),
    'critical_discharge_against_advice': ('Discharge against medical advice in critical condition: final label of the frozen dataset (documentation rule, '
                                          'then review of every such discharge by a pediatric clinician; the rule label kept when the reviewer was unsure)',
                                          'TRUE/FALSE', 'AE_freeze_v1.2 admission_summary: dama_critical_final'),
    'critical_discharge_rule_only': ('Critical discharge by the earlier, stricter documentation rule without clinician review (last 600 characters of the '
                                     'discharge summary state withdrawal or refusal of treatment or an anticipated risk of death, and a critical condition); '
                                     'sensitivity analysis only', 'TRUE/FALSE', 'Discharge summary (rule-based text search; text not exported)'),
    'death_or_critical_discharge': ('Primary outcome: in-hospital death or critical discharge against medical advice', 'TRUE/FALSE', 'Derived'),
    'hsctn_early_ngL': ('Early hs-cTn: first result reported from 12 h before to 48 h after admission', 'ng/L (export ng/mL x 1,000)',
                        '实验室检查_血液心肌酶谱测定: 超敏心肌钙蛋白定量'),
    'hsctn_early_hours': ('Hours from admission to the early hs-cTn report', 'hours', 'Report time (sampling time not exported)'),
    'hsctn_max_ngL': ('Highest hs-cTn during the admission (all reports)', 'ng/L', 'As hsctn_early_ngL'),
    'hsctn_n_reports': ('Number of hs-cTn reports during the admission', 'count', 'As hsctn_early_ngL'),
    'ckmb_early_U_L': ('Early CK-MB activity (first result from 12 h before to 48 h after admission)', 'U/L', '实验室检查_血液心肌酶谱测定: 肌酸激酶同工酶-MB定量'),
    'ckmb_max_U_L': ('Highest CK-MB activity during the admission', 'U/L', 'As ckmb_early_U_L'),
    'ntprobnp_early_pg_mL': ('Early NT-proBNP (first result from 12 h before to 48 h after admission)', 'pg/mL', '实验室检查_血液N末端B型钠尿肽前体测定'),
    'ntprobnp_early_hours': ('Hours from admission to the early NT-proBNP report', 'hours', 'Report time'),
    'ntprobnp_max_pg_mL': ('Highest NT-proBNP during the admission', 'pg/mL', 'As ntprobnp_early_pg_mL'),
    'lactate_early_mmol_L': ('Early blood lactate (first result from 12 h before to 48 h after admission)', 'mmol/L', '实验室检查_血液乳酸测定'),
    'lactate_early_hours': ('Hours from admission to the early lactate report', 'hours', 'Report time'),
    'lactate_max_mmol_L': ('Highest blood lactate during the admission', 'mmol/L', 'As lactate_early_mmol_L'),
    'lactate_ge_4': ('Early lactate of 4 mmol/L or higher; children without an early lactate are coded FALSE, as in Table 1', 'TRUE/FALSE', 'Derived'),
    'qtcf_early_ms': ('QTc by the Fridericia formula (QT/RR^(1/3), RR = 60/heart rate) on the first ECG from 12 h before to 48 h after admission', 'ms',
                      '心电图报告: QT and HR; recordings with HR outside 30-300/min or QT outside 150-700 ms discarded'),
    'qtcb_early_ms': ('QTc by the Bazett formula (QT/RR^(1/2)) on the same ECG', 'ms', 'As qtcf_early_ms'),
    'ecg_early_hours': ('Hours from admission to that ECG', 'hours', 'ECG time'),
    'qtcb_max_ms': ('Longest Bazett QTc during the admission', 'ms', 'As qtcf_early_ms'),
    'lvef_early_min_pct': ('Lowest left ventricular ejection fraction on echocardiograms from 24 h before to 72 h after admission', '%', '儿童心脏彩超: 射血分数'),
    'lvef_min_pct': ('Lowest left ventricular ejection fraction during the admission', '%', 'As lvef_early_min_pct'),
    'invasive_ventilation': ('Invasive mechanical ventilation at any time during the admission: intubation or endotracheal-tube order, procedure code 96.04 '
                             'or 96.70-96.72, or a ventilator order without evidence of noninvasive ventilation; intubation for planned procedures under '
                             'general anesthesia not counted', 'TRUE/FALSE', 'AE_freeze_v1.2 admission_summary: imv_ever'),
    'ventilation_start_hours': ('Hours from admission to the first invasive ventilation order or procedure (procedure dates count from 00:00)', 'hours',
                                'AE_freeze_v1.2 admission_summary: imv_first_time'),
    'vasoactive_infusion': ('Continuous intravenous infusion of epinephrine, norepinephrine, dopamine, dobutamine, milrinone, or vasopressin at any time; '
                            'intravenous push, unspecified routes, isoproterenol, and posterior pituitary injection not counted', 'TRUE/FALSE',
                            'AE_freeze_v1.2 admission_summary: vasoactive_ever'),
    'vasoactive_start_hours': ('Hours from admission to the first vasoactive infusion order', 'hours', 'AE_freeze_v1.2 admission_summary: vasoactive_first_time'),
    'vasopressor_start_hours': ('Hours from admission to the first vasopressor infusion order (dobutamine and milrinone excluded)', 'hours',
                                'AE_freeze_v1.2 support_events (vasoactive events without dobutamine or milrinone)'),
    'status_epilepticus': ('Status epilepticus in the discharge diagnoses', 'TRUE/FALSE', 'Discharge diagnoses'),
    'septic_shock': ('Septic shock in the discharge diagnoses', 'TRUE/FALSE', 'Discharge diagnoses'),
    'shock_diagnosis': ('Any shock in the discharge diagnoses', 'TRUE/FALSE', 'Discharge diagnoses'),
    'cardiac_arrest_documented': ('Cardiac arrest or CPR documented: resuscitation order, procedure code 99.6x, arrest diagnosis, or affirmative statement '
                                  'in the hospital course (warnings, consent discussions, refusals, and negations not counted)', 'TRUE/FALSE',
                                  'Orders; procedures; diagnoses; discharge summary; AE_freeze_v1.2 admission_summary: cpr_ever'),
    'cardiac_arrest_procedure_day': ('Calendar day of the first dated resuscitation order or CPR procedure code (0 = day of admission); blank when undated',
                                     'day', 'Order or procedure date (earliest of the notebook rule and AE_freeze_v1.2 cpr_first_time)'),
    'methylprednisolone': ('Methylprednisolone ordered during the admission', 'TRUE/FALSE', 'Medication orders'),
    'iv_immunoglobulin': ('Intravenous immunoglobulin ordered during the admission', 'TRUE/FALSE', 'Medication orders'),
    'tocilizumab': ('Tocilizumab ordered during the admission', 'TRUE/FALSE', 'Medication orders'),
    'in_primary_cohort': ('Primary cohort: PICU stay, early hs-cTn, and no cardiac arrest documented before or on the day of the measurement', 'TRUE/FALSE', 'Derived'),
    'excluded_arrest_rule_hsctn': ('PICU child with early hs-cTn excluded by the cardiac arrest rule', 'TRUE/FALSE', 'Derived'),
    'myocardial_injury': (f'Myocardial injury: early hs-cTnT above the 99th-percentile upper reference limit ({URL_NGL} ng/L)', 'TRUE/FALSE; blank outside the primary cohort', 'Derived'),
    'myocardial_injury_2x_url': (f'Early hs-cTnT above twice the upper reference limit ({ALT_NGL} ng/L)', 'TRUE/FALSE; blank outside the primary cohort', 'Derived'),
    'in_lvef_cohort': ('LVEF cohort: PICU stay, LVEF within 72 h, cardiac arrest rule applied', 'TRUE/FALSE', 'Derived'),
    'lvef_lt_55': ('Lowest LVEF within 72 h below 55%', 'TRUE/FALSE; blank outside the LVEF cohort', 'Derived'),
    'in_qtc_cohort': ('QTc cohort: PICU stay, ECG within 48 h, cardiac arrest rule applied', 'TRUE/FALSE', 'Derived'),
    'qtcf_ge_460': ('Early Fridericia QTc of 460 ms or longer', 'TRUE/FALSE; blank outside the QTc cohort', 'Derived'),
    'qtcb_ge_460': ('Early Bazett QTc of 460 ms or longer', 'TRUE/FALSE; blank outside the QTc cohort', 'Derived'),
    'in_ntprobnp_cohort': ('NT-proBNP cohort: PICU stay, NT-proBNP within 48 h, cardiac arrest rule applied', 'TRUE/FALSE', 'Derived'),
    'vasoactive_before_hsctn': ('Vasoactive infusion started at or before the first hs-cTn report', 'TRUE/FALSE; blank outside the primary cohort', 'Derived'),
    'ventilation_before_hsctn': ('Invasive ventilation started at or before the first hs-cTn report', 'TRUE/FALSE; blank outside the primary cohort', 'Derived'),
    'in_deterioration_cohort': ('Deterioration cohort: primary cohort without vasoactive infusion or invasive ventilation at the first hs-cTn report',
                                'TRUE/FALSE', 'Derived'),
    'vasoactive_start_within_7d': ('Vasoactive infusion started within 7 days (168 h) after the first hs-cTn report',
                                   'TRUE/FALSE; blank outside the deterioration cohort', 'Derived'),
    'ventilation_start_within_7d': ('Invasive ventilation started within 7 days after the first hs-cTn report', 'TRUE/FALSE; blank outside the deterioration cohort',
                                    'Derived'),
    'death_or_critical_discharge_within_7d': ('Death or critical discharge within 7 days after the first hs-cTn report',
                                              'TRUE/FALSE; blank outside the deterioration cohort', 'Derived (event time = discharge time)'),
    'deterioration_7d': ('Deterioration (primary): start of vasoactive infusion or invasive ventilation, or death or critical discharge, within 7 days after '
                         'the first hs-cTn report', 'TRUE/FALSE; blank outside the deterioration cohort', 'Derived'),
    'deterioration_72h': ('Deterioration with every component within 72 h of the report', 'TRUE/FALSE; blank outside the deterioration cohort', 'Derived'),
    'deterioration_any_time': ('Deterioration with every component at any time during the admission', 'TRUE/FALSE; blank outside the deterioration cohort',
                               'Derived'),
    'hours_hsctn_to_escalation': ('Hours from the first hs-cTn report to the first start of vasoactive infusion or invasive ventilation (any time)', 'hours',
                                  'Derived; blank when no escalation or outside the deterioration cohort'),
}
IMG_FEAT = {'thal_bilat': 'bilateral thalamic lesions (side-aware parsing)', 'thal_any': 'a thalamic lesion on either side', 'brainstem': 'a brainstem lesion',
            'diffuse_edema': 'diffuse cerebral edema (global swelling, herniation, cisternal effacement, or bilateral loss of gray-white differentiation with '
                             'effacement)', 'hemispheric': 'unilateral hemispheric swelling or restricted diffusion',
            'splenial': 'a splenial lesion of the corpus callosum', 'subcortical_dwi': 'bilateral or multilobar subcortical white-matter restricted diffusion',
            'lobar_wm_dwi': 'extensive lobar white-matter restricted diffusion', 'any_dwi': 'any restricted diffusion',
            'normal': 'a normal report (no abnormality in the conclusion)', 'imp_ane': 'a radiologist impression of acute necrotizing encephalopathy',
            'imp_mers': 'a radiologist impression of MERS or reversible splenial lesion syndrome', 'imp_aesd': 'a radiologist impression of AESD'}
DD_EV = {
    'patient': DD_A['patient'], 'vis': DD_A['vis'],
    'clinician_label': DD_A['clinician_label'], 'ites_class': DD_A['ites_class'], 'ites_algorithmic': DD_A['ites_class_algorithmic'],
    'ites_tier': DD_A['ites_tier'], 'ites_reason': DD_A['ites_reason'], 'needs_review': DD_A['ites_needs_review'], 'review_reasons': DD_A['ites_review_reasons'],
    'adjudicated_class': ('Category entered by the adjudicator (blank = algorithmic category kept)', 'as ites_class', 'output_ites/ites_adjudicated.csv'),
    'fever_text': ('Fever mentioned, not negated, in the chief complaint or history of present illness', 'TRUE/FALSE', 'Admission note (text not exported)'),
    'fever_course': ('Fever mentioned in the first 400 characters of the hospital course', 'TRUE/FALSE', 'Discharge summary (text not exported)'),
    'tmax_72h': ('Highest recorded temperature from 24 h before to 72 h after admission', 'degrees C', '患者生命体征'),
    'fever': ('Febrile illness documented: fever_text, fever_course, or tmax_72h of 38.0 degrees C or higher', 'TRUE/FALSE', 'Derived'),
    'encephalopathy_text': ('Altered consciousness documented (coma, stupor, somnolence, lethargy, confusion, delirium, irritability, or GCS)', 'TRUE/FALSE',
                            'Admission note and hospital course (text not exported)'),
    'seizure_text': ('Seizures documented', 'TRUE/FALSE', 'Admission note and hospital course'),
    'biphasic_text': ('Cue for a second seizure cluster (for example, recurrent seizures)', 'TRUE/FALSE', 'History of present illness and hospital course'),
    'prior_epilepsy': ('Known epilepsy or infantile spasms before the admission', 'TRUE/FALSE', 'Past history'),
    'prior_neuro_disease': ('Known cerebral palsy, developmental delay, genetic-metabolic or mitochondrial disease, or hydrocephalus', 'TRUE/FALSE', 'Past history'),
    'mri_n': ('Number of brain MRI reports during the admission (blank = none)', 'count', 'MRI报告'),
    'mri_first_day': ('Days from admission to the first brain MRI', 'days', 'MRI报告'),
    'ct_n': ('Number of head CT reports during the admission (blank = none)', 'count', 'CT报告'),
    'ct_first_day': ('Days from admission to the first head CT', 'days', 'CT报告'),
    'eeg_n': ('Number of EEG reports during the admission (blank = none)', 'count', '脑电图检查'),
    'eeg_slow': ('Diffuse slowing on any EEG', 'TRUE/FALSE; blank = no EEG', '脑电图检查: background and impression'),
    'eeg_suppressed': ('Background suppression (electrocerebral silence, burst suppression, or low voltage) on any EEG', 'TRUE/FALSE; blank = no EEG', '脑电图检查'),
    'eeg_seizure': ('Electrographic seizures or status epilepticus on any EEG', 'TRUE/FALSE; blank = no EEG', '脑电图检查'),
    'csf_wbc_max': ('Highest cerebrospinal fluid white-cell count', 'x10^6/L', '实验室检查_脑脊液常规'),
    'csf_protein_max': ('Highest cerebrospinal fluid protein', 'g/L', '实验室检查_脑脊液生化'),
    'csf_virus_positive': ('Positive cerebrospinal fluid viral PCR', 'TRUE/FALSE', '实验室检查_脑脊液病毒DNA测定'),
    'csf_pleocytosis': ('Cerebrospinal fluid white-cell count above 20 x10^6/L', 'TRUE/FALSE; blank = no CSF count', 'Derived'),
    'hb_min': ('Lowest hemoglobin from 12 h before to 72 h after admission', 'g/L', '实验室检查_血常规'),
    'plt_min': ('Lowest platelet count, same window', 'x10^9/L', '实验室检查_血常规'),
    'inr_max': ('Highest INR, same window', 'ratio', '实验室检查_凝血功能检查'),
    'fib_low': ('Fibrinogen below the laboratory lower limit, same window', 'TRUE/FALSE', '实验室检查_凝血功能检查'),
    'ph_min': ('Lowest blood pH, same window', 'pH', '实验室检查_血气分析'),
    'be_min': ('Lowest base excess, same window', 'mmol/L', '实验室检查_血气分析'),
    'alt_x_uln': ('Highest ALT as a multiple of the age-specific upper limit, same window', 'ratio', '实验室检查_血液肝功能测定'),
    'ast_x_uln': ('Highest AST as a multiple of the age-specific upper limit, same window', 'ratio', '实验室检查_血液肝功能测定'),
    'cr_x_uln': ('Highest creatinine as a multiple of the age-specific upper limit, same window', 'ratio', '实验室检查_血液肾功能测定'),
    'ammonia_max': ('Highest blood ammonia, same window', 'as exported (umol/L)', '实验室检查_血氨AMMO'),
    'na_max': ('Highest serum sodium, same window', 'mmol/L', '实验室检查_血液电解质测定'),
    'mof_anemia': ('ASEM organ item: hemoglobin below 100 g/L', 'TRUE/FALSE', 'Derived'),
    'mof_thrombocytopenia': ('ASEM organ item: platelets below 100 x10^9/L', 'TRUE/FALSE', 'Derived'),
    'mof_dic': ('ASEM organ item: DIC (INR of 1.5 or more with low fibrinogen, or DIC diagnosis)', 'TRUE/FALSE', 'Derived'),
    'mof_acidosis': ('ASEM organ item: pH below 7.30 or base excess of -8 mmol/L or lower', 'TRUE/FALSE', 'Derived'),
    'mof_liver': ('ASEM organ item: ALT or AST above twice the upper limit', 'TRUE/FALSE', 'Derived'),
    'mof_renal': ('ASEM organ item: creatinine above the upper limit or acute kidney injury diagnosis', 'TRUE/FALSE', 'Derived'),
    'mof_count': ('Number of ASEM organ items present', '0-6', 'Derived'),
    'shock': ('Shock: vasoactive infusion within 72 h of admission or shock diagnosis (hemorrhagic and cardiogenic shock excluded)', 'TRUE/FALSE', 'Derived'),
    'septic_shock': ('Septic shock in the discharge diagnoses', 'TRUE/FALSE', 'Discharge diagnoses'),
    'sepsis_dx': ('Sepsis or septic shock in the discharge diagnoses', 'TRUE/FALSE', 'Discharge diagnoses'),
    'cpr': ('Cardiac arrest or CPR documented (as cardiac_arrest_documented)', 'TRUE/FALSE', 'Derived'),
    'neonate': ('Age younger than 28 days', 'TRUE/FALSE', 'Derived'),
    'exclusion_diagnoses': ('Alternative diagnoses named without qualification (exclude ITES; categories as in eTable 1)', 'semicolon-separated categories',
                            'Discharge diagnoses'),
    'qualified_exclusion_diagnoses': ('Alternative diagnoses qualified by "?" or "possible" (flag for review only)', 'semicolon-separated categories',
                                      'Discharge diagnoses'),
}
DD_M = {
    'patient': DD_A['patient'], 'vis': DD_A['vis'],
    'measure': ('Measurement', 'hs-cTn; CK-MB activity; NT-proBNP; Lactate; ECG heart rate; QT interval; QTcF; QTcB; LVEF', 'See rows "Measurements_long (measure)"'),
    'unit': ('Unit of value', 'text', 'Derived'),
    'value': ('Result (one row per laboratory report, ECG, or echocardiogram; laboratory duplicates removed by report number)', 'see unit', 'See measure'),
    'hours_from_admission': ('Hours from admission to the report (ECG and echocardiography: examination time)', 'hours', 'Report or examination time'),
    'in_early_window': ('Within the window used for the early value (12 h before to 48 h after admission; LVEF 24 h before to 72 h after)', 'TRUE/FALSE', 'Derived'),
}
MEASURE_SRC = [('hs-cTn', 'ng/L', '实验室检查_血液心肌酶谱测定: 超敏心肌钙蛋白定量-定量结果 (ng/mL x 1,000); both exports'),
               ('CK-MB activity', 'U/L', '实验室检查_血液心肌酶谱测定: 肌酸激酶同工酶-MB定量-定量结果; both exports'),
               ('NT-proBNP', 'pg/mL', '实验室检查_血液N末端B型钠尿肽前体测定: N末端B型利钠肽原定量-定量结果; both exports'),
               ('Lactate', 'mmol/L', '实验室检查_血液乳酸测定: 乳酸定量-定量结果; both exports'),
               ('ECG heart rate', 'beats/min', '心电图报告: HR (recordings with HR 30-300/min and QT 150-700 ms only)'),
               ('QT interval', 'ms', '心电图报告: QT'), ('QTcF', 'ms', 'QT/RR^(1/3), RR = 60/HR'), ('QTcB', 'ms', 'QT/RR^(1/2), equals the machine QTc'),
               ('LVEF', '%', '儿童心脏彩超: 射血分数')]

def ev_feature_desc(col):
    m = re.fullmatch(r'(mri|ct)_(.+?)(_day2plus|_day)?', col)
    if not m or m.group(2) not in IMG_FEAT:
        return None
    mod, feat = ('Brain MRI' if m.group(1) == 'mri' else 'Head CT'), IMG_FEAT[m.group(2)]
    src = ('MRI报告' if m.group(1) == 'mri' else 'CT报告') + ': findings and conclusion (clause-level negation)'
    if m.group(3) == '_day':
        return (f'{mod}: days from admission to the first report showing {feat}', 'days', src)
    if m.group(3) == '_day2plus':
        return (f'{mod}: {feat} on a report from day 2 after admission onward', 'TRUE/FALSE', src)
    return (f'{mod}: {feat} on any report during the admission', 'TRUE/FALSE; blank = no report', src)

def kind_of(s):
    if str(s.dtype) in ('bool', 'boolean'):
        return 'TRUE/FALSE'
    if pd.api.types.is_integer_dtype(s):
        return 'integer'
    if pd.api.types.is_float_dtype(s):
        return 'number'
    if s.dropna().map(type).eq(type(pd.Timestamp(0).date())).all() and s.notna().any():
        return 'date'
    return 'text'

dict_rows, undocumented = [], []
for sheet_, df_, desc_ in [('Analysis_dataset', A, DD_A), ('ITES_evidence', EVS, DD_EV), ('Measurements_long', meas, DD_M)]:
    for col_ in df_.columns:
        d_ = desc_.get(col_) or ev_feature_desc(col_)
        if d_ is None:
            undocumented.append(f'{sheet_}.{col_}')
            d_ = ('Derived in the notebook (see code)', '', '')
        dict_rows.append({'Sheet': sheet_, 'Variable': col_, 'Description': d_[0], 'Type': kind_of(df_[col_]), 'Unit or coding': d_[1],
                          'Source or derivation': d_[2], 'Non-missing, n': int(df_[col_].notna().sum()), 'Missing, n': int(df_[col_].isna().sum())})
for m_, u_, s_ in MEASURE_SRC:
    sub_ = meas[meas.measure == m_]
    dict_rows.append({'Sheet': 'Measurements_long (measure)', 'Variable': m_, 'Description': f'{m_} results of the index admissions', 'Type': 'number',
                      'Unit or coding': u_, 'Source or derivation': s_, 'Non-missing, n': len(sub_), 'Missing, n': 0})
dictionary = pd.DataFrame(dict_rows)
if undocumented:
    print('Variables without a dictionary entry:', undocumented)

# Descriptive summary by analytic population.
POPS = [('All index admissions', pd.Series(True, index=A.index)), ('PICU stay', A.picu_stay), ('Primary cohort', A.in_primary_cohort),
        ('Deterioration cohort', A.in_deterioration_cohort)]
POP_COLS = [f'{n_} (n = {int(m_.sum())})' for n_, m_ in POPS]
NO_SUMMARY = {'patient', 'vis', 'admission_date', 'discharge_date', 'ites_reason', 'ites_review_reasons'}
LEVEL_ORDER = {'admission_period': ['Pre-2020', '2020-Nov 2022', 'Dec 2022-Jan 2023', 'Feb 2023 onward'], 'syndrome_group': ORDER_G}
desc_rows = []
for col_ in A.columns:
    if col_ in NO_SUMMARY:
        continue
    s_all = A[col_]
    binary = str(s_all.dtype) in ('bool', 'boolean')
    numeric = pd.api.types.is_numeric_dtype(s_all) and not binary and col_ != 'admission_year'
    if not binary and not numeric:
        levels = s_all.astype('string').dropna().value_counts().index.tolist()
        if col_ == 'admission_year':
            levels = sorted(levels)
        elif col_ in LEVEL_ORDER:
            levels = [l for l in LEVEL_ORDER[col_] if l in levels] + [l for l in levels if l not in LEVEL_ORDER[col_]]
        for lvl in levels:
            row_ = {'Variable': col_, 'Level': lvl}
            for (n_, m_), pc in zip(POPS, POP_COLS):
                s_ = s_all[m_.astype(bool)].astype('string').dropna()
                k_ = int((s_ == lvl).sum())
                row_[pc] = f'{k_} ({rnd(100 * k_ / len(s_))}%)' if len(s_) else ''
            row_['Missing, all (n)'] = int(s_all.isna().sum())
            desc_rows.append(row_)
        continue
    row_ = {'Variable': col_, 'Level': ''}
    dec = 0 if pd.api.types.is_integer_dtype(s_all) else 1
    for (n_, m_), pc in zip(POPS, POP_COLS):
        s_ = s_all[m_.astype(bool)].dropna()
        if not len(s_):
            row_[pc] = ''
        elif binary:
            k_ = int(s_.astype(bool).sum())
            row_[pc] = f'{k_}/{len(s_)} ({rnd(100 * k_ / len(s_))}%)'
        else:
            s_ = s_.astype(float)
            row_[pc] = (f'{rnd(s_.median(), dec)} ({rnd(s_.quantile(.25), dec)}-{rnd(s_.quantile(.75), dec)}); '
                        f'range {rnd(s_.min(), dec)}-{rnd(s_.max(), dec)}; n = {len(s_)}')
    row_['Missing, all (n)'] = int(s_all.isna().sum())
    desc_rows.append(row_)
desc = pd.DataFrame(desc_rows)

msum = (meas.groupby(['measure', 'unit'], sort=False)
            .agg(reports=('value', 'size'), admissions=('vis', 'nunique'), reports_in_early_window=('in_early_window', 'sum'),
                 median=('value', 'median'), q1=('value', lambda s: s.quantile(.25)), q3=('value', lambda s: s.quantile(.75)),
                 minimum=('value', 'min'), maximum=('value', 'max'))
            .reset_index())
for c_ in ['median', 'q1', 'q3', 'minimum', 'maximum']:
    msum[c_] = msum[c_].round(2)

adjudicated = 'adjudicated_class' in ev.columns
readme = pd.DataFrame([
    ('Title', 'Acute encephalopathy cardiac study: cleaned dataset'),
    ('INTERNAL USE ONLY', 'Contains research patient and visit identifiers and admission dates. Keep within the study team; never upload as '
                          'Supplemental Digital Content or share outside the study.'),
    ('Generated', f'{pd.Timestamp.now():%Y-%m-%d %H:%M} by AES_publication_pipeline.ipynb from {WB.name} and {WB_SUPP.name}'),
    ('Unit of analysis', 'One row per child: the first acute-phase acute encephalopathy admission with a recorded discharge disposition (index admission); '
                         f'{len(A)} children.'),
    ('Syndrome categories', '2024 international consensus definitions of infection-triggered encephalopathy syndromes (Sakuma et al, Dev Med Child Neurol '
                            '2025;67:195-207), operationalized as in eTable 1; ' + ('adjudicated categories applied.' if adjudicated else
                                                                                   'algorithmic categories (no adjudication file found).')),
    ('Sheet: Data_dictionary', 'Every variable of the three dataset sheets: description, type, unit or coding, source in the export, and missing counts.'),
    ('Sheet: Analysis_dataset', 'Demographics, syndrome classification, outcomes, cardiac measures, organ support, treatments, and analytic-population '
                                'flags (one row per child). Reproduces every cohort and count of the manuscript.'),
    ('Sheet: ITES_evidence', 'Structured evidence behind each consensus category: fever, altered consciousness, seizures, MRI/CT/EEG features, CSF, '
                             'first-72-hour laboratory extremes, ASEM organ items, shock, and exclusion diagnoses (one row per child).'),
    ('Sheet: Measurements_long', 'Every hs-cTn, CK-MB, NT-proBNP, lactate, ECG (heart rate, QT, QTcF, QTcB), and LVEF result of the index admissions, '
                                 'with hours from admission (one row per result).'),
    ('Sheet: Measurement_summary', 'Number of results and admissions, and the distribution of values, for each measure.'),
    ('Sheet: Cohort_flow', 'Derivation of the analytic populations with outcome counts (as eTable 2).'),
    ('Sheet: Descriptive_summary', 'Every variable summarized in all index admissions, PICU children, the primary cohort, and the deterioration cohort.'),
    ('Conventions', 'TRUE/FALSE flags; blank = not measured, no report, or outside the analytic population named in the dictionary. Times are hours '
                    'from admission based on report times (sampling times were not exported). Free text from medical records is not included.'),
    ('Key definitions', f'Myocardial injury: early hs-cTnT above {URL_NGL} ng/L. Primary cohort: PICU children with early hs-cTnT after the cardiac arrest '
                        'rule. Deterioration cohort: primary cohort without vasoactive infusion or invasive ventilation at the first hs-cTnT report; '
                        'deterioration = start of either, or death or critical discharge, within 7 days after the report. Critical discharge, organ '
                        'support, and resuscitation follow the frozen dataset AE_freeze_v1.2 (see the data dictionary).'),
], columns=['Item', 'Description'])

XL = DATA / 'AES_cleaned_dataset_INTERNAL.xlsx'
SHEETS = [('README', readme), ('Data_dictionary', dictionary), ('Analysis_dataset', A), ('ITES_evidence', EVS), ('Measurements_long', meas),
          ('Measurement_summary', msum), ('Cohort_flow', flow), ('Descriptive_summary', desc)]
WIDTH = {'README': {'A': 26, 'B': 140}, 'Data_dictionary': {'A': 26, 'B': 34, 'C': 70, 'D': 11, 'E': 40, 'F': 50},
         'Descriptive_summary': {'A': 34, 'B': 30}, 'Cohort_flow': {'A': 80}}
with pd.ExcelWriter(XL, engine='openpyxl', date_format='YYYY-MM-DD') as xw:
    for name_, df_ in SHEETS:
        df_.to_excel(xw, sheet_name=name_, index=False)
        ws = xw.sheets[name_]
        for c_ in ws[1]:
            c_.font = XFont(bold=True)
            c_.alignment = XAlign(wrap_text=True, vertical='top')
        for j_ in range(1, ws.max_column + 1):
            letter = ws.cell(1, j_).column_letter
            ws.column_dimensions[letter].width = WIDTH.get(name_, {}).get(letter, 18 if name_ not in ('Descriptive_summary',) else 34)
        if name_ in ('README', 'Data_dictionary', 'Descriptive_summary'):
            for row_ in ws.iter_rows(min_row=2):
                for c_ in row_:
                    c_.alignment = XAlign(wrap_text=True, vertical='top')
        if name_ in ('Data_dictionary', 'Analysis_dataset', 'ITES_evidence', 'Measurements_long', 'Descriptive_summary'):
            ws.auto_filter.ref = ws.dimensions
            ws.freeze_panes = 'C2' if name_ in ('Analysis_dataset', 'ITES_evidence', 'Measurements_long') else 'A2'
    xw.sheets['README']['A2'].font = XFont(bold=True, color='C00000')
    xw.sheets['README']['B2'].font = XFont(bold=True, color='C00000')
A.to_csv(DATA / 'analysis_dataset_INTERNAL.csv', index=False, encoding='utf-8-sig')
EVS.to_csv(DATA / 'ites_evidence_INTERNAL.csv', index=False, encoding='utf-8-sig')
meas.to_csv(DATA / 'measurements_long_INTERNAL.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([(n_, df_.shape[0], df_.shape[1]) for n_, df_ in SHEETS], columns=['sheet', 'rows', 'columns'])

## Part 6. Output inventory and identifier check

Research identifiers and admission timestamps must appear only in `cleaned_data_INTERNAL/`; every other output is
checked for them.

In [ ]:
# Output inventory and identifier check.
import zipfile

def text_of(path):
    if path.suffix in ('.json', '.csv', '.txt'):
        return path.read_text(encoding='utf-8', errors='ignore')
    if path.suffix in ('.docx', '.xlsx'):
        with zipfile.ZipFile(path) as z:
            # document content only (docProps/ holds the file's own creation time, not patient data)
            return ' '.join(z.read(n).decode('utf-8', 'ignore') for n in z.namelist() if n.endswith('.xml') and not n.startswith('docProps/'))
    return ''

ID_SET = set(idx.vis.astype(str)) | set(idx.patient.astype(str))
STAMP = re.compile(r'\b(?:19|20)\d{2}-\d{2}-\d{2}[ T]\d{2}:\d{2}')
problems = []
for path in sorted(p for p in PUB.rglob('*') if p.is_file() and DATA not in p.parents):
    t_ = text_of(path)
    n_ids, n_stamps = len(ID_SET & set(re.findall(r'\d{6,}', t_))), len(STAMP.findall(t_))
    if n_ids or n_stamps:
        problems.append((str(path.relative_to(PUB)), n_ids, n_stamps))
if problems:
    raise RuntimeError(f'Identifiers or timestamps found outside cleaned_data_INTERNAL/: {problems}')
print('Identifier check passed: no research IDs or admission timestamps outside cleaned_data_INTERNAL/.')
print(f"Key results: primary cohort n = {COH['primary']}; myocardial injury {INJ['primary']}; deterioration "
      f"{DET['injured']} vs {DET['not_injured']}, adjusted RR {rr_cell(det_primary['RR (95% CI)'])}; fragility index {FRAG}.")
inventory = pd.DataFrame([(str(p.parent.relative_to(PUB)), p.name, round(p.stat().st_size / 1e6, 2))
                          for p in sorted(PUB.rglob('*')) if p.is_file()], columns=['folder', 'file', 'MB'])
inventory